# Text → STL: end-to-end pipeline
Type a plain-English description of a **pipe gasket, washer/spacer, L-bracket, rectangular container, or cylindrical container** → get a checked parameter table → confirm → download an STL.

**How to run:** Runtime ▸ Change runtime type ▸ **T4 GPU** → run the cells top to bottom (Runtime ▸ Run all). When asked, upload `24679_Project1_Data.xlsx`. The last-but-one cell starts the app and shows it right in the notebook (plain HTML page + a small Flask server, no Gradio).

| Your pipeline diagram box | Where it lives |
|---|---|
| Input text description → GUI text box | GUI cell (§10) |
| Preprocess to numeric decimal form | §2 `normalize_text` |
| Trained-from-scratch model → part type | §6 (PyTorch classifier, random init) |
| Fine-tuned model → parameters | §7 (LoRA on Qwen2.5) |
| JSON with standard schema for all parts | §3 `SCHEMA`, `params_to_json` |
| Validation (negatives, missing, range) + off-the-shelf model verifies? | §4 rules (always on) + optional LLM verifier (`USE_LLM_VERIFIER`) |
| Table for the user to confirm/edit → confirm button | GUI cell (§10) |
| JSON → CAD script → CAD file → 3D preview, download, start over | §5 CAD + GUI cell |
| Add features (holes at positions you specify) after the first build | §5 hole schema + checks, §5c data template, GUI "Add features" card |
| User edits loop back | changing the part-type dropdown re-runs the extractor; every confirmed edit is logged to `work/feedback.jsonl` |

The deterministic parts (preprocessing, validation, CAD) run and self-test **before** the slow training cells, so a problem there shows up in seconds.

In [ ]:
# Runtime > Change runtime type > T4 GPU (needed for the fine-tuning cell). Then run this cell once.
# If Colab asks you to restart the runtime after installing, do that and continue from the next cell.
!pip -q install cadquery trimesh flask "transformers>=4.44" peft accelerate openpyxl scikit-learn
# Colab ships an old torchao (0.10) that newer peft refuses to import; we don't use it, so remove it.
!pip -q uninstall -y torchao

## 0. Setup & settings
Everything you might want to tune is at the top of this cell.

In [ ]:
import os, re, json, math, random, time, zipfile, copy
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")   # must be set before torch touches the GPU; reduces memory fragmentation
from pathlib import Path
from contextlib import nullcontext
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ CONFIG
SEED = 42
DATA_PATH = None                            # None = Colab asks you to upload the .xlsx every run (so an old copy is never used by accident). Locally: set a path.
WORKDIR = Path("work"); WORKDIR.mkdir(exist_ok=True)

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # fine-tuned model. Use "Qwen/Qwen2.5-0.5B-Instruct" for a faster/lighter run
LORA_EPOCHS = 5                             # fewer epochs than before: with augmentation there is ~2.5x more training data per epoch
LORA_LR = 2e-4
BATCH_SIZE = 8                              # examples per optimizer step (effective batch)
MICRO_BATCH = 2                             # examples per forward/backward pass; BATCH_SIZE is reached by gradient accumulation. Lower to 1 if you still run out of GPU memory
GRAD_CHECKPOINTING = True                   # recompute activations instead of storing them: ~30% slower but needs far less GPU memory (needed for the 1.5B model on a T4)

MAKE_FEATURE_TEMPLATE = False               # optional: section 5c can write a blank template if you want to add MORE hand-written hole sentences
LOAD_FEATURE_DATA = True                    # hole-feature training data (built-in hand-written sentences + generated ones). False = no feature training
FEATURE_DATA_PATH = None                    # None = use the hole sentences built into this notebook. Optional: path to your own edited copy of hole_features_filled.xlsx
FEATURE_GENERATED_PER_TYPE = 100            # extra generated hole sentences per part type (3 types). 0 = only the hand-written ones
AUGMENT_PER_TYPE = 100                      # synthetic training samples per part type (5 types x 100 = 500). Set 0 to turn augmentation off
AUG_SEED = 7
TARGET_MANUAL_SAMPLES = 500                 # assignment goal, only used for a progress line
EXCLUDE_AUDIT_FLAGGED = False               # True = drop rows whose label numbers don't appear in their text
GROUP_SPLIT = True                          # True = a given pipe size / screw size never appears in both train and test
TEST_FRAC, VAL_FRAC = 0.15, 0.15

NUM_TOL = 0.0075                            # inches: tolerance for "value appears in text" and for eval correctness
SNAP_TO_STANDARD = True                     # snap values within tolerance of a known catalog size to that size...
SNAP_PART_TYPES = {"pipe_gasket", "washer", "bracket"}   # ...but ONLY for catalog parts. Container sizes are free-form (3.58 vs 3.57 are both real)
RECT_DIMS_ARE_OUTER = True                  # rectangular container: length/width are OUTSIDE dimensions (walls + floor are cut inward from them)
USE_LLM_VERIFIER = False                    # optional off-the-shelf LLM check (weak with small models, see Stage 3)

random.seed(SEED); np.random.seed(SEED)

def show_df(df):
    """pretty table in Colab/Jupyter, plain text elsewhere"""
    try: display(df)
    except NameError: print(df.to_string())

## 1. Load and clean the spreadsheet
All sheets are combined. The `L_brackets` sheet's text column is headed `n`; it is renamed to `text`. Rows without text stay available for the catalog tables but can't be used to train the language models.

In [ ]:
# ------------------------------------------------------------------ LOAD + CLEAN THE SPREADSHEET
if DATA_PATH is None or not Path(DATA_PATH).exists():
    try:
        from google.colab import files            # only exists inside Colab
        print("Upload your main dataset spreadsheet (.xlsx).")
        for _f in files.upload():
            if any(n.startswith("Holes_") for n in pd.ExcelFile(_f).sheet_names): FEATURE_DATA_PATH = _f      # the hole-feature workbook
            else: DATA_PATH = _f                                                                            # the main part dataset
    except ImportError:
        raise FileNotFoundError("Set DATA_PATH to your .xlsx file (the notebook is not running in Colab).")
    if DATA_PATH is None: raise FileNotFoundError("No main dataset was uploaded (a workbook without Holes_* sheets).")
print("Using file:", DATA_PATH, "| hole-feature workbook:", FEATURE_DATA_PATH)

xl = pd.ExcelFile(DATA_PATH)
frames = []
for sheet in xl.sheet_names:
    d = xl.parse(sheet)
    if "text" not in d.columns and "n" in d.columns:   # the L_brackets sheet's text column is headed "n"
        d = d.rename(columns={"n": "text"})
    d["sheet"] = sheet
    d["sheet_row"] = np.arange(len(d)) + 2              # Excel row number (header = row 1)
    frames.append(d)
raw_all = pd.concat(frames, ignore_index=True)
raw_all["text"] = raw_all["text"].astype("string").str.strip()

has_text = raw_all["text"].notna() & (raw_all["text"] != "")
summary = raw_all.assign(has_text=has_text).groupby("sheet").agg(rows=("has_text", "size"), with_text=("has_text", "sum"))
summary["missing_text"] = summary["rows"] - summary["with_text"]
print("sheets found:", xl.sheet_names)
print(summary, "\n")

df_all = raw_all[has_text].copy().reset_index(drop=True)
print(f"Rows with real text (used for ML): {len(df_all)}  |  goal: {TARGET_MANUAL_SAMPLES}  |  still needed: {max(0, TARGET_MANUAL_SAMPLES - len(df_all))}")
print("Rows without text are kept for lookup tables / CAD standards, but not used to train the language models.")

## 2. Preprocessing — everything to decimal numbers
Fractions (`1 1/16`), mixed numbers, word fractions (`a sixteenth`, `three and a half`), number words (`eight bolt holes`, `a dozen`), leading-dot decimals (`.0625`), glued units (`4.62OD`), and screw-size phrasing (`triple aught`, `quarter twenty`, `#8`, `metric 3.9`).  
The **same function** runs on training text and on what a user types, so the models never see two different formats.

In [ ]:
# ------------------------------------------------------------------ PREPROCESSING: everything -> decimal numbers
# Pipeline box: "Data is preprocessed to make all dimensions and numbers numeric decimal form".
# The SAME function is used for training text and for live user input (no train/serve skew).

NUM_WORDS = {"zero":0,"one":1,"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
             "eleven":11,"twelve":12,"thirteen":13,"fourteen":14,"fifteen":15,"sixteen":16,"seventeen":17,
             "eighteen":18,"nineteen":19,"twenty":20,"thirty":30,"forty":40,"fifty":50}
_UNITS = "one|two|three|four|five|six|seven|eight|nine"
_WORDNUM = "|".join(sorted(NUM_WORDS, key=len, reverse=True))
WORD_FRACS = {"sixteenth": 1/16, "eighth": 1/8, "quarter": 1/4, "thirty-second": 1/32, "sixty-fourth": 1/64}

def dec(x, places=4):
    """0.0625 -> '0.0625', 3.5 -> '3.5', 4.0 -> '4'  (max 4 decimals, no trailing zeros)"""
    return f"{x:.{places}f}".rstrip("0").rstrip(".")

def _word_or_digit(s):
    s = s.lower()
    return float(NUM_WORDS[s]) if s in NUM_WORDS else float(s)

_AUGHT   = [(re.compile(r"\btriple[- ]aught\b", re.I), "No. 000"),
            (re.compile(r"\bdouble[- ]aught\b", re.I), "No. 00"),
            (re.compile(r"\bsingle[- ]aught\b", re.I), "No. 0")]
_QTR20   = re.compile(r"\bquarter[- ]twenty\b", re.I)
_FRACWORD = {"half": 0.5, "quarter": 0.25, "eighth": 0.125, "sixteenth": 0.0625, "three quarters": 0.75, "three-quarters": 0.75}
_AND_FRAC = re.compile(rf"\b(an|{_WORDNUM}|\d+(?:\.\d+)?)(?:\s+inch(?:es)?)?\s+and\s+(?:(?:a|an)\s+(half|quarter|eighth|sixteenth)|(three[- ]quarters))\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_DOZEN   = re.compile(r"\b(?:a\s+)?dozen\b", re.I)
_HALF_IN = re.compile(r"\b(?:a\s+)?half(?:[- ]an?)?[- ]inch(?:es)?\b", re.I)
_AN_INCH = re.compile(r"\ban\s+inch\b", re.I)
_WORDFRAC= re.compile(r"\b(?:an?\s+)?(sixteenth|eighth|quarter|thirty[- ]second|sixty[- ]fourth)\b(?:\s+of\s+an)?(?:\s+inch(?:es)?)?", re.I)
_COUNT_CTX = r'(?=\s*(?:inch|inches|in\b|"|bolts?\b|holes?\b|mounting\b|mount\b|rivets?\b|points?\b|hardware\b|screws?\b|feet\b|foot\b|ft\b|millimet\w*|mm\b|centimet\w*|cm\b|meters?\b|metres?\b))'
_TENS_UNITS = re.compile(rf"\b(twenty|thirty|forty|fifty)[- ]({_UNITS})\b{_COUNT_CTX}", re.I)
_WORDNUM_CTX= re.compile(rf"\b({_WORDNUM})\b{_COUNT_CTX}", re.I)
_NO_SIZE = re.compile(r"(?<!\w)(?:no\.?|number|#)\s*(\d+)\b", re.I)
_SIZE_N  = re.compile(r"\bsize\s+(\d+)\b(?=\s+(?:screw|bolt|hardware))", re.I)
_METRIC  = re.compile(r"\bmetric\s+(\d+(?:\.\d+)?)\b", re.I)
_MIXED   = re.compile(r"(?<![\d.])(\d+)\s+(\d+)/(\d+)(?!\d)")
_FRAC    = re.compile(r"(?<![\d.])(\d+)/(\d+)(?!\d)")
_LEADDOT = re.compile(r"(?<!\d)\.(\d+)")
_DIMX    = re.compile(r"(?<=\d)\s*[xX\u00d7]\s*(?=\d)")          # 3x11 / 3 X 11 / 2x4x6  ->  "3 x 11"
_NUMRE   = r"(\d+(?:\.\d+)?)"
# ---- units -> inches (the whole pipeline works in inches, so the models only ever see inches)
_FT_AND_HALF = re.compile(rf"\b({_WORDNUM}|\d+(?:\.\d+)?)\s+and\s+a\s+half\s+(?:feet|foot|ft)\b", re.I)
_FOOT_WORDS  = [(re.compile(r"\b(?:a|one)\s+foot\s+and\s+a\s+half\b", re.I), "18 inch"),
                (re.compile(r"\bhalf\s+(?:a\s+)?foot\b", re.I), "6 inch"),
                (re.compile(r"\b(?:a|one)\s+foot\b", re.I), "12 inch")]
_FT_IN   = re.compile(_NUMRE + r"\s*(?:feet|foot|ft\b\.?|')\s*(?:and\s+)?" + _NUMRE + r"\s*(?:inches|inch|in\b|\")", re.I)     # 5 ft 6 in / 5'6"
_FT      = re.compile(_NUMRE + r"\s*(?:feet\b|foot\b|ft\b\.?)|" + _NUMRE + r"\s*'(?![A-Za-z0-9])", re.I)                        # 2 feet / 2 ft / 2'
_MM_SCREW= re.compile(_NUMRE + r"\s*mm\s*(?=(?:machine\s+)?(?:screws?|bolts?|fasteners?)\b)", re.I)                          # "6 mm screw" is the size M6, not a length
_MM      = re.compile(_NUMRE + r"\s*(?:millimet(?:er|re)s?|mm)\b", re.I)
_CM      = re.compile(_NUMRE + r"\s*(?:centimet(?:er|re)s?|cm)\b", re.I)
_METERS  = re.compile(_NUMRE + r"\s*(?:meters?|metres?)\b", re.I)
_M_ABBR  = re.compile(r"(?<![A-Za-z])" + _NUMRE + r"\s*m\b(?!\s*\d)")                                                          # bare lowercase 'm' after a number (M5 is a screw, so no capital M)
_GLUE    = re.compile(r"(\d)([A-Za-z])")

def normalize_text(text: str) -> str:
    t = str(text)
    for pat, rep in _AUGHT: t = pat.sub(rep, t)
    t = _QTR20.sub("0.25", t)
    t = _DOZEN.sub("12", t)
    t = _FT_AND_HALF.sub(lambda m: dec((_word_or_digit(m.group(1)) + 0.5) * 12) + " inch", t)
    for pat, rep in _FOOT_WORDS: t = pat.sub(rep, t)
    t = _AND_FRAC.sub(lambda m: dec((1.0 if m.group(1).lower() == "an" else _word_or_digit(m.group(1)))
                                    + (0.75 if m.group(3) else _FRACWORD[m.group(2).lower()])) + " inch", t)
    t = _HALF_IN.sub("0.5 inch", t)
    t = _WORDFRAC.sub(lambda m: dec(WORD_FRACS[re.sub(r"\s", "-", m.group(1).lower())]) + " inch", t)
    t = _AN_INCH.sub("1 inch", t)                               # after the word fractions: "a quarter of an inch" must not leave a stray "1 inch"
    t = _TENS_UNITS.sub(lambda m: str(NUM_WORDS[m.group(1).lower()] + NUM_WORDS[m.group(2).lower()]), t)
    t = _WORDNUM_CTX.sub(lambda m: str(NUM_WORDS[m.group(1).lower()]), t)
    t = _NO_SIZE.sub(lambda m: f"No. {m.group(1)}", t)
    t = _SIZE_N.sub(lambda m: f"No. {m.group(1)}", t)
    t = _METRIC.sub(lambda m: f"M{m.group(1)}", t)
    t = _MIXED.sub(lambda m: dec(int(m.group(1)) + int(m.group(2)) / int(m.group(3))), t)
    t = _FRAC.sub(lambda m: dec(int(m.group(1)) / int(m.group(2))), t)
    t = _LEADDOT.sub(r"0.\1", t)
    t = _DIMX.sub(" x ", t)
    t = _FT_IN.sub(lambda m: dec(float(m.group(1)) * 12 + float(m.group(2))) + " inch", t)
    t = _FT.sub(lambda m: dec(float(m.group(1) or m.group(2)) * 12) + " inch", t)
    t = _MM_SCREW.sub(lambda m: "M" + dec(float(m.group(1))) + " ", t)
    t = _MM.sub(lambda m: dec(float(m.group(1)) / 25.4) + " inch", t)
    t = _CM.sub(lambda m: dec(float(m.group(1)) / 2.54) + " inch", t)
    t = _METERS.sub(lambda m: dec(float(m.group(1)) * 39.3701) + " inch", t)
    t = _M_ABBR.sub(lambda m: dec(float(m.group(1)) * 39.3701) + " inch", t)
    t = _GLUE.sub(r"\1 \2", t)
    return re.sub(r"\s+", " ", t).strip()

def numbers_in(text_norm: str):
    """All standalone numbers in already-normalized text (used for the 'is this value in the text?' check)."""
    return [float(x) for x in re.findall(r"(?<![\w.])\d+(?:\.\d+)?", text_norm)]

def tol_for(v: float) -> float:
    """How close counts as 'the same number'. Absolute 0.0075 in for normal sizes, but much tighter for tiny values
    (a 0.012" washer must not be confused with a 0.015" one)."""
    return min(NUM_TOL, max(0.0006, 0.05 * abs(v)))

# ---- quick self-tests (they run every time so a broken edit is caught immediately)
_tests = {
    'make a pipe gasket 1 1/16" ID and 3 7/8" OD. 1/16" thick': ['1.0625"', '3.875"', '0.0625"'],
    "They should be a sixteenth inch thick": ["0.0625 inch thick"],
    "3 and a half inch pipe with a seven and a half inch flange": ["3.5 inch pipe", "7.5 inch flange"],
    "fastened at four mount points, eight bolt holes": ["4 mount points", "8 bolt holes"],
    "4.62OD and 1.66ID, 3.75od, 3.5in": ["4.62 OD", "1.66 ID", "3.75 od", "3.5 in"],
    "washer for a triple aught screw": ["No. 000 screw"],
    "a quarter twenty screw": ["0.25 screw"],
    "for a no1 screw": ["No. 1 screw"],
    "spacer for a #8 screw": ["No. 8 screw"],
    "washer for a size ten screw": ["No. 10 screw"],
    "for a metric 3.9 bolt": ["M3.9 bolt"],
    'it is .0625" tall': ['0.0625" tall'],
    "id should be half an inch, make it an inch wide": ["0.5 inch", "1 inch wide"],
    "square base 3x11 inches, a 2x4x6 box": ["3 x 11 inches", "2 x 4 x 6 box"],
    "a nine inch pipe on a four inch flange with twenty four bolts": ["9 inch pipe", "4 inch flange", "24 bolts"],
    "one leg measuring 1 inch and one measuring 2": ["one leg"],           # 'one' must NOT be converted here
    "12 3/4 pipe with a 20 1/2 flange": ["12.75 pipe", "20.5 flange"],
    "washer for an inch and a half screw": ["1.5 inch screw"],
    "an inch and a quarter wide": ["1.25 inch wide"],
    "a quarter of an inch hole, an eighth of an inch thick": ["0.25 inch hole", "0.125 inch thick"],
    "make it an inch wide": ["1 inch wide"],
    "make a box thats 40inches by 12.73 inches and 1/2\" thick. i want it to be two feet tall": ["40 inches by 12.73 inches", "0.5\" thick", "24 inch tall"],
    "Box 4 x 8 in. 1/2\" thick 1.75\" tall": ["4 x 8 in.", "0.5\" thick", "1.75\" tall"],
    "2 ft long, 5' x 3', a foot and a half, half a foot, two and a half feet": ["24 inch long", "60 inch x 36 inch", "18 inch", "6 inch", "30 inch"],
    "5 feet 6 inches and 5'6\" and 1 1/2 feet": ["66 inch and 66 inch and 18 inch"],
    "300 mm wide, 2.5 cm thick, 1 meter tall, 2m long, 25.4 mm hole": ["11.811 inch wide", "0.9843 inch thick", "39.3701 inch tall", "78.7402 inch long", "1 inch hole"],
    "washer for a 6 mm screw, M5 washer, no. 8 screw": ["M6 screw", "M5 washer", "No. 8 screw"],
    "gasket for 4 and a quarter flange": ["4.25 inch flange"],
    "it should have a dozen holes": ["12 holes"],
    "a 12 and a half inch wide gasket": ["12.5 inch wide"],
}
for src, must in _tests.items():
    out = normalize_text(src)
    assert all(m in out for m in must), f"normalizer test failed:\n  in : {src}\n  out: {out}\n  need: {must}"
assert tol_for(0.84) == NUM_TOL and tol_for(0.0118) < 0.001 and tol_for(0.0625) < NUM_TOL
print("normalizer self-tests passed")

## 3. Standard schema, catalog tables, train/val/test split
* One JSON shape for every part (5 part types) (same key names wherever a concept is shared, e.g. `thickness_in`).
* **Catalog tables** are built from every sheet row: known sizes (used to snap `0.84` → `0.8438` for gaskets, washers and brackets only; container sizes are free-form so they are never snapped) and the washer size lookup (`M7` → its dimensions).
* With `GROUP_SPLIT = True`, a given pipe size / screw size never appears in both train and test, so test scores reflect genuinely unseen sizes.

In [ ]:
# ------------------------------------------------------------------ ONE STANDARD JSON SCHEMA FOR ALL PARTS
PART_TYPES = ["pipe_gasket", "washer", "bracket", "rect_container", "cyl_container"]
SCHEMA = {                                             # key order here = key order in the JSON the model learns to write
    "pipe_gasket": ["od_in", "id_in", "thickness_in", "hole_count"],
    "washer":      ["screw_size", "od_in", "id_in", "thickness_in"],
    "bracket":     ["length_a_in", "length_b_in", "height_in", "thickness_in"],   # height_in = bracket width (your column name)
    "rect_container": ["length_a_in", "length_b_in", "height_in", "thickness_in"],  # open-top box; thickness = wall (and floor) thickness
    "cyl_container":  ["id_in", "height_in", "thickness_in"],                       # open-top cup; id = INSIDE diameter
}
INT_FIELDS, STR_FIELDS = {"hole_count"}, {"screw_size"}
NUM_FIELDS = {pt: [f for f in fs if f not in STR_FIELDS] for pt, fs in SCHEMA.items()}

def canon_screw_size(s):
    """'No. 10'->'No. 10', 'M2.5'->'M2.5', '1 1/8"'->'1.125'. Returns None if it is not a recognisable screw size
    (the fine-tuned model can emit anything, e.g. a copy of the user's text, so this must never raise)."""
    s2 = normalize_text(str(s).replace('"', "").strip())
    m = re.fullmatch(r"(?i)m\s*(\d+(?:\.\d+)?)", s2)
    if m: return "M" + m.group(1)
    m = re.fullmatch(r"(?i)no\.?\s*(\d+)", s2)
    if m: return "No. " + m.group(1)
    if re.fullmatch(r"\d+(?:\.\d+)?", s2): return s2
    return None

def screw_in_text(sz, text_norm) -> bool:
    if not sz: return False
    if sz.startswith("M"):  return re.search(rf"\b{re.escape(sz)}\b", text_norm, re.I) is not None
    if sz.startswith("No"): return re.search(rf"\bno\.? ?{sz[4:]}\b", text_norm, re.I) is not None
    try: return any(abs(float(sz) - n) <= 1e-3 for n in numbers_in(text_norm))
    except ValueError: return False

def coerce_value(field, v):
    """Turn whatever came back (str/float/None/junk) into a clean typed value, or None."""
    if v is None or (isinstance(v, float) and math.isnan(v)) or (isinstance(v, str) and not v.strip()):
        return None
    try:
        if field in STR_FIELDS: return canon_screw_size(v)
        x = float(normalize_text(v)) if isinstance(v, str) else float(v)
        if field in INT_FIELDS:
            return int(round(x)) if abs(x - round(x)) < 1e-9 else None
        return round(x, 4)
    except (ValueError, TypeError):
        return None

def row_to_params(r) -> dict:
    pt = r["part_type"]
    p = {"part_type": pt}
    for f in SCHEMA[pt]:
        p[f] = coerce_value(f, r[f])
    return p

def params_to_json(p: dict) -> str:
    """Canonical compact JSON (fixed key order) - this exact string is what the fine-tuned model learns to output."""
    ordered = {"part_type": p["part_type"], **{f: p.get(f) for f in SCHEMA[p["part_type"]]}}
    return json.dumps(ordered, separators=(",", ":"))

# ---- every part_type in the sheet must be known to the schema
_unknown = set(raw_all.part_type.dropna()) - set(PART_TYPES)
assert not _unknown, f"spreadsheet has part types the notebook doesn't know: {_unknown}"
missing_types = set(PART_TYPES) - set(raw_all.part_type)
if missing_types: print("note: no rows in the sheet for", missing_types)

# ---- build the modelling table
data = df_all.copy()
data["text_norm"] = data["text"].map(normalize_text)
data["params"]    = data.apply(row_to_params, axis=1)
data["target"]    = data["params"].map(params_to_json)
assert data["params"].map(lambda p: all(v is not None for v in p.values())).all(), "some labelled rows have empty label cells"

# ---- catalog tables built from EVERY sheet row (including rows that have no text yet - geometry is still valid catalog data)
def _col(df, c): return pd.to_numeric(df[c], errors="coerce").dropna().round(4)

def _digits(v): return len(f"{v:.4f}".rstrip("0").split(".")[1]) if "." in f"{v:.4f}".rstrip("0") else 0

def canonical_values(values):
    """Your sheet stores some sizes at two precisions (4.62 AND 4.625, 0.84 AND 0.8438). Merge values that are within
    tol_for() of each other and keep the most precise one, so snapping has ONE catalog value per real size."""
    out, cluster = [], []
    for v in sorted(set(values)):
        if cluster and v - cluster[0] > tol_for(cluster[0]):
            out.append(max(cluster, key=lambda x: (_digits(x), x))); cluster = []
        cluster.append(v)
    if cluster: out.append(max(cluster, key=lambda x: (_digits(x), x)))
    return out

def _num_fields(pt): return [f for f in SCHEMA[pt] if f not in STR_FIELDS and f not in INT_FIELDS]
STANDARDS = {}                                          # known sizes per field. Used to snap (catalog parts only) and for range warnings
for _pt in PART_TYPES:
    _sub = raw_all[raw_all.part_type == _pt]
    STANDARDS[_pt] = {f: (canonical_values(_col(_sub, f)) if _pt in SNAP_PART_TYPES else sorted(set(_col(_sub, f))))
                      for f in _num_fields(_pt)}
_dupe_sizes = sum(len(set(_col(raw_all[raw_all.part_type == pt], f))) - len(canonical_values(_col(raw_all[raw_all.part_type == pt], f)))
                  for pt in ("pipe_gasket", "washer") for f in ["od_in", "id_in", "thickness_in"])
print(f"label precision check: {_dupe_sizes} sizes appear at two different precisions in your sheet (e.g. 4.62 vs 4.625)")
WASHER_LOOKUP = {}                                      # 'M7' -> {od_in, id_in, thickness_in}: fills in a washer when only the size is given
for _, r in raw_all[raw_all.part_type == "washer"].iterrows():
    assert canon_screw_size(r["screw_size"]), f"washer row {r['sheet_row']}: screw_size {r['screw_size']!r} is not a recognisable size"
    WASHER_LOOKUP.setdefault(canon_screw_size(r["screw_size"]).lower(), {f: round(float(r[f]), 4) for f in ["od_in", "id_in", "thickness_in"]})
RANGES = {pt: {f: (float(min(v)), float(max(v))) for f, v in fs.items()} for pt, fs in STANDARDS.items()}
print("washer sizes in lookup table:", len(WASHER_LOOKUP), "| example M7 ->", WASHER_LOOKUP.get("m7"))

# ---- train / val / test split
def _group_key(r):
    p = r["params"]
    if r["part_type"] == "pipe_gasket": return ("g", round(p["od_in"], 1), round(p["id_in"], 1))
    if r["part_type"] == "washer":      return ("w", p["screw_size"])
    if r["part_type"] == "bracket":     return ("b", p["length_a_in"], p["length_b_in"])
    if r["part_type"] == "rect_container": return ("r", p["length_a_in"])        # same length never in both train and test
    return ("c", p["id_in"])

def make_split(df):
    rng = random.Random(SEED)
    df = df.copy(); df["group"] = df.apply(_group_key, axis=1) if GROUP_SPLIT else np.arange(len(df))
    df["split"] = "train"
    for pt, sub in df.groupby("part_type"):
        groups = sorted(set(sub["group"]), key=str); rng.shuffle(groups)
        n_test, n_val, got = TEST_FRAC * len(sub), VAL_FRAC * len(sub), {"test": 0, "val": 0}
        for g in groups:
            rows = sub.index[sub["group"] == g]
            if got["test"] < n_test:  s = "test"
            elif got["val"] < n_val:  s = "val"
            else:                     s = "train"
            if s != "train": got[s] += len(rows)
            df.loc[rows, "split"] = s
    return df

flagged_ids = set()    # filled in by the audit cell below
data = make_split(data)
print(data.groupby(["part_type", "split"]).size().unstack(fill_value=0))
assert set(data.loc[data.split == "train", "part_type"]) == set(PART_TYPES), "train split is missing a part type"

## 4. Audit — do the labels agree with their own text?
A label whose numbers don't appear in its text teaches the model to invent values. The list below gives the sheet name and **Excel row** of each suspect row so you can fix it in the spreadsheet (or set `EXCLUDE_AUDIT_FLAGGED = True`).

In [ ]:
# ------------------------------------------------------------------ AUDIT: do the label numbers actually appear in the text?
def audit_row(r):
    nums, bad = numbers_in(r["text_norm"]), []
    for f in NUM_FIELDS[r["part_type"]]:
        v = r["params"][f]
        ok = (int(v) in [int(n) for n in nums if n == int(n)]) if f in INT_FIELDS else any(abs(v - n) <= tol_for(v) for n in nums)
        if not ok: bad.append((f, v))
    if r["part_type"] == "washer":
        sz = r["params"]["screw_size"]
        if not screw_in_text(sz, r["text_norm"]): bad.append(("screw_size", sz))
    return bad

data["audit_bad"] = data.apply(audit_row, axis=1)
audit_report = data[data.audit_bad.map(len) > 0][["sheet", "sheet_row", "text", "audit_bad"]].copy()
audit_report["numbers_in_text"] = data.loc[audit_report.index, "text_norm"].map(numbers_in)
pd.set_option("display.max_colwidth", 120, "display.width", 220)
print(f"{len(audit_report)} of {len(data)} rows have a label value that is NOT in their own text.")
print("Fix these in the spreadsheet (sheet + Excel row shown), or set EXCLUDE_AUDIT_FLAGGED = True.\n")
show_df(audit_report)

dups = data[data.duplicated("text_norm", keep=False)]
print(f"\nduplicate texts after normalization: {len(dups)}")

if EXCLUDE_AUDIT_FLAGGED and len(audit_report):
    data = data.drop(index=audit_report.index).reset_index(drop=True)
    print(f"dropped {len(audit_report)} flagged rows; {len(data)} remain")
    data = make_split(data.drop(columns=["split", "group"]))
train_df, val_df, test_df = (data[data.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
print(f"\ntrain {len(train_df)} | val {len(val_df)} | test {len(test_df)}")

## 5. Validation layer + CAD generator (deterministic — no ML)
**Validation** turns the model's JSON into the table the user sees. Each value gets a status:

| status | meaning |
|---|---|
| ✅ extracted | the model found it and the number really is in your text |
| ⚠️ not found in your text | the model produced it but it isn't in your description (possible hallucination) |
| ❌ missing | not extracted |
| 🔷 snapped / standard size | matched to a catalog size, or filled from the washer size table (`M7` alone is enough) — values you state explicitly are never overwritten |
| 🟨 assumed default | needed for CAD but not in your data (gasket bolt-circle diameter and bolt-hole size) — editable |
| ✏️ edited | changed by the user |

Hard rules (positive values, OD > ID, bolt holes inside the ring, …) block CAD generation; range warnings don't.

In [ ]:
# ------------------------------------------------------------------ STAGE 3: VALIDATION (rules first, optional off-the-shelf LLM second)
# Pipeline box: "Validation checks for negative values, missing data, out of range, etc."
STATUS_LABEL = {
    "extracted":   "✅ extracted",
    "missing":     "❌ missing",
    "not_in_text": "⚠️ not found in your text",
    "snapped":     "🔷 snapped to catalog size",
    "lookup":      "🔷 from standard-size table",
    "default":     "🟨 assumed default",
    "edited":      "✏️ edited by you",
    "llm_flag":    "⚠️ verifier disagrees",
}
DERIVED_GASKET = ["bolt_circle_in", "bolt_hole_in"]      # not in the dataset -> derived defaults the user can override

def fmt_value(v):
    if v is None: return ""
    if isinstance(v, int) and not isinstance(v, bool): return str(v)
    if isinstance(v, float): return dec(v)
    return str(v)

def snap_value(part_type, field, v):
    if not SNAP_TO_STANDARD or part_type not in SNAP_PART_TYPES or field not in STANDARDS[part_type]: return v
    best = min(STANDARDS[part_type][field], key=lambda s: abs(s - v))
    return best if abs(best - v) <= tol_for(v) else v

def default_bolt_geometry(od, idd, n=None):
    if od <= idd: return None, None                               # invalid OD/ID: the rules will report it
    w = (od - idd) / 2                                            # radial width of the gasket ring
    bc = (od + idd) / 2
    hd = min(max(0.4 * w, 0.19), 1.25, 0.6 * w)
    if n and n >= 2: hd = min(hd, 0.6 * bc * math.sin(math.pi / n))   # many bolts -> smaller default holes so neighbours never touch
    return round(bc, 4), round(hd, 4)

def make_table(part_type, text_norm, raw_params):
    """model output (dict or None) -> list of row dicts {field, value, status, note}"""
    nums, raw = numbers_in(text_norm), (raw_params or {})
    rows = [{"field": f, "value": coerce_value(f, raw.get(f)), "status": "extracted", "note": ""} for f in SCHEMA[part_type]]
    R = {r["field"]: r for r in rows}
    for r in rows:
        if r["value"] is None: r.update(status="missing", note="the model did not find this in your text")

    std = WASHER_LOOKUP.get((R["screw_size"]["value"] or "").lower()) if part_type == "washer" else None
    if std:                                                        # fill only what's missing (partial overrides survive)
        for f, sv in std.items():
            if R[f]["value"] is None:
                R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")

    for f in SCHEMA[part_type]:                                    # grounding check + snapping for model-extracted values
        r, v = R[f], R[f]["value"]
        if r["status"] != "extracted": continue
        if f in STR_FIELDS: grounded = screw_in_text(v, text_norm)
        elif f in INT_FIELDS: grounded = v in [int(n) for n in nums if n == int(n)]
        else: grounded = any(abs(v - n) <= tol_for(v) for n in nums)
        if not grounded:
            r.update(status="not_in_text", note="this value is not in your description - please check it"); continue
        if f not in STR_FIELDS and f not in INT_FIELDS:
            sv = snap_value(part_type, f, v)
            if sv != v: r.update(value=sv, status="snapped", note=f"you wrote ~{dec(v)}; catalog size is {dec(sv)}")
        if std and f in std and abs(R[f]["value"] - std[f]) > tol_for(std[f]):
            r["note"] = (r["note"] + " | " if r["note"] else "") + f"differs from standard {R['screw_size']['value']} ({dec(std[f])})"

    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"], R["hole_count"]["value"])
        rows += [{"field": "bolt_circle_in", "value": bc, "status": "default", "note": "assumed: halfway between ID and OD (not in your data)"},
                 {"field": "bolt_hole_in",   "value": hd, "status": "default", "note": "assumed hole size (not in your data)"}]
    return rows

def rows_to_params(part_type, rows):
    p = {"part_type": part_type}
    for r in rows: p[r["field"]] = coerce_value(r["field"], r["value"])
    return p

def apply_edits(part_type, old_rows, ui_values):
    """ui_values: {field: text typed in the table}. Only changed cells get 'edited'; everything else keeps its status."""
    rows, edited = copy.deepcopy(old_rows), set()
    for r in rows:
        if r["field"] not in ui_values: continue
        nv = coerce_value(r["field"], ui_values[r["field"]])
        if nv != r["value"]:
            r.update(value=nv, status="edited" if nv is not None else "missing", note="changed by you" if nv is not None else "cleared")
            edited.add(r["field"])
    R = {r["field"]: r for r in rows}
    if part_type == "washer" and R["screw_size"]["value"]:         # user typed a size -> fill blanks from the standard table
        std = WASHER_LOOKUP.get(R["screw_size"]["value"].lower())
        for f, sv in (std or {}).items():
            if R[f]["value"] is None: R[f].update(value=sv, status="lookup", note=f"standard size for {R['screw_size']['value']}")
    if part_type == "pipe_gasket" and R["od_in"]["value"] and R["id_in"]["value"]:
        bc, hd = default_bolt_geometry(R["od_in"]["value"], R["id_in"]["value"], R["hole_count"]["value"])
        for f, dv in zip(DERIVED_GASKET, (bc, hd)):                # follow OD/ID edits unless the user set them
            if f not in R: rows.append({"field": f, "value": dv, "status": "default", "note": "assumed (not in your data)"}); R[f] = rows[-1]
            elif f not in edited and R[f]["status"] == "default": R[f]["value"] = dv
    return rows, edited

def check_rules(part_type, p):
    """hard errors block CAD generation; warnings are just shown"""
    issues = []
    err  = lambda f, m: issues.append({"level": "error", "field": f, "msg": m})
    warn = lambda f, m: issues.append({"level": "warning", "field": f, "msg": m})
    def check_present(fields):
        for f in fields:
            v = p.get(f)
            if v is None: err(f, "missing - fill it in the table")
            elif f not in STR_FIELDS and v <= 0: err(f, "must be greater than zero")
    check_present(SCHEMA[part_type])
    if issues: return issues
    if part_type in ("pipe_gasket", "washer") and p["od_in"] <= p["id_in"]:
        err("od_in", "outer diameter must be larger than inner diameter"); return issues
    if part_type == "bracket" and p["thickness_in"] >= min(p["length_a_in"], p["length_b_in"]):
        err("thickness_in", "thickness must be smaller than both leg lengths"); return issues
    if part_type == "rect_container":
        inner = min(p["length_a_in"], p["length_b_in"]) - 2 * p["thickness_in"] if RECT_DIMS_ARE_OUTER else min(p["length_a_in"], p["length_b_in"])
        if inner <= 0: err("thickness_in", "walls are too thick: nothing is left inside the container"); return issues
        if p["height_in"] <= p["thickness_in"]: err("height_in", "height must be greater than the floor/wall thickness"); return issues
    if part_type == "cyl_container" and p["height_in"] <= p["thickness_in"]:
        err("height_in", "height must be greater than the floor/wall thickness"); return issues
    if part_type == "pipe_gasket":
        check_present(DERIVED_GASKET)
        if issues: return issues
        n, bc, hd = p["hole_count"], p["bolt_circle_in"], p["bolt_hole_in"]
        if not 1 <= n <= 64: err("hole_count", "expected between 1 and 64 bolt holes"); return issues
        if bc / 2 - hd / 2 <= p["id_in"] / 2 or bc / 2 + hd / 2 >= p["od_in"] / 2:
            err("bolt_circle_in", "bolt holes would fall outside the gasket ring - adjust bolt circle / hole size"); return issues
        if n >= 2 and bc * math.sin(math.pi / n) <= hd:
            err("bolt_hole_in", "neighbouring bolt holes overlap - reduce the hole size or the number of holes"); return issues
        if n >= 2 and bc * math.sin(math.pi / n) < 1.1 * hd:
            warn("hole_count", "bolt holes nearly touch (very thin material between them) - consider a smaller hole size")
    for f, (lo, hi) in RANGES[part_type].items():
        if p[f] < 0.5 * lo or p[f] > 2 * hi: warn(f, f"unusual size (catalog range is {dec(lo)} to {dec(hi)} in)")
    return issues

def llm_verify(part_type, text_norm, params):
    """OPTIONAL off-the-shelf check: ask the UNTUNED base model which values look wrong. Small models are weak verifiers -
    that's why the deterministic rules above do the heavy lifting and this is off by default."""
    shown = {k: params[k] for k in SCHEMA[part_type] if params.get(k) is not None}
    prompt = (f"Description: {text_norm}\nExtracted values: {json.dumps(shown)}\n"
              "List the keys whose value does NOT match the description, as a JSON list ([] if all match).\nAnswer: ")
    out = generate([prompt], use_adapter=False, max_new_tokens=40)[0]
    m = re.search(r"\[[^\]]*\]", out)
    try:    return [k for k in json.loads(m.group(0)) if k in SCHEMA[part_type]] if m else []
    except Exception: return []

def rows_to_df(rows):
    return pd.DataFrame([[r["field"], fmt_value(r["value"]), STATUS_LABEL[r["status"]], r["note"]] for r in rows],
                        columns=["field", "value", "status", "note"])

**Add features (holes):** the schema, position conventions, centring defaults and geometric checks for holes on brackets and containers. A hole must lie fully inside its wall/leg, clear of the bend, rims, floor and its neighbours.

In [ ]:
# ------------------------------------------------------------------ ADD FEATURES: holes at user-specified locations
# Available for L-brackets and both containers (gaskets/washers have no add-on features).
# Everything is in inches. Positions follow the conventions in HOLE_SPEC (shown to the user in the GUI).
MAX_HOLES = 12
HOLE_EPS = 0.02            # in: the cutter starts/ends slightly outside the wall so cuts never leave paper-thin skins

HOLE_SPEC = {
    "bracket": {
        "faces": ["leg_a", "leg_b"],
        "keys": ["face", "from_end_in", "across_in", "diameter_in"],
        "labels": ["Leg", "Distance from the leg's free end (in)", "Height above the bottom face (in)", "Hole diameter (in)"],
        "hint": "leg_a runs left-to-right in the preview and leg_b runs away from you. Distance is measured from the leg's free end to the "
                "hole centre. Height is measured up from the bottom face (the face lying on the print bed). Leave a position blank to centre the hole.",
    },
    "rect_container": {
        "faces": ["front", "back", "left", "right"],
        "keys": ["face", "along_in", "up_in", "diameter_in"],
        "labels": ["Wall", "Distance from the wall's left end (in)", "Height above the bottom (in)", "Hole diameter (in)"],
        "hint": "Walls are named as seen in the preview: front faces you, back is behind, left/right are the sides (length_a runs left-to-right). "
                "Distance is measured from the wall's left end as you look at it from outside. Leave a position blank to centre the hole.",
    },
    "cyl_container": {
        "faces": ["side"],
        "keys": ["face", "angle_deg", "up_in", "diameter_in"],
        "labels": ["Surface", "Angle around the cup (deg)", "Height above the bottom (in)", "Hole diameter (in)"],
        "hint": "Angle is measured counter-clockwise looking down from above: 0 = right, 90 = back, 180 = left, 270 = front. "
                "Leave the height blank to centre the hole vertically.",
    },
}
HOLE_PART_TYPES = set(HOLE_SPEC)
_FACE_ALIAS = {"rear": "back", "a": "leg_a", "b": "leg_b", "leg_1": "leg_a", "leg_2": "leg_b"}
_DIRECTION_WORDS = re.compile(r"\b(front|back|rear|behind|left|right|opposite|north|south|east|west)\b", re.I)

def rect_outer(p):
    a, b, t = p["length_a_in"], p["length_b_in"], p["thickness_in"]
    return (a, b) if RECT_DIMS_ARE_OUTER else (a + 2 * t, b + 2 * t)

def leg_length(p, face): return p["length_a_in"] if face == "leg_a" else p["length_b_in"]
def wall_length(p, face):
    oa, ob = rect_outer(p); return oa if face in ("front", "back") else ob

def hole_defaults(pt, p, face):
    """positions used when the user leaves one blank = centred (angle has no sensible default)"""
    t, k = p["thickness_in"], HOLE_SPEC[pt]["keys"]
    if pt == "bracket":        return {k[1]: round((leg_length(p, face) - t) / 2, 4), k[2]: round(p["height_in"] / 2, 4)}
    if pt == "rect_container": return {k[1]: round(wall_length(p, face) / 2, 4), k[2]: round((p["height_in"] + t) / 2, 4)}
    return {k[2]: round((p["height_in"] + t) / 2, 4)}

def coerce_face(pt, v):
    if v is None: return None
    s = re.sub(r"[\s\-]+", "_", str(v).strip().lower()); s = _FACE_ALIAS.get(s, s)
    return s if s in HOLE_SPEC[pt]["faces"] else None

def _cell(value, status, note=""): return {"value": value, "status": status, "note": note}

def _grounded(pt, key, v, text_norm):
    nums = numbers_in(text_norm)
    if key == "angle_deg":
        return any(abs(((v - n + 180) % 360) - 180) <= 0.5 for n in nums) or bool(_DIRECTION_WORDS.search(text_norm))
    return any(abs(v - n) <= tol_for(v) for n in nums)

def make_hole_rows(pt, p, text_norm, raw_holes):
    """model output (list of dicts, or None) -> table rows. Blank positions become 'centred' defaults; values not in the text are flagged."""
    spec, rows = HOLE_SPEC[pt], []
    for raw in (raw_holes if isinstance(raw_holes, list) else [])[:MAX_HOLES]:
        raw = raw if isinstance(raw, dict) else {}
        k = spec["keys"]; face = coerce_face(pt, raw.get(k[0]))
        if face is None and len(spec["faces"]) == 1: face = spec["faces"][0]
        cells = {"face": _cell(face, "extracted") if face else _cell(None, "missing", "which wall / leg?")}
        defaults = hole_defaults(pt, p, face) if face else {}
        for ck, key in zip(("p1", "p2", "d"), k[1:]):
            v = coerce_value(key, raw.get(key))
            if v is None:
                if key in defaults: cells[ck] = _cell(defaults[key], "default", "centred (you didn't give a position)")
                else:               cells[ck] = _cell(None, "missing", "not found in your description")
            elif not _grounded(pt, key, v, text_norm):
                cells[ck] = _cell(v, "not_in_text", "this number is not in your description - please check it")
            else:
                cells[ck] = _cell(v, "extracted")
        rows.append({"cells": cells})
    return rows

def resolve_hole_rows(pt, p, ui_rows, prev_rows=None):
    """UI rows ({face,p1,p2,d} as typed text) -> (rows with statuses, holes dicts). Blank positions are centred; changed cells become 'edited'."""
    spec, rows, holes = HOLE_SPEC[pt], [], []
    k = spec["keys"]
    for i, ur in enumerate((ui_rows or [])[:MAX_HOLES]):
        prev = prev_rows[i]["cells"] if prev_rows and i < len(prev_rows) else None
        face = coerce_face(pt, ur.get("face"))
        if face is None and len(spec["faces"]) == 1: face = spec["faces"][0]
        face_changed = bool(prev) and prev["face"]["value"] != face
        cells = {"face": (_cell(face, prev["face"]["status"] if prev and not face_changed else "edited") if face else _cell(None, "missing", "choose a wall / leg"))}
        defaults = hole_defaults(pt, p, face) if face else {}
        h = {k[0]: face}
        for ck, key in zip(("p1", "p2", "d"), k[1:]):
            v = coerce_value(key, ur.get(ck)); pc = prev.get(ck) if prev else None
            untouched = pc is not None and pc["value"] is not None and v is not None and abs(v - pc["value"]) <= 1e-9
            if untouched and pc["status"] == "default" and face_changed: v = None                # an UNTOUCHED centred value belonged to the old face -> recompute
            if v is None:
                if key in defaults: cells[ck] = _cell(defaults[key], "default", "centred (blank = centred)")
                else:               cells[ck] = _cell(None, "missing", "required")
            elif pc and pc["value"] is not None and abs(v - pc["value"]) <= 1e-9:
                cells[ck] = _cell(v, pc["status"], pc["note"])                                    # untouched -> keep its original status
            else:
                cells[ck] = _cell(v, "edited", "entered/changed by you")
            h[key] = cells[ck]["value"]
        rows.append({"cells": cells}); holes.append(h)
    return rows, holes

def check_holes(pt, p, holes):
    """hard errors block the build; warnings are shown. Every hole must lie fully inside its wall/leg and clear of its neighbours."""
    issues, k = [], HOLE_SPEC[pt]["keys"]
    err  = lambda i, f, m: issues.append({"level": "error",   "hole": i, "field": f, "msg": m})
    warn = lambda i, f, m: issues.append({"level": "warning", "hole": i, "field": f, "msg": m})
    if len(holes) > MAX_HOLES: err(None, None, f"at most {MAX_HOLES} holes")
    for i, h in enumerate(holes):
        for key in k:
            if h.get(key) is None: err(i, key, "missing - fill it in the table")
    if issues: return issues
    t, hgt = p["thickness_in"], p["height_in"]
    pos = []                                                       # (face, u, v) in a flat 2-D layout of each face, for the overlap test
    for i, h in enumerate(holes):
        d = h[k[3]]; r = d / 2; u, v = h[k[1]], h[k[2]]
        if d <= 0: err(i, k[3], "diameter must be greater than zero"); pos.append(None); continue
        if pt == "bracket":
            free = leg_length(p, h["face"]) - t
            if u - r < 0:      err(i, k[1], "hole breaks through the leg's free end")
            if u + r > free:   err(i, k[1], f"hole runs into the bend (this leg is only {dec(free)} in long past the bend)")
            if v - r < 0 or v + r > hgt: err(i, k[2], f"hole breaks through the bracket's edge (bracket is {dec(hgt)} in wide)")
            edge = min(u - r, free - u - r, v - r, hgt - v - r)
            if edge >= 0 and edge < 0.05: warn(i, k[1], "less than 0.05 in of material between the hole and an edge")
            pos.append((h["face"], u, v))
        else:
            if pt == "rect_container":
                L = wall_length(p, h["face"])
                if u - r < 0 or u + r > L: err(i, k[1], f"hole runs off the wall (wall is {dec(L)} in long)")
                elif u - r < t or u + r > L - t: warn(i, k[1], "hole is within one wall-thickness of a corner")
                x = u
            else:
                R = p["id_in"] / 2 + t
                if not 0 <= u <= 360: err(i, k[1], "angle must be between 0 and 360 degrees")
                x = u                                              # converted to arc length below
            if v - r < 0:      err(i, k[2], "hole breaks through the bottom")
            elif v + r > hgt:  err(i, k[2], f"hole breaks through the top rim (container is {dec(hgt)} in tall)")
            elif v - r < t:    warn(i, k[2], "hole dips into the floor - raise it to keep the cut inside the wall")
            pos.append((h["face"], x, v))
    for i in range(len(holes)):
        for j in range(i + 1, len(holes)):
            if pos[i] is None or pos[j] is None or pos[i][0] != pos[j][0]: continue
            if pt == "cyl_container":
                da = abs(pos[i][1] - pos[j][1]); da = min(da, 360 - da)
                dist = math.hypot(math.radians(da) * (p["id_in"] / 2 + t), pos[i][2] - pos[j][2])
            else:
                dist = math.hypot(pos[i][1] - pos[j][1], pos[i][2] - pos[j][2])
            if dist <= (holes[i][k[3]] + holes[j][k[3]]) / 2:
                err(j, k[1], f"hole {j + 1} overlaps hole {i + 1}")
    return issues

def holes_to_json(pt, holes):
    k = HOLE_SPEC[pt]["keys"]
    return json.dumps({"holes": [{key: h.get(key) for key in k} for h in holes]}, separators=(",", ":"))

def audit_holes(pt, holes, text_norm):
    """dataset check: every numeric hole value that is NOT blank must be readable in its own text"""
    k, bad = HOLE_SPEC[pt]["keys"], []
    for i, h in enumerate(holes):
        for key in k[1:]:
            v = h.get(key)
            if v is not None and not _grounded(pt, key, v, text_norm): bad.append((i, key, v))
    return bad
# ---- regression tests for the edit logic (these caught a real bug: a typed value must never be replaced by a default)
_bp = {"length_a_in": 4.0, "length_b_in": 2.0, "height_in": 1.0, "thickness_in": 0.125}
_prev = [{"cells": {"face": _cell("leg_a", "extracted"), "p1": _cell(1.9375, "default"), "p2": _cell(0.5, "default"), "d": _cell(0.25, "extracted")}}]
_r, _h = resolve_hole_rows("bracket", _bp, [{"face": "leg_b", "p1": "1.9375", "p2": "0.95", "d": "0.25"}], _prev)      # face changed, p1 untouched, p2 typed by the user
assert _h[0]["across_in"] == 0.95 and _r[0]["cells"]["p2"]["status"] == "edited", _h                               # typed value survives
assert _h[0]["from_end_in"] == round((2.0 - 0.125) / 2, 4) and _r[0]["cells"]["p1"]["status"] == "default", _h      # untouched centred value is recomputed for the new leg
_r, _h = resolve_hole_rows("bracket", _bp, [{"face": "leg_a", "p1": "", "p2": "", "d": "0.25"}], None)
assert _h[0]["from_end_in"] == 1.9375 and _h[0]["across_in"] == 0.5, _h                                           # blank = centred
_CARDINAL_WORDS = {"front": 270, "back": 90, "rear": 90, "left": 180, "right": 0}      # cup directions -> angle (matches HOLE_SPEC hint)
def semantic_hole_problems(pt, holes, text_norm):
    """beyond 'numbers are present': named walls / directions must agree with the labels (rect walls must be named; cup directions must match angles)"""
    t, probs, k = text_norm.lower(), [], HOLE_SPEC[pt]["keys"]
    if pt == "rect_container":
        for f in {h["face"] for h in holes}:
            if not re.search(rf"\b{'(?:back|rear)' if f == 'back' else f}\b", t): probs.append(f"wall '{f}' is not named in the text")
    if pt == "cyl_container":
        angs = {float(h[k[1]]) for h in holes if h.get(k[1]) is not None}
        named = {w for w in _CARDINAL_WORDS if re.search(rf"\b{w}\b", t)}
        for w in named:
            if _CARDINAL_WORDS[w] not in angs: probs.append(f"text says '{w}' but no hole is at {_CARDINAL_WORDS[w]} degrees")
        nums = numbers_in(text_norm)
        for a in angs & {0.0, 90.0, 180.0, 270.0}:
            if a not in nums and not any(_CARDINAL_WORDS[w] == a for w in named): probs.append(f"angle {a:g} is neither written as a number nor named")
    return probs
print("hole schema ready for:", sorted(HOLE_PART_TYPES))

**CAD:** parameters (inches) → CadQuery solid → scaled ×25.4 → STL in mm. The self-test checks that volumes match the maths, which proves the bolt holes are really cut.

In [ ]:
# ------------------------------------------------------------------ STAGE 4: JSON -> CAD -> STL  (deterministic, no ML)
import cadquery as cq, trimesh
IN_TO_MM = 25.4        # STL has no units and slicers assume mm, so geometry built in inches is scaled once, at export

def build_gasket(p):
    od, idd, t, n = p["od_in"], p["id_in"], p["thickness_in"], int(p["hole_count"])
    bc, hd = p["bolt_circle_in"], p["bolt_hole_in"]
    ring = cq.Workplane("XY").circle(od / 2).circle(idd / 2).extrude(t)
    pts = [((bc / 2) * math.cos(2 * math.pi * i / n), (bc / 2) * math.sin(2 * math.pi * i / n)) for i in range(n)]
    holes = cq.Workplane("XY").workplane(offset=-t).pushPoints(pts).circle(hd / 2).extrude(3 * t)   # oversized so it cuts clean through
    return ring.cut(holes)

def build_washer(p):
    return cq.Workplane("XY").circle(p["od_in"] / 2).circle(p["id_in"] / 2).extrude(p["thickness_in"])

def build_bracket(p):
    a, b, w, t = p["length_a_in"], p["length_b_in"], p["height_in"], p["thickness_in"]
    return cq.Workplane("XY").polyline([(0, 0), (a, 0), (a, t), (t, t), (t, b), (0, b)]).close().extrude(w)   # L profile, extruded by the width

def build_rect_container(p):
    """open-top box. Walls and floor are both `thickness` thick. length_a along X, length_b along Y, height along Z."""
    a, b, h, t = p["length_a_in"], p["length_b_in"], p["height_in"], p["thickness_in"]
    oa, ob = (a, b) if RECT_DIMS_ARE_OUTER else (a + 2 * t, b + 2 * t)
    outer = cq.Workplane("XY").box(oa, ob, h, centered=(True, True, False))
    cavity = cq.Workplane("XY").workplane(offset=t).box(oa - 2 * t, ob - 2 * t, h, centered=(True, True, False))   # runs past the top -> open
    return outer.cut(cavity)

def build_cyl_container(p):
    """open-top cup. id_in is the INSIDE diameter, so outside diameter = id + 2*thickness. Floor = wall thickness."""
    idd, h, t = p["id_in"], p["height_in"], p["thickness_in"]
    outer = cq.Workplane("XY").circle(idd / 2 + t).extrude(h)
    cavity = cq.Workplane("XY").workplane(offset=t).circle(idd / 2).extrude(h)
    return outer.cut(cavity)

BUILDERS = {"pipe_gasket": build_gasket, "washer": build_washer, "bracket": build_bracket,
            "rect_container": build_rect_container, "cyl_container": build_cyl_container}

def hole_cutters(pt, p, holes):
    """one cylinder per hole, positioned from the user's numbers (see HOLE_SPEC for the conventions). Each cutter spans only the wall it
    pierces (plus a small overshoot), so it can never cut the opposite wall."""
    k, e, t, out = HOLE_SPEC[pt]["keys"], HOLE_EPS, p["thickness_in"], []
    V = cq.Vector
    for h in holes:
        r, u, v, face = h[k[3]] / 2, h[k[1]], h[k[2]], h["face"]
        if pt == "bracket":                                   # L profile lies flat in XY; leg_a along +X, leg_b along +Y; width is Z
            if face == "leg_a": pnt, d = V(p["length_a_in"] - u, -e, v), V(0, 1, 0)
            else:               pnt, d = V(-e, p["length_b_in"] - u, v), V(1, 0, 0)
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, pnt, d))
        elif pt == "rect_container":                          # centred in XY; front = -Y wall; positions measured from the wall's left end seen from outside
            oa, ob = rect_outer(p)
            pnt, d = {"front": (V(-oa / 2 + u, -ob / 2 - e, v), V(0, 1, 0)),
                      "back":  (V(oa / 2 - u, ob / 2 + e, v),   V(0, -1, 0)),
                      "left":  (V(-oa / 2 - e, ob / 2 - u, v),  V(1, 0, 0)),
                      "right": (V(oa / 2 + e, -ob / 2 + u, v),  V(-1, 0, 0))}[face]
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, pnt, d))
        else:                                                 # cup: radial hole at an angle (0 = +X, counter-clockwise from above)
            R, th = p["id_in"] / 2 + t, math.radians(u)
            out.append(cq.Solid.makeCylinder(r, t + 2 * e, V((R + e) * math.cos(th), (R + e) * math.sin(th), v), V(-math.cos(th), -math.sin(th), 0)))
    return out

def build_stl(params: dict, out_path: str, holes=None) -> dict:
    wp = BUILDERS[params["part_type"]](params)
    if holes:
        wp = wp.cut(cq.Compound.makeCompound(hole_cutters(params["part_type"], params, holes)))
    solid = wp.val().scale(IN_TO_MM)
    cq.exporters.export(solid, str(out_path), tolerance=0.01, angularTolerance=0.05)
    m = trimesh.load(str(out_path))
    return {"watertight": bool(m.is_watertight), "volume_mm3": round(float(m.volume), 2), "size_mm": [round(float(x), 2) for x in m.extents]}

# ---- tests: geometry must match the maths (this is how we know the holes were REALLY cut)
def _cuin(x): return x * IN_TO_MM ** 3
_g = {"part_type": "pipe_gasket", "od_in": 3.5, "id_in": 0.8438, "thickness_in": 0.0625, "hole_count": 4, "bolt_circle_in": 2.1719, "bolt_hole_in": 0.5325}
_i = build_stl(_g, WORKDIR / "test_gasket.stl")
_expect = _cuin(math.pi / 4 * (3.5**2 - 0.8438**2) * 0.0625 - 4 * math.pi / 4 * 0.5325**2 * 0.0625)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01, (_i, _expect)
_w = {"part_type": "washer", "od_in": 0.5512, "id_in": 0.2913, "thickness_in": 0.063}
_i = build_stl(_w, WORKDIR / "test_washer.stl")
assert _i["watertight"] and abs(_i["size_mm"][0] - 0.5512 * 25.4) < 0.05, _i
_b = {"part_type": "bracket", "length_a_in": 2.0, "length_b_in": 1.5, "height_in": 0.5, "thickness_in": 0.125}
_i = build_stl(_b, WORKDIR / "test_bracket.stl")
_expect = _cuin((2.0 * 0.125 + (1.5 - 0.125) * 0.125) * 0.5)
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and sorted(_i["size_mm"], reverse=True)[0] == round(2.0 * 25.4, 2), (_i, _expect)
_r = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
_i = build_stl(_r, WORKDIR / "test_rect.stl")
_a, _b, _h, _t = 4.5, 2.88, 1.67, 0.092
_expect = _cuin(_a * _b * _h - (_a - 2 * _t) * (_b - 2 * _t) * (_h - _t)) if RECT_DIMS_ARE_OUTER else None
if _expect: assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and abs(_i["size_mm"][2] - _h * 25.4) < 0.05, (_i, _expect)
_cy = {"part_type": "cyl_container", "id_in": 2.75, "height_in": 4.0, "thickness_in": 0.0963}
_i = build_stl(_cy, WORKDIR / "test_cyl.stl")
_expect = _cuin(math.pi * ((2.75 / 2 + 0.0963) ** 2 * 4.0 - (2.75 / 2) ** 2 * (4.0 - 0.0963)))
assert _i["watertight"] and abs(_i["volume_mm3"] - _expect) / _expect < 0.01 and abs(_i["size_mm"][0] - (2.75 + 2 * 0.0963) * 25.4) < 0.05, (_i, _expect)
def _check_holes_cut(pt, p, holes, centre_pts, expect_each):
    """for every hole: the point at its centre (mid-wall) must be EMPTY, a point 2 radii away along the wall must be SOLID, and the
    removed volume must match the maths. This is what proves the hole is where we said, on the face we said."""
    base = _cuin_vol(BUILDERS[pt](p))
    wp = BUILDERS[pt](p).cut(cq.Compound.makeCompound(hole_cutters(pt, p, holes)))
    for (cx, cy, cz), (sx, sy, sz) in centre_pts:
        assert not wp.val().isInside(cq.Vector(cx, cy, cz)), ("hole centre is still solid", pt, (cx, cy, cz))
        assert wp.val().isInside(cq.Vector(sx, sy, sz)), ("material next to the hole is missing", pt, (sx, sy, sz))
    removed = base - _cuin_vol(wp)
    assert abs(removed - expect_each * len(holes)) / (expect_each * len(holes)) < 0.03, (pt, removed, expect_each * len(holes))
def _cuin_vol(w): return w.val().Volume()      # still in cubic inches here (scaled only at export)

_t, _r = 0.125, 0.125
_bp = {"part_type": "bracket", "length_a_in": 4.0, "length_b_in": 2.0, "height_in": 1.0, "thickness_in": _t}
_ha = {"face": "leg_a", "from_end_in": 1.0, "across_in": 0.4, "diameter_in": 0.25}
_hb = {"face": "leg_b", "from_end_in": 0.75, "across_in": 0.6, "diameter_in": 0.25}
_check_holes_cut("bracket", _bp, [_ha, _hb], [((3.0, _t / 2, 0.4), (3.0, _t / 2, 0.4 + 2 * _r)), ((_t / 2, 1.25, 0.6), (_t / 2, 1.25, 0.6 + 2 * _r))], math.pi * _r**2 * _t)
_rp = {"part_type": "rect_container", "length_a_in": 4.5, "length_b_in": 2.88, "height_in": 1.67, "thickness_in": 0.092}
_hs = [{"face": f, "along_in": 1.0, "up_in": 0.9, "diameter_in": 0.25} for f in ("front", "back", "left", "right")]
_oa, _ob, _wt = 4.5, 2.88, 0.092
_pts = [((-_oa / 2 + 1.0, -_ob / 2 + _wt / 2, 0.9), (-_oa / 2 + 1.0 + 0.3, -_ob / 2 + _wt / 2, 0.9)),       # front: 1.0 from the left end seen from outside (-X end)
        ((_oa / 2 - 1.0, _ob / 2 - _wt / 2, 0.9), (_oa / 2 - 1.0 - 0.3, _ob / 2 - _wt / 2, 0.9)),            # back: left end seen from outside is the +X end
        ((-_oa / 2 + _wt / 2, _ob / 2 - 1.0, 0.9), (-_oa / 2 + _wt / 2, _ob / 2 - 1.0 - 0.3, 0.9)),          # left: left end seen from outside is the +Y end
        ((_oa / 2 - _wt / 2, -_ob / 2 + 1.0, 0.9), (_oa / 2 - _wt / 2, -_ob / 2 + 1.0 + 0.3, 0.9))]          # right: left end seen from outside is the -Y end
_check_holes_cut("rect_container", _rp, _hs, _pts, math.pi * 0.125**2 * _wt)
_cp = {"part_type": "cyl_container", "id_in": 2.75, "height_in": 4.0, "thickness_in": 0.0963}
_Rm = 2.75 / 2 + 0.0963 / 2                                                                                    # mid-wall radius
_hc = [{"face": "side", "angle_deg": a, "up_in": 2.0, "diameter_in": 0.5} for a in (0, 90, 180, 270, 45)]
_pts = [((_Rm * math.cos(math.radians(a)), _Rm * math.sin(math.radians(a)), 2.0), (_Rm * math.cos(math.radians(a)), _Rm * math.sin(math.radians(a)), 2.0 + 0.5)) for a in (0, 90, 180, 270, 45)]
_check_holes_cut("cyl_container", _cp, _hc, _pts, math.pi * 0.25**2 * 0.0963)
_i0 = build_stl(_bp, WORKDIR / "t_nohole.stl"); _i1 = build_stl(_bp, WORKDIR / "t_hole.stl", [_ha, _hb])
assert _i1["watertight"] and _i0["volume_mm3"] - _i1["volume_mm3"] > 0 and abs((_i0["volume_mm3"] - _i1["volume_mm3"]) - 2 * math.pi * 0.125**2 * 0.125 * 25.4**3) / (2 * math.pi * 0.125**2 * 0.125 * 25.4**3) < 0.03, (_i0, _i1)
print("hole self-tests passed (every hole is empty at its centre, solid beside it, correct volume removed; front/back/left/right and angles verified)")
print("CAD self-tests passed (watertight, volumes match the maths, holes/cavities really cut, mm scaling correct)")

## 5b. Data augmentation (synthetic samples, stored separately)
Adds `AUGMENT_PER_TYPE` x 5 = **500 synthetic training samples**, generated by code from your own real training texts (nobody writes new sentences):
* **value swap** - a number that clearly belongs to one label field is replaced by another real value of that field, and the label is updated to match (not for washers, whose sizes are tied to the screw-size table);
* **surface noise** - unit wording (`inch` / `inches` / `in`), one typo, lower-casing, dropped final period.

Safety rules: sources are **train-split rows only**; validation and test stay 100% real; every synthetic row must pass the same audit and CAD rules as real data; no text may duplicate a real one. They are saved to their own file so you can report them as synthetic data (your real sample count stays 501).

In [ ]:
# ------------------------------------------------------------------ DATA AUGMENTATION (synthetic - kept separate from your real data)
# Every synthetic sample is made BY CODE from one of your real TRAINING rows. No new sentences are written by anyone.
#   1. value swap : numbers that are clearly one label field (one unambiguous match in the text) are replaced by another real
#                   value of that same field, and the label is updated to match -> a genuinely new (text, label) pair.
#                   (Not done for washers: their sizes are tied to the screw-size table, so washers get step 2 only.)
#   2. surface noise : unit wording (inch / inches / in), one typo in a long word, lower-casing, dropped final period.
# Safety rules: only train-split rows are used as sources, value pools come from train rows only, validation/test stay 100% real,
# every sample must pass the same label-in-text audit and CAD rules as real data, and no text may duplicate an existing one.
SUBSTITUTE_TYPES = {"pipe_gasket", "bracket", "rect_container", "cyl_container"}
NUM_TOK = re.compile(r"(?<![\w.])\d+(?:\.\d+)?")

for _d in (train_df, val_df, test_df):
    _d["source"] = "real"
train_real_df = train_df.copy()

def _unambiguous_tokens(row):
    """{token_index: [fields]} for label fields that match exactly one number in the text and aren't shared with an unrelated field"""
    text, pt = row["text_norm"], row["part_type"]
    toks = [(m.start(), m.end(), float(m.group())) for m in NUM_TOK.finditer(text)]
    f2t = {}
    for f in NUM_FIELDS[pt]:
        v = row["params"][f]
        if f in INT_FIELDS: hits = [i for i, (s, e, x) in enumerate(toks) if x == v and text[s:e].isdigit()]
        else:               hits = [i for i, (s, e, x) in enumerate(toks) if abs(x - v) <= tol_for(v)]
        if len(hits) == 1: f2t[f] = hits[0]
    t2f = {}
    for f, i in f2t.items(): t2f.setdefault(i, []).append(f)
    return toks, {i: fs for i, fs in t2f.items() if len(fs) == 1 or set(fs) <= {"length_a_in", "length_b_in"}}   # equal legs may share a number

def _noise(text, rng):
    unit = rng.choice([" inches", " inch", " in"])
    t = re.sub(r"(?<=\d)\s*(?:inches|inch|in)\b(?!\.\d)", unit, text)
    if rng.random() < 0.35:
        words = [m for m in re.finditer(r"[A-Za-z]{5,}", t)]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            t = t[:m.start()] + "".join(w) + t[m.end():]
    if rng.random() < 0.20: t = t.lower()
    if rng.random() < 0.50: t = t.rstrip(". ")
    return t

def build_augmented(src_df, per_type, seed):
    rng = random.Random(seed)
    src_ok = src_df[src_df.audit_bad.map(len) == 0]
    seen = set(data.text_norm)                                          # never duplicate any real text (train, val or test)
    pools = {pt: {f: sorted({r["params"][f] for _, r in src_ok[src_ok.part_type == pt].iterrows()}) for f in NUM_FIELDS[pt]} for pt in PART_TYPES}
    out, stats = [], {}
    for pt in PART_TYPES:
        sources = [r for _, r in src_ok[src_ok.part_type == pt].iterrows()]; rng.shuffle(sources)   # round-robin below -> every real row used about equally
        made, tries = 0, 0
        while made < per_type and tries < per_type * 60 and sources:
            tries += 1
            src = sources[tries % len(sources)]
            text, params = src["text_norm"], dict(src["params"])
            if pt in SUBSTITUTE_TYPES:
                toks, groups = _unambiguous_tokens(src)
                if not groups: continue
                for i in sorted(groups, reverse=True):
                    f0 = groups[i][0]; old = params[f0]
                    new = rng.choice([v for v in pools[pt][f0] if (v != old if f0 in INT_FIELDS else abs(v - old) > tol_for(old))] or [old])
                    for f in groups[i]: params[f] = new
                    s, e, _ = toks[i]; text = text[:s] + (str(int(new)) if f0 in INT_FIELDS else dec(new)) + text[e:]
            text_final = _noise(text, rng); text_norm = normalize_text(text_final)
            if text_norm in seen or text_norm == src["text_norm"]: continue
            probe = {"part_type": pt, "params": params, "text_norm": text_norm}
            if audit_row(probe): continue                               # every label value must be readable in its own text
            rows = make_table(pt, text_norm, params); full = rows_to_params(pt, rows)
            if any(i["level"] == "error" for i in check_rules(pt, full)): continue
            if any(full[f] != params[f] for f in SCHEMA[pt]): continue  # snapping must not have altered the label
            p = {"part_type": pt, **{f: params[f] for f in SCHEMA[pt]}}
            seen.add(text_norm); made += 1
            out.append({"sheet": "augmented", "sheet_row": src["sheet_row"], "text": text_final, "part_type": pt, "text_norm": text_norm,
                        "params": p, "target": params_to_json(p), "split": "train", "group": src["group"], "audit_bad": [],
                        "source": "augmented", "aug_from": f"{src['sheet']}:{src['sheet_row']}", "orig_text": src["text"]})
        stats[pt] = (made, tries)
    return pd.DataFrame(out), stats

if AUGMENT_PER_TYPE > 0:
    aug_df, _stats = build_augmented(train_real_df, AUGMENT_PER_TYPE, AUG_SEED)
    short = {pt: n for pt, (n, _) in _stats.items() if n < AUGMENT_PER_TYPE}
    if short: print("WARNING: could not reach the target for", short)
    train_df = pd.concat([train_real_df, aug_df], ignore_index=True)
else:
    aug_df = pd.DataFrame(columns=train_real_df.columns)

_cnt = pd.DataFrame({"real (train)": train_real_df.part_type.value_counts(), "augmented": aug_df.part_type.value_counts() if len(aug_df) else 0}).reindex(PART_TYPES).fillna(0).astype(int)
_cnt["train total"] = _cnt.sum(axis=1)
_cnt["val (real)"] = val_df.part_type.value_counts().reindex(PART_TYPES).fillna(0).astype(int)
_cnt["test (real)"] = test_df.part_type.value_counts().reindex(PART_TYPES).fillna(0).astype(int)
show_df(_cnt)
print(f"\nDataset size: {len(data)} real + {len(aug_df)} augmented = {len(data) + len(aug_df)}  |  "
      f"training uses {len(train_df)} rows; validation ({len(val_df)}) and test ({len(test_df)}) are real only.")
print("Augmented rows are stored separately (artifacts/augmented_samples.csv) - report them as synthetic data.\n")
for _pt in PART_TYPES[:5]:
    if len(aug_df) and (aug_df.part_type == _pt).any():
        _r = aug_df[aug_df.part_type == _pt].iloc[0]
        print(f"[{_pt}]\n  real     : {_r.orig_text}\n  synthetic: {_r.text}\n  label    : {_r.target}")

## 5c. Add features: hole training data (written by Claude, stored separately)
After the first build, the app lets you **add holes** to L-brackets and both containers. The editable hole table works with no training; to also let users *describe* holes in words, the fine-tuned model is trained on a second task: description -> holes JSON.

That training data is **synthetic and written by Claude** (not by the team), so it is kept apart from your 501 manual samples:
* **120 hand-written sentences**, built into this notebook (nothing to upload), each describing 1-3 holes on a real base part from your dataset. The same rows are also in `hole_features_filled.xlsx` if you want to read or edit them.
* **generated sentences** - `FEATURE_GENERATED_PER_TYPE` more per part type, rendered from phrase banks (fractions/decimals/words, many openers, structures, wall/leg names, typos). Always training-only.
* **Evaluation is on the hand-written rows only** (`val`/`test` never contain generated text), so test accuracy measures differently-worded text, not the generator's own phrasing.

The next cell can also write a blank template if you want to add more hand-written rows (`MAKE_FEATURE_TEMPLATE = True`).

In [ ]:
# ------------------------------------------------------------------ 5c: LABEL-FIRST TEMPLATE for hole-feature sentences
# The numbers are generated here (real base parts from YOUR sheet + hole specs that are guaranteed geometrically valid).
# The SENTENCES are left blank for your team to write, so the text-to-holes model is trained on human wording only.
TEMPLATE_ROWS_PER_TYPE = 40                     # rows per part type (3 types). Change and re-run to get a bigger/smaller template
TEMPLATE_PATH = "hole_features_template.xlsx"
TEMPLATE_SEED = 11
HOLE_DIAMETERS = [0.125, 0.1875, 0.25, 0.3125, 0.375, 0.5, 0.625, 0.75]
FEAT_SHEETS = {"bracket": "Holes_bracket", "rect_container": "Holes_rect", "cyl_container": "Holes_cyl"}

def _grid(lo, hi, step=0.125):
    out, x = [], math.ceil(lo / step - 1e-9) * step
    while x <= hi + 1e-9: out.append(round(x, 4)); x += step
    return out

def _resolve_for_check(pt, p, holes):
    """blank (None) positions count as centred, exactly as the app treats them"""
    out = []
    for h in holes:
        d = hole_defaults(pt, p, h["face"])
        out.append({**h, **{key: v for key, v in d.items() if h.get(key) is None}})
    return out

def sample_holes(pt, p, rng, n):
    k, t, holes = HOLE_SPEC[pt]["keys"], p["thickness_in"], []
    for _ in range(n):
        for _attempt in range(80):
            d = rng.choice(HOLE_DIAMETERS); r, m = d / 2, 0.1
            face = rng.choice(HOLE_SPEC[pt]["faces"])
            if pt == "bracket":
                us = _grid(r + m, leg_length(p, face) - t - r - m); vs = _grid(r + m, p["height_in"] - r - m)
            elif pt == "rect_container":
                us = _grid(t + r + m, wall_length(p, face) - t - r - m); vs = _grid(t + r + m, p["height_in"] - r - m)
            else:
                us = [float(a) for a in range(0, 360, 15)]; vs = _grid(t + r + m, p["height_in"] - r - m)
            if not us or not vs: continue
            u = rng.choice(us) if (pt == "cyl_container" or rng.random() > 0.15) else None       # ~15% of positions along the part left blank = "centred"
            v = rng.choice(vs) if rng.random() > 0.30 else None                                  # ~30% of heights left blank = "centred"
            cand = {k[0]: face, k[1]: u, k[2]: v, k[3]: d}
            if not check_holes(pt, p, _resolve_for_check(pt, p, holes + [cand])):                # no errors AND no warnings
                holes.append(cand); break
        else:
            return None
    return holes

def make_template(rows_per_type, seed):
    rng, sheets = random.Random(seed), {}
    for pt, sheet in FEAT_SHEETS.items():
        bases = [r for _, r in raw_all[raw_all.part_type == pt].iterrows()]; rng.shuffle(bases); out = []
        for r in bases * 3:                                                                       # cycle if more rows than distinct base parts are asked for
            if len(out) >= rows_per_type: break
            p = {"part_type": pt, **{f: float(r[f]) for f in SCHEMA[pt]}}
            if any(i["level"] == "error" for i in check_rules(pt, p)): continue
            holes = sample_holes(pt, p, rng, rng.choices([1, 2, 3], [0.5, 0.3, 0.2])[0])
            if holes is None: continue
            row = {"base_ref": f"{r['sheet']}:{r['sheet_row']}", **{f: p[f] for f in SCHEMA[pt]}, "n_holes": len(holes)}
            for n in range(1, 4):
                for key in HOLE_SPEC[pt]["keys"]:
                    row[f"h{n}_{key}"] = holes[n - 1][key] if n <= len(holes) else None
            row["text"] = None                                                                    # <- YOU write this: how a user would ask for these holes
            out.append(row)
        sheets[sheet] = pd.DataFrame(out)
    return sheets

INSTRUCTIONS = [
    "HOW TO FILL THIS TEMPLATE",
    "Each row already contains a real base part from your dataset and 1-3 holes (face, positions, diameter). Your job: in the `text` column, write the sentence a real user would type to ask for exactly those holes.",
    "Write in your own words and vary your style between rows (different phrasing, units, fractions vs decimals, formal vs sloppy). Describe ONLY the holes - the base part is already known.",
    "Every number in the hole columns must be recognisable in your sentence (fractions are fine: 0.25 can be written 1/4). Do not edit the number columns; if a row seems unrealistic, leave its text blank and it will be skipped.",
    "A BLANK position cell means the hole is centred. For those holes, say 'centred' / 'in the middle' (or just do not give a position along that direction).",
    "",
    "CONVENTIONS (the app uses the same ones - mention them naturally so the model can learn them):",
    "Holes_bracket: leg_a runs left-to-right in the preview and leg_b runs away from you. from_end_in = distance from the leg's FREE end to the hole centre. across_in = height above the bottom face.",
    "Holes_rect: front faces you, back is behind, left/right are the sides; along_in = distance from the wall's LEFT end as seen from outside; up_in = height above the bottom of the container.",
    "Holes_cyl: angle_deg is counter-clockwise seen from above: 0 = right, 90 = back, 180 = left, 270 = front. up_in = height above the bottom.",
    "Hole diameters are in inches. h1/h2/h3 are the holes; unused hole columns are blank.",
]
if MAKE_FEATURE_TEMPLATE:
    _sheets = make_template(TEMPLATE_ROWS_PER_TYPE, TEMPLATE_SEED)
    with pd.ExcelWriter(TEMPLATE_PATH) as _w:
        pd.DataFrame({"README": INSTRUCTIONS}).to_excel(_w, sheet_name="Instructions", index=False)
        for _name, _d in _sheets.items(): _d.to_excel(_w, sheet_name=_name, index=False)
    print(f"wrote {TEMPLATE_PATH}: " + ", ".join(f"{n} {len(d)} rows" for n, d in _sheets.items()) + "  (the `text` column is blank - write the sentences, then set LOAD_FEATURE_DATA = True)")
    try:
        from google.colab import files; files.download(TEMPLATE_PATH)
    except ImportError:
        pass

In [ ]:
# ------------------------------------------------------------------ GENERATED hole sentences (SYNTHETIC - written by Claude as phrase banks, stored separately)
# The 120 hand-written sentences in hole_features_filled.xlsx are too few to train on alone, so this adds `FEATURE_GENERATED_PER_TYPE` more per part type.
# Each sentence is rendered FROM its label (the label values are the single source of truth), using many ways to say the same thing:
# fractions / decimals / words, inch / in / ", different openers, sentence structures, wall/leg names, centring phrases, typos, casing.
# Base parts come from your real dataset rows that the hand-written set did NOT use. These rows only ever go into TRAINING.
from math import gcd

def _frac(v):
    """0.1875 -> '3/16', 1.375 -> '1 3/8', 4.0 -> '4'  (None if v is not a multiple of 1/16)"""
    x = round(v * 16)
    if abs(v * 16 - x) > 1e-6: return None
    whole, rem = divmod(x, 16)
    if rem == 0: return str(whole)
    g = gcd(rem, 16); fr = f"{rem // g}/{16 // g}"
    return f"{whole} {fr}" if whole else fr

_UNITS = ["", " in", " inch", " inches", '"', " inches", " inch"]
def _len(v, rng, words=False):
    """a length written in a random human style, with a random unit"""
    if words and v in (0.25, 0.5) and rng.random() < 0.25:
        return rng.choice(["quarter inch", "quarter of an inch"] if v == 0.25 else ["half inch"])
    body = _frac(v) if (rng.random() < 0.55 and _frac(v) is not None) else dec(v)
    if body.startswith("0.") and rng.random() < 0.15: body = body[1:]
    unit = rng.choice(_UNITS)
    if unit == " inch" and v > 1: unit = " inches"
    return body + unit

def _a(text): return ("an " if text[:1] == "8" or text.startswith("11") or text.startswith("18") else "a ") + text

def _dia(d, rng):
    L = _len(d, rng, words=True)
    return rng.choice([f"{L} hole", f"{L} diameter hole", f"{L} dia hole", f"{L} through hole", f"{L} round hole", f"{L} dia. hole"])

_DEAD = ["dead center", "in the exact center", "centered both ways", "dead center of it"]

def _pos_bracket(u, v, rng):
    if u is None and v is None: return rng.choice(_DEAD)
    pu = (rng.choice(["centered along the leg", "in the middle of the leg", "centered lengthwise", "halfway along the leg"]) if u is None else
          (lambda P: rng.choice([f"{P} from the end", f"{P} in from the end", f"{P} from the free end", f"{P} from the tip", f"{P} back from the end of the leg"]))(_len(u, rng, True)))
    pv = (rng.choice(["centered across the width", "centered on the width", "in the middle of the width", "halfway across the width"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom face", f"{P} from the bottom", f"{P} off the bottom", f"{P} up from the bottom face"]))(_len(v, rng, True)))
    a, b = (pu, pv) if rng.random() < 0.75 else (pv, pu)
    return a + rng.choice([" and ", ", ", ", and "]) + b

def _pos_rect(u, v, rng):
    if u is None and v is None: return rng.choice(_DEAD)
    pu = (rng.choice(["centered left to right", "centered side to side", "horizontally centered", "in the middle left to right"]) if u is None else
          (lambda P: rng.choice([f"{P} from the left end", f"{P} from the left edge", f"{P} in from the left end", f"{P} from the left corner", f"{P} from the left as you look at it"]))(_len(u, rng, True)))
    pv = (rng.choice(["centered vertically", "vertically centered", "halfway up", "mid-height", "halfway up the wall"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom", f"{P} up", f"{P} from the bottom", f"{P} off the floor"]))(_len(v, rng, True)))
    a, b = (pu, pv) if rng.random() < 0.8 else (pv, pu)
    return a + rng.choice([" and ", ", ", ", and "]) + b

_CARD = {270: ["front", "the front", "front side"], 90: ["back", "the back", "back side", "rear"], 180: ["left", "the left", "left side"], 0: ["right", "the right", "right side"]}
def _pos_cyl(u, v, rng):
    A = int(round(u))
    ang = (rng.choice([f"on the {rng.choice(_CARD[A]).replace('the ', '')} of the cup", f"on the {rng.choice(_CARD[A]).replace('the ', '')}"]) if (A in _CARD and rng.random() < 0.45) else
           rng.choice([f"at {A} degrees", f"at an angle of {A} degrees", f"at {A}°", f"at {A} deg", f"{A} degrees around the cup"]))
    up = (rng.choice(["centered vertically", "vertically centered", "halfway up", "mid-height", "in the middle of the height"]) if v is None else
          (lambda P: rng.choice([f"{P} up from the bottom", f"{P} above the bottom", f"{P} up", f"{P} from the bottom"]))(_len(v, rng, True)))
    return ang + rng.choice([", ", ", and ", " and "]) + up

def _face_ref(pt, face, base, rng):
    if pt == "bracket":
        a, b = base["length_a_in"], base["length_b_in"]
        refs = ["leg A", "leg a", "the first leg"] if face == "leg_a" else ["leg B", "leg b", "the second leg"]
        if abs(a - b) > 1e-6:
            longer = (face == "leg_a") == (a > b)
            refs += (["the longer leg", "the long leg"] if longer else ["the shorter leg", "the short leg"]) * 2
            refs.append(f"the {dec(round(base['length_a_in' if face == 'leg_a' else 'length_b_in'], 2))} inch leg")
        return rng.choice(refs)
    if pt == "rect_container":
        w = "rear" if (face == "back" and rng.random() < 0.15) else face
        return rng.choice([f"the {w} wall", f"the {w} wall", f"the {w} side", f"the {w}", f"{w} wall"])
    return "the cup"

_OPEN = ["Add", "Drill", "Cut", "Put", "Make", "I need", "I want", "Please add", "Can you add", "Give it", "Create"]
def _clause(pt, h, base, rng, k):
    face = _face_ref(pt, h["face"], base, rng); dia = _dia(h[k[3]], rng)
    pos = {"bracket": _pos_bracket, "rect_container": _pos_rect, "cyl_container": _pos_cyl}[pt](h[k[1]], h[k[2]], rng)
    where = "" if pt == "cyl_container" else rng.choice([f" in {face}", f" in {face}", f" through {face}", f" on {face}"])
    forms = [(True, f"{_a(dia)}{where}, {pos}"), (True, f"{_a(dia)}{where}: {pos}"), (False, f"{dia}{where}, {pos}")]
    if pt != "cyl_container": forms += [(False, f"in {face}: {dia}, {pos}"), (False, f"{face}: {dia}, {pos}"), (False, f"{face}, {dia}, {pos}")]
    return rng.choice(forms)

TYPO_RATE = 0.12                                                                         # chance a sentence gets one typo (set 0 to disable)
_PROTECT = {"front", "back", "left", "right", "rear", "wall", "side", "degrees", "inch", "inches", "center", "centered", "middle", "halfway", "vertically", "horizontally", "bottom", "diameter"}
def render_sentence(pt, base, holes, rng):
    k = HOLE_SPEC[pt]["keys"]; n = len(holes)
    clauses = [_clause(pt, h, base, rng, k) for h in holes]
    cap = lambda s: s[:1].upper() + s[1:]
    if n == 1:
        opener_ok, c = clauses[0]
        text = f"{rng.choice(_OPEN)} {c}" if (opener_ok and rng.random() < 0.7) else cap(c)
    else:
        word = {2: "two", 3: "three"}[n]; mode = rng.choice(["list", "sentences", "joined"])
        if mode == "list":
            text = rng.choice([f"{cap(word)} holes: ", f"{rng.choice(_OPEN)} {word} holes. ", f"{cap(word)} holes please. ", ""]) + "; ".join(c for _, c in clauses)
        elif mode == "sentences":
            text = " ".join(cap(c) + "." for _, c in clauses).rstrip(".")
        else:
            text = (f"{rng.choice(_OPEN)} " if clauses[0][0] else "") + (clauses[0][1] if clauses[0][0] else cap(clauses[0][1])) + "".join(rng.choice([", and ", " plus ", ", also ", "; then "]) + c for _, c in clauses[1:])
    if rng.random() < 0.12: text += rng.choice([", please", " please", " thanks"])
    if rng.random() < 0.55 and not text.endswith("."): text += "."
    if rng.random() < TYPO_RATE:                                                             # one typo in an unprotected word
        words = [m for m in re.finditer(r"[A-Za-z]{5,}", text) if m.group().lower() not in _PROTECT]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            text = text[:m.start()] + "".join(w) + text[m.end():]
    if rng.random() < 0.18: text = text.lower()
    return text

def generate_feature_rows(per_type, seed, exclude_refs=()):
    rng, rows = random.Random(seed), []
    for pt in FEAT_SHEETS:
        bases = [r for _, r in raw_all[raw_all.part_type == pt].iterrows() if f"{r['sheet']}:{r['sheet_row']}" not in set(exclude_refs)]
        bases = bases or [r for _, r in raw_all[raw_all.part_type == pt].iterrows()]
        rng.shuffle(bases); made, seen, tries = 0, set(), 0
        while made < per_type and tries < per_type * 40:
            r = bases[tries % len(bases)]; tries += 1
            base = {f: float(r[f]) for f in SCHEMA[pt]}; p = {"part_type": pt, **base}
            if any(i["level"] == "error" for i in check_rules(pt, p)): continue
            holes = sample_holes(pt, p, rng, rng.choices([1, 2, 3], [0.45, 0.35, 0.2])[0])
            if holes is None: continue
            text = render_sentence(pt, base, holes, rng); tn = normalize_text(text)
            if tn in seen or audit_holes(pt, holes, tn) or semantic_hole_problems(pt, holes, tn): continue      # label must be readable AND consistent
            seen.add(tn); made += 1
            rows.append({"sheet": "generated", "sheet_row": len(rows) + 1, "part_type": pt, "base": base, "holes": holes, "text": text,
                         "group": f"{r['sheet']}:{r['sheet_row']}|gen", "source": "claude_generated"})
    return pd.DataFrame(rows)
print("hole-sentence generator ready")

**Load the hole data.** Checks every row (numbers readable in the sentence, walls/directions consistent with the labels, hole geometry valid), reports problems by sheet and Excel row, and splits by base part. Set `LOAD_FEATURE_DATA = False` to skip feature training.

In [ ]:
# ------------------------------------------------------------------ the 120 hand-written hole sentences, built into the notebook (SYNTHETIC - written by Claude)
# One record per sentence: base part (from your dataset), the holes it describes, and the sentence. Nothing to upload. Same content as hole_features_filled.xlsx.
FEATURE_WRITTEN_JSON = r'''[{"sheet":"Holes_bracket","sheet_row":2,"part_type":"bracket","base":{"length_a_in":1.5625,"length_b_in":1.5625,"height_in":1.0,"thickness_in":0.12},"holes":[{"face":"leg_a","from_end_in":null,"across_in":0.375,"diameter_in":0.1875}],"text":"Put a 3/16 hole in leg A, centered along the leg and 3/8 inch up from the bottom.","group":"L_brackets:68","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":3,"part_type":"bracket","base":{"length_a_in":4.0,"length_b_in":4.0,"height_in":0.875,"thickness_in":0.12},"holes":[{"face":"leg_b","from_end_in":3.25,"across_in":0.5,"diameter_in":0.3125},{"face":"leg_a","from_end_in":3.5,"across_in":0.375,"diameter_in":0.3125}],"text":"Two 5/16 inch holes please: one in leg B, 3.25 inches from the end and 1/2 inch up from the bottom, and one in leg A, 3.5 inches from the end and 3/8 inch up.","group":"L_brackets:12","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":4,"part_type":"bracket","base":{"length_a_in":1.875,"length_b_in":2.375,"height_in":2.375,"thickness_in":0.1},"holes":[{"face":"leg_a","from_end_in":null,"across_in":null,"diameter_in":0.625}],"text":"I need a 5/8 inch hole dead center in the shorter leg.","group":"L_brackets:41","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":5,"part_type":"bracket","base":{"length_a_in":2.75,"length_b_in":2.75,"height_in":0.8125,"thickness_in":0.24},"holes":[{"face":"leg_b","from_end_in":1.75,"across_in":null,"diameter_in":0.125}],"text":"drill a 1/8 hole into leg B 1.75 in from the end, centered on the width","group":"L_brackets:56","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":6,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":1.5,"thickness_in":0.12},"holes":[{"face":"leg_a","from_end_in":1.0,"across_in":0.5,"diameter_in":0.3125}],"text":"Add a 5/16\" diameter hole to leg A. It should sit 1\" from the free end and 1/2\" up from the bottom face.","group":"L_brackets:85","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":7,"part_type":"bracket","base":{"length_a_in":3.1496,"length_b_in":1.5748,"height_in":1.9685,"thickness_in":0.1181},"holes":[{"face":"leg_a","from_end_in":1.125,"across_in":1.0,"diameter_in":0.75},{"face":"leg_b","from_end_in":0.875,"across_in":null,"diameter_in":0.3125},{"face":"leg_a","from_end_in":null,"across_in":0.75,"diameter_in":0.125}],"text":"On the long leg I want a 3/4 inch hole 1 1/8 inch from the end and 1 inch up, plus a tiny 1/8 hole centered along the leg at 3/4 inch up. The short leg gets a 5/16 inch hole 7/8 inch from its end, centered across the width.","group":"L_brackets:81","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":8,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":1.25,"thickness_in":0.06},"holes":[{"face":"leg_b","from_end_in":1.0,"across_in":0.75,"diameter_in":0.625}],"text":"5/8 inch hole in leg B, 1 inch from the end, 3/4 inch from the bottom.","group":"L_brackets:38","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":9,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":1.0,"height_in":1.0,"thickness_in":0.105},"holes":[{"face":"leg_a","from_end_in":1.25,"across_in":0.75,"diameter_in":0.1875}],"text":"Drill a 3/16 hole through the 2 inch leg, 1.25 inches from the end and 0.75 inches up from the bottom.","group":"L_brackets:84","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":10,"part_type":"bracket","base":{"length_a_in":9.75,"length_b_in":9.75,"height_in":5.0,"thickness_in":0.25},"holes":[{"face":"leg_a","from_end_in":3.625,"across_in":0.875,"diameter_in":0.375},{"face":"leg_b","from_end_in":1.25,"across_in":null,"diameter_in":0.375}],"text":"Two 3/8 inch holes please. Leg A: 3 5/8 inches from the end, 7/8 inch up. Leg B: 1 1/4 inches from the end, centered in the width.","group":"L_brackets:49","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":11,"part_type":"bracket","base":{"length_a_in":2.75,"length_b_in":3.375,"height_in":0.625,"thickness_in":0.19},"holes":[{"face":"leg_a","from_end_in":2.0,"across_in":0.25,"diameter_in":0.25}],"text":"put a quarter inch hole in the shorter leg, 2 inches from its end and 1/4 inch up from the bottom","group":"L_brackets:60","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":12,"part_type":"bracket","base":{"length_a_in":1.8125,"length_b_in":1.8125,"height_in":0.5,"thickness_in":0.12},"holes":[{"face":"leg_b","from_end_in":0.75,"across_in":0.25,"diameter_in":0.125},{"face":"leg_a","from_end_in":0.875,"across_in":0.25,"diameter_in":0.1875}],"text":"Two small holes, both 1/4 inch up from the bottom. Leg B gets a 1/8 inch one 3/4 inch from the end; leg A gets a 3/16 inch one 7/8 inch from the end.","group":"L_brackets:27","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":13,"part_type":"bracket","base":{"length_a_in":5.625,"length_b_in":5.625,"height_in":5.0,"thickness_in":0.13},"holes":[{"face":"leg_b","from_end_in":null,"across_in":0.875,"diameter_in":0.5},{"face":"leg_a","from_end_in":5.0,"across_in":null,"diameter_in":0.625}],"text":"Leg B: a 1/2 inch hole centered along the leg, 7/8 inch up from the bottom. Leg A: a 5/8 inch hole 5 inches from the end, centered on the width.","group":"L_brackets:47","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":14,"part_type":"bracket","base":{"length_a_in":8.0,"length_b_in":8.0,"height_in":1.25,"thickness_in":0.22},"holes":[{"face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.625},{"face":"leg_b","from_end_in":6.0,"across_in":0.875,"diameter_in":0.1875}],"text":"add a 5/8 hole to leg A 1 1/8 inch from the end and centered, and a 3/16 hole to leg B 6 inches from the end and 7/8 inch off the bottom","group":"L_brackets:15","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":15,"part_type":"bracket","base":{"length_a_in":0.5625,"length_b_in":0.5625,"height_in":1.0,"thickness_in":0.06},"holes":[{"face":"leg_b","from_end_in":0.25,"across_in":null,"diameter_in":0.125},{"face":"leg_a","from_end_in":0.25,"across_in":0.5,"diameter_in":0.25}],"text":"Leg B needs a 1/8 inch hole a quarter inch from the end, centered. Leg A needs a 1/4 inch hole, also 1/4 inch from the end, 1/2 inch up from the bottom.","group":"L_brackets:32","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":16,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":2.0,"height_in":2.5,"thickness_in":0.125},"holes":[{"face":"leg_b","from_end_in":0.625,"across_in":1.75,"diameter_in":0.1875},{"face":"leg_b","from_end_in":1.375,"across_in":0.5,"diameter_in":0.75},{"face":"leg_a","from_end_in":0.875,"across_in":0.875,"diameter_in":0.375}],"text":"Leg B gets two holes: a 3/16 inch hole 5/8 inch from the end and 1 3/4 inch up, and a 3/4 inch hole 1 3/8 inches from the end and 1/2 inch up. Leg A gets one 3/8 inch hole, 7/8 inch from the end and 7/8 inch up.","group":"L_brackets:76","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":17,"part_type":"bracket","base":{"length_a_in":3.625,"length_b_in":3.625,"height_in":2.0,"thickness_in":0.11},"holes":[{"face":"leg_a","from_end_in":1.0,"across_in":null,"diameter_in":0.125},{"face":"leg_a","from_end_in":0.5,"across_in":0.5,"diameter_in":0.375}],"text":"Two holes on leg A: 1/8 inch diameter, 1 inch from the end and centered across; 3/8 inch diameter, 1/2 inch from the end and 1/2 inch up.","group":"L_brackets:45","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":18,"part_type":"bracket","base":{"length_a_in":7.875,"length_b_in":9.4375,"height_in":0.75,"thickness_in":0.31},"holes":[{"face":"leg_a","from_end_in":1.875,"across_in":0.375,"diameter_in":0.5}],"text":"On the shorter leg add a half inch hole, 1 7/8 inch in from the end, 3/8 inch up from the bottom face.","group":"L_brackets:63","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":19,"part_type":"bracket","base":{"length_a_in":0.75,"length_b_in":0.75,"height_in":1.5,"thickness_in":0.04},"holes":[{"face":"leg_b","from_end_in":0.25,"across_in":null,"diameter_in":0.125}],"text":"tiny 1/8 hole in leg B a quarter inch from the end, centered","group":"L_brackets:36","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":20,"part_type":"bracket","base":{"length_a_in":3.0,"length_b_in":3.0,"height_in":0.75,"thickness_in":0.1},"holes":[{"face":"leg_a","from_end_in":1.25,"across_in":null,"diameter_in":0.5}],"text":"Please add a 0.5 inch hole to leg A, 1.25 inches from the end, centered on the width.","group":"L_brackets:21","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":21,"part_type":"bracket","base":{"length_a_in":2.375,"length_b_in":2.375,"height_in":0.6875,"thickness_in":0.08},"holes":[{"face":"leg_a","from_end_in":1.5,"across_in":0.375,"diameter_in":0.3125},{"face":"leg_a","from_end_in":2.0,"across_in":0.375,"diameter_in":0.3125},{"face":"leg_a","from_end_in":0.5,"across_in":0.375,"diameter_in":0.3125}],"text":"Leg A needs three 5/16 inch holes in a row, all 3/8 inch up from the bottom, at 1/2 inch, 1 1/2 inches and 2 inches from the end.","group":"L_brackets:8","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":22,"part_type":"bracket","base":{"length_a_in":0.6875,"length_b_in":1.0,"height_in":0.5,"thickness_in":0.12},"holes":[{"face":"leg_b","from_end_in":0.25,"across_in":0.25,"diameter_in":0.125}],"text":"1/8 inch hole in the longer leg, 1/4 inch from the end and 1/4 inch up from the bottom.","group":"L_brackets:24","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":23,"part_type":"bracket","base":{"length_a_in":6.0,"length_b_in":6.0,"height_in":1.125,"thickness_in":0.18},"holes":[{"face":"leg_a","from_end_in":1.0,"across_in":0.5,"diameter_in":0.75},{"face":"leg_b","from_end_in":1.25,"across_in":0.75,"diameter_in":0.25}],"text":"Leg A: 3/4 inch hole 1 inch from the end, half an inch off the bottom. Leg B: 1/4 inch hole, 1 1/4 inches from the end, 3/4 inch up.","group":"L_brackets:23","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":24,"part_type":"bracket","base":{"length_a_in":5.625,"length_b_in":5.625,"height_in":5.0,"thickness_in":0.25},"holes":[{"face":"leg_b","from_end_in":1.125,"across_in":4.0,"diameter_in":0.75},{"face":"leg_a","from_end_in":4.75,"across_in":null,"diameter_in":0.375},{"face":"leg_a","from_end_in":1.625,"across_in":2.625,"diameter_in":0.625}],"text":"Three holes. Leg B: a 3/4 inch hole 1 1/8 inches from the end and 4 inches up from the bottom. Leg A: a 3/8 inch hole 4 3/4 inches from the end, centered across the width, and a 5/8 inch hole 1 5/8 inches from the end and 2 5/8 inches up.","group":"L_brackets:48","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":25,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":0.9375,"thickness_in":0.05},"holes":[{"face":"leg_b","from_end_in":0.875,"across_in":null,"diameter_in":0.3125},{"face":"leg_a","from_end_in":0.5,"across_in":0.5,"diameter_in":0.625}],"text":"Cut a 5/16 diameter hole in leg B, 7/8\" from the end, centered. Cut a 5/8 diameter hole in leg A, 1/2\" from the end and 1/2\" up from the bottom.","group":"L_brackets:4","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":26,"part_type":"bracket","base":{"length_a_in":1.375,"length_b_in":1.375,"height_in":0.75,"thickness_in":0.04},"holes":[{"face":"leg_b","from_end_in":null,"across_in":null,"diameter_in":0.1875},{"face":"leg_a","from_end_in":0.875,"across_in":null,"diameter_in":0.5}],"text":"I want a 3/16 inch hole dead center in leg B and a 1/2 inch hole in leg A that's 7/8 inch from the end and centered across the width.","group":"L_brackets:29","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":27,"part_type":"bracket","base":{"length_a_in":3.5,"length_b_in":3.5,"height_in":0.75,"thickness_in":0.12},"holes":[{"face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.125}],"text":"Add a 1/8 inch hole to leg A, 1 1/8 inch from the end, in the middle of the width.","group":"L_brackets:11","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":28,"part_type":"bracket","base":{"length_a_in":3.5625,"length_b_in":3.5625,"height_in":0.8125,"thickness_in":0.24},"holes":[{"face":"leg_a","from_end_in":0.75,"across_in":0.375,"diameter_in":0.5}],"text":"half inch hole in leg A, 3/4 inch from the end, 3/8 inch from the bottom","group":"L_brackets:57","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":29,"part_type":"bracket","base":{"length_a_in":6.5,"length_b_in":6.5,"height_in":1.25,"thickness_in":0.11},"holes":[{"face":"leg_a","from_end_in":3.75,"across_in":null,"diameter_in":0.375},{"face":"leg_a","from_end_in":5.5,"across_in":0.5,"diameter_in":0.625},{"face":"leg_b","from_end_in":1.875,"across_in":null,"diameter_in":0.5}],"text":"Leg A: 3/8 inch hole 3 3/4 inches from the end, centered across, and a 5/8 inch hole 5 1/2 inches from the end, 1/2 inch up. Leg B: a 1/2 inch hole 1 7/8 inches from the end, centered across.","group":"L_brackets:54","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":30,"part_type":"bracket","base":{"length_a_in":1.0,"length_b_in":1.0,"height_in":0.5,"thickness_in":0.08},"holes":[{"face":"leg_a","from_end_in":0.375,"across_in":0.25,"diameter_in":0.1875},{"face":"leg_b","from_end_in":0.375,"across_in":0.25,"diameter_in":0.1875},{"face":"leg_b","from_end_in":0.625,"across_in":0.25,"diameter_in":0.1875}],"text":"Three 3/16 inch holes, all 1/4 inch up from the bottom. One in leg A 3/8 inch from the end. Two in leg B, at 3/8 inch and 5/8 inch from the end.","group":"L_brackets:5","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":31,"part_type":"bracket","base":{"length_a_in":2.0,"length_b_in":3.5625,"height_in":0.8125,"thickness_in":0.24},"holes":[{"face":"leg_a","from_end_in":0.375,"across_in":0.375,"diameter_in":0.5}],"text":"Put a 1/2 inch hole in the short leg, 3/8 inch from the end and 3/8 inch up from the bottom.","group":"L_brackets:65","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":32,"part_type":"bracket","base":{"length_a_in":4.75,"length_b_in":4.75,"height_in":1.25,"thickness_in":0.11},"holes":[{"face":"leg_b","from_end_in":1.875,"across_in":0.375,"diameter_in":0.5}],"text":"Leg B, one 1/2 inch hole, 1 7/8 inches from the end, 3/8 inch up.","group":"L_brackets:53","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":33,"part_type":"bracket","base":{"length_a_in":2.5625,"length_b_in":4.75,"height_in":0.8125,"thickness_in":0.24},"holes":[{"face":"leg_b","from_end_in":3.375,"across_in":0.375,"diameter_in":0.125},{"face":"leg_b","from_end_in":0.75,"across_in":0.375,"diameter_in":0.5}],"text":"On the long leg: a 1/8 inch hole 3 3/8 inches from the end and a 1/2 inch hole 3/4 inch from the end, both 3/8 inch up from the bottom.","group":"L_brackets:66","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":34,"part_type":"bracket","base":{"length_a_in":2.9528,"length_b_in":1.9685,"height_in":1.9685,"thickness_in":0.1181},"holes":[{"face":"leg_a","from_end_in":1.875,"across_in":0.75,"diameter_in":0.625},{"face":"leg_a","from_end_in":0.75,"across_in":1.0,"diameter_in":0.625},{"face":"leg_b","from_end_in":0.875,"across_in":1.625,"diameter_in":0.3125}],"text":"Long leg: two 5/8 inch holes, one 1 7/8 inches from the end and 3/4 inch up, the other 3/4 inch from the end and 1 inch up. Short leg: one 5/16 inch hole 7/8 inch from the end and 1 5/8 inches up.","group":"L_brackets:91","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":35,"part_type":"bracket","base":{"length_a_in":8.25,"length_b_in":8.25,"height_in":5.0,"thickness_in":0.25},"holes":[{"face":"leg_a","from_end_in":4.125,"across_in":1.125,"diameter_in":0.25}],"text":"Drill a 1/4 inch hole in leg A, 4 1/8 inches from the end, 1 1/8 inches up from the bottom face.","group":"L_brackets:50","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":36,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":1.5,"thickness_in":0.06},"holes":[{"face":"leg_a","from_end_in":0.625,"across_in":null,"diameter_in":0.125},{"face":"leg_b","from_end_in":0.625,"across_in":0.875,"diameter_in":0.375}],"text":"Leg A gets a 1/8 inch hole 5/8 inch from the end, centered across. Leg B gets a 3/8 inch hole 5/8 inch from the end and 7/8 inch up.","group":"L_brackets:34","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":37,"part_type":"bracket","base":{"length_a_in":2.9528,"length_b_in":2.3622,"height_in":2.9528,"thickness_in":0.1181},"holes":[{"face":"leg_b","from_end_in":0.375,"across_in":0.375,"diameter_in":0.1875}],"text":"3/16 inch hole in the shorter leg, 3/8 inch from the end and 3/8 inch up from the bottom.","group":"L_brackets:92","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":38,"part_type":"bracket","base":{"length_a_in":9.375,"length_b_in":11.875,"height_in":0.8125,"thickness_in":0.31},"holes":[{"face":"leg_b","from_end_in":2.875,"across_in":0.5,"diameter_in":0.25}],"text":"quarter inch hole in the longer leg, 2 7/8 inches from the end, 1/2 inch up","group":"L_brackets:64","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":39,"part_type":"bracket","base":{"length_a_in":1.5,"length_b_in":1.5,"height_in":0.5,"thickness_in":0.07},"holes":[{"face":"leg_b","from_end_in":1.0,"across_in":null,"diameter_in":0.125},{"face":"leg_a","from_end_in":0.25,"across_in":0.25,"diameter_in":0.125}],"text":"Two 1/8 inch holes. Leg B: 1 inch from the end, centered across. Leg A: 1/4 inch from the end and 1/4 inch up from the bottom.","group":"L_brackets:18","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":40,"part_type":"bracket","base":{"length_a_in":2.5,"length_b_in":2.5,"height_in":5.0,"thickness_in":0.12},"holes":[{"face":"leg_a","from_end_in":1.875,"across_in":3.0,"diameter_in":0.625},{"face":"leg_a","from_end_in":2.125,"across_in":null,"diameter_in":0.1875}],"text":"Leg A: 5/8 inch hole at 1 7/8 inches from the end and 3 inches up; and a 3/16 inch hole 2 1/8 inches from the end, centered across.","group":"L_brackets:46","source":"claude_written"},{"sheet":"Holes_bracket","sheet_row":41,"part_type":"bracket","base":{"length_a_in":1.375,"length_b_in":1.375,"height_in":2.75,"thickness_in":0.06},"holes":[{"face":"leg_a","from_end_in":1.125,"across_in":null,"diameter_in":0.125},{"face":"leg_a","from_end_in":null,"across_in":0.75,"diameter_in":0.3125}],"text":"On leg A, a 1/8 inch hole 1 1/8 inches from the end (centered across the width), and a 5/16 inch hole centered along the leg at 3/4 inch up.","group":"L_brackets:37","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":2,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":3.13,"height_in":3.29,"thickness_in":0.092},"holes":[{"face":"left","along_in":1.125,"up_in":1.875,"diameter_in":0.5}],"text":"Cut a 1/2 inch hole in the left wall, 1 1/8 inch from its left end and 1 7/8 inches up from the bottom.","group":"Rect_container:40","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":3,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":4.5,"height_in":4.95,"thickness_in":0.132},"holes":[{"face":"left","along_in":null,"up_in":null,"diameter_in":0.75},{"face":"right","along_in":3.625,"up_in":null,"diameter_in":0.75}],"text":"I want two 3/4 inch holes. One goes dead center in the left wall. The other is in the right wall, 3 5/8 inches from the left end, centered vertically.","group":"Rect_container:70","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":4,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":2.98,"height_in":1.19,"thickness_in":0.111},"holes":[{"face":"left","along_in":1.625,"up_in":null,"diameter_in":0.1875}],"text":"3/16 inch hole in the left side, 1.625 inches from the left edge, halfway up the wall.","group":"Rect_container:44","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":5,"part_type":"rect_container","base":{"length_a_in":8.0,"length_b_in":3.78,"height_in":4.08,"thickness_in":0.133},"holes":[{"face":"left","along_in":0.875,"up_in":2.25,"diameter_in":0.5}],"text":"Drill a half inch hole through the left wall 7/8 inch from the left end and 2 1/4 inches above the bottom.","group":"Rect_container:86","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":6,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":3.18,"height_in":1.18,"thickness_in":0.089},"holes":[{"face":"right","along_in":1.625,"up_in":0.625,"diameter_in":0.375},{"face":"back","along_in":null,"up_in":0.625,"diameter_in":0.625}],"text":"Two holes: a 3/8 inch hole in the right wall 1 5/8 inches from the left end and 5/8 inch up, and a 5/8 inch hole in the back wall centered side to side at 5/8 inch up.","group":"Rect_container:31","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":7,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.0,"height_in":2.0,"thickness_in":0.11},"holes":[{"face":"right","along_in":2.875,"up_in":1.375,"diameter_in":0.625}],"text":"Put a 5/8\" hole in the right wall, 2 7/8\" from the left end, 1 3/8\" up from the bottom.","group":"Rect_container:57","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":8,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.97,"height_in":2.63,"thickness_in":0.07},"holes":[{"face":"back","along_in":null,"up_in":1.5,"diameter_in":0.25},{"face":"back","along_in":1.25,"up_in":null,"diameter_in":0.125}],"text":"Back wall, two holes. First a 1/4 inch hole centered left to right, 1.5 inches up. Second a 1/8 inch hole 1 1/4 inches from the left end, centered vertically.","group":"Rect_container:10","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":9,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":2.0,"height_in":4.0,"thickness_in":0.09},"holes":[{"face":"right","along_in":null,"up_in":3.5,"diameter_in":0.125}],"text":"small 1/8 inch hole on the right side, centered horizontally, 3.5 inches up from the bottom","group":"Rect_container:90","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":10,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":1.94,"height_in":2.4,"thickness_in":0.091},"holes":[{"face":"front","along_in":0.875,"up_in":0.625,"diameter_in":0.5},{"face":"back","along_in":null,"up_in":1.125,"diameter_in":0.125}],"text":"Add a 1/2 inch hole to the front wall, 7/8 inch from the left end and 5/8 inch up. Also add a 1/8 inch hole in the back wall, centered across, 1 1/8 inches up.","group":"Rect_container:80","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":11,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.38,"height_in":0.78,"thickness_in":0.063},"holes":[{"face":"right","along_in":0.875,"up_in":0.5,"diameter_in":0.3125}],"text":"5/16 inch hole in the right wall, 7/8 inch from the left end, half inch up from the bottom","group":"Rect_container:11","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":12,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.91,"height_in":3.8,"thickness_in":0.087},"holes":[{"face":"left","along_in":2.375,"up_in":3.375,"diameter_in":0.5}],"text":"Left wall: one 1/2 inch hole, 2 3/8 inches from the left end and 3 3/8 inches up.","group":"Rect_container:27","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":13,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":3.15,"height_in":3.28,"thickness_in":0.12},"holes":[{"face":"right","along_in":1.75,"up_in":1.875,"diameter_in":0.125}],"text":"Drill a 1/8 inch hole in the right side, 1 3/4 inch in from the left end and 1 7/8 inches up.","group":"Rect_container:65","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":14,"part_type":"rect_container","base":{"length_a_in":6.0,"length_b_in":4.69,"height_in":4.64,"thickness_in":0.123},"holes":[{"face":"front","along_in":0.625,"up_in":2.25,"diameter_in":0.75},{"face":"back","along_in":1.875,"up_in":2.125,"diameter_in":0.625},{"face":"left","along_in":3.75,"up_in":2.5,"diameter_in":0.25}],"text":"Three holes. Front wall: 3/4 inch hole, 5/8 inch from the left end, 2 1/4 inches up. Back wall: 5/8 inch hole, 1 7/8 inches from the left end, 2 1/8 inches up. Left wall: 1/4 inch hole, 3 3/4 inches from the left end, 2 1/2 inches up.","group":"Rect_container:59","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":15,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.54,"height_in":3.04,"thickness_in":0.08},"holes":[{"face":"right","along_in":0.625,"up_in":0.75,"diameter_in":0.125},{"face":"front","along_in":0.375,"up_in":0.5,"diameter_in":0.25},{"face":"front","along_in":1.875,"up_in":null,"diameter_in":0.5}],"text":"On the right wall add a 1/8 inch hole 5/8 inch from the left end and 3/4 inch up. On the front wall add a 1/4 inch hole 3/8 inch from the left end and 1/2 inch up, and a 1/2 inch hole 1 7/8 inches from the left end, centered vertically.","group":"Rect_container:79","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":16,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":2.33,"height_in":1.72,"thickness_in":0.062},"holes":[{"face":"back","along_in":1.125,"up_in":0.875,"diameter_in":0.75}],"text":"I need a 3/4 inch hole in the back wall, 1 1/8 inches from the left end and 7/8 inch up.","group":"Rect_container:15","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":17,"part_type":"rect_container","base":{"length_a_in":4.0,"length_b_in":2.72,"height_in":3.16,"thickness_in":0.083},"holes":[{"face":"left","along_in":1.375,"up_in":1.875,"diameter_in":0.75},{"face":"right","along_in":1.875,"up_in":null,"diameter_in":0.1875}],"text":"Left wall gets a 3/4 inch hole 1 3/8 inches from the left end and 1 7/8 inches up. Right wall gets a 3/16 inch hole 1 7/8 inches from the left end, centered vertically.","group":"Rect_container:33","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":18,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.99,"height_in":3.84,"thickness_in":0.081},"holes":[{"face":"left","along_in":0.625,"up_in":1.375,"diameter_in":0.625},{"face":"right","along_in":2.0,"up_in":3.0,"diameter_in":0.125},{"face":"back","along_in":1.75,"up_in":null,"diameter_in":0.625}],"text":"Three holes: a 5/8 inch one in the left wall at 5/8 inch from the left end and 1 3/8 inches up; a 1/8 inch one in the right wall 2 inches from the left end and 3 inches up; and a 5/8 inch one in the back wall 1 3/4 inches from the left end, centered vertically.","group":"Rect_container:30","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":19,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":1.47,"height_in":0.67,"thickness_in":0.075},"holes":[{"face":"left","along_in":0.375,"up_in":null,"diameter_in":0.25},{"face":"front","along_in":0.875,"up_in":0.375,"diameter_in":0.25},{"face":"right","along_in":0.375,"up_in":0.375,"diameter_in":0.1875}],"text":"left wall: 1/4 inch hole, 3/8 inch from the left end, centered vertically. front wall: 1/4 inch hole, 7/8 inch from the left end, 3/8 inch up. right wall: 3/16 inch hole, 3/8 inch from the left end, 3/8 inch up.","group":"Rect_container:20","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":20,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.43,"height_in":1.69,"thickness_in":0.082},"holes":[{"face":"right","along_in":1.0,"up_in":null,"diameter_in":0.3125}],"text":"A 5/16 inch hole in the right wall, 1 inch from the left end, vertically centered.","group":"Rect_container:12","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":21,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":2.6,"height_in":2.11,"thickness_in":0.098},"holes":[{"face":"front","along_in":2.375,"up_in":0.875,"diameter_in":0.3125},{"face":"left","along_in":1.5,"up_in":1.375,"diameter_in":0.5},{"face":"right","along_in":0.75,"up_in":0.625,"diameter_in":0.625}],"text":"Front wall: 5/16 inch hole 2 3/8 inches from the left end, 7/8 inch up. Left wall: 1/2 inch hole 1.5 inches from the left end, 1 3/8 inches up. Right wall: 5/8 inch hole 3/4 inch from the left end, 5/8 inch up.","group":"Rect_container:36","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":22,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.81,"height_in":4.28,"thickness_in":0.101},"holes":[{"face":"back","along_in":0.75,"up_in":0.625,"diameter_in":0.75},{"face":"back","along_in":1.875,"up_in":3.25,"diameter_in":0.3125},{"face":"left","along_in":0.75,"up_in":null,"diameter_in":0.375}],"text":"Back wall gets two holes: 3/4 inch, 3/4 inch from the left end and 5/8 inch up, and 5/16 inch, 1 7/8 inches from the left end and 3 1/4 inches up. The left wall gets one 3/8 inch hole 3/4 inch from the left end, centered vertically.","group":"Rect_container:25","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":23,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.19,"height_in":2.18,"thickness_in":0.074},"holes":[{"face":"left","along_in":0.875,"up_in":1.75,"diameter_in":0.1875}],"text":"3/16 inch hole in the left wall, 7/8 inch from the left end, 1 3/4 inches up from the bottom","group":"Rect_container:24","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":24,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":3.69,"height_in":5.68,"thickness_in":0.119},"holes":[{"face":"right","along_in":null,"up_in":null,"diameter_in":0.75},{"face":"back","along_in":1.75,"up_in":null,"diameter_in":0.1875}],"text":"Put a 3/4 inch hole in the exact center of the right wall, and a 3/16 inch hole in the back wall, 1 3/4 inches from the left end and centered vertically.","group":"Rect_container:53","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":25,"part_type":"rect_container","base":{"length_a_in":2.5,"length_b_in":1.74,"height_in":1.62,"thickness_in":0.065},"holes":[{"face":"front","along_in":null,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole dead center on the front wall","group":"Rect_container:13","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":26,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.04,"height_in":1.14,"thickness_in":0.07},"holes":[{"face":"right","along_in":1.125,"up_in":0.625,"diameter_in":0.625},{"face":"back","along_in":1.875,"up_in":0.75,"diameter_in":0.25},{"face":"left","along_in":0.875,"up_in":0.625,"diameter_in":0.125}],"text":"Right wall: a 5/8 inch hole, 1 1/8 inches from the left end and 5/8 inch up. Back wall: a 1/4 inch hole, 1 7/8 inches from the left end and 3/4 inch up. Left wall: a 1/8 inch hole, 7/8 inch from the left end and 5/8 inch up.","group":"Rect_container:22","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":27,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.33,"height_in":1.33,"thickness_in":0.059},"holes":[{"face":"back","along_in":1.0,"up_in":0.625,"diameter_in":0.625}],"text":"add a 5/8 inch hole to the back, 1 inch from the left end and 5/8 inch up","group":"Rect_container:9","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":28,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.69,"height_in":5.16,"thickness_in":0.106},"holes":[{"face":"back","along_in":4.5,"up_in":2.0,"diameter_in":0.375}],"text":"Back wall, 3/8 inch hole, 4 1/2 inches from the left end and 2 inches up from the bottom.","group":"Rect_container:49","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":29,"part_type":"rect_container","base":{"length_a_in":2.0,"length_b_in":1.01,"height_in":0.5,"thickness_in":0.066},"holes":[{"face":"left","along_in":0.5,"up_in":null,"diameter_in":0.125},{"face":"back","along_in":0.375,"up_in":0.25,"diameter_in":0.125},{"face":"back","along_in":1.125,"up_in":0.25,"diameter_in":0.125}],"text":"Three 1/8 inch holes. One in the left wall, 1/2 inch from the left end, centered vertically. Two in the back wall, both 1/4 inch up, at 3/8 inch and 1 1/8 inches from the left end.","group":"Rect_container:7","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":30,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.16,"height_in":1.62,"thickness_in":0.067},"holes":[{"face":"left","along_in":1.375,"up_in":1.125,"diameter_in":0.625}],"text":"Drill a 5/8 inch hole in the left wall at 1 3/8 inches from the left end and 1 1/8 inches up.","group":"Rect_container:23","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":31,"part_type":"rect_container","base":{"length_a_in":4.5,"length_b_in":2.88,"height_in":1.67,"thickness_in":0.092},"holes":[{"face":"front","along_in":2.625,"up_in":1.125,"diameter_in":0.625}],"text":"Front wall hole: 5/8 inch diameter, 2 5/8 inches from the left end, 1 1/8 inches up from the bottom.","group":"Rect_container:37","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":32,"part_type":"rect_container","base":{"length_a_in":3.0,"length_b_in":2.85,"height_in":2.65,"thickness_in":0.086},"holes":[{"face":"left","along_in":1.375,"up_in":1.875,"diameter_in":0.125}],"text":"small hole, 1/8 inch, in the left wall 1 3/8 inches from the left end and 1 7/8 inches up","group":"Rect_container:26","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":33,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.49,"height_in":1.02,"thickness_in":0.054},"holes":[{"face":"left","along_in":0.5,"up_in":null,"diameter_in":0.5},{"face":"front","along_in":0.5,"up_in":0.5,"diameter_in":0.625}],"text":"Left wall: a 1/2 inch hole 1/2 inch from the left end, centered vertically. Front wall: a 5/8 inch hole 1/2 inch from the left end and 1/2 inch up.","group":"Rect_container:6","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":34,"part_type":"rect_container","base":{"length_a_in":6.5,"length_b_in":3.88,"height_in":5.32,"thickness_in":0.123},"holes":[{"face":"back","along_in":5.875,"up_in":4.875,"diameter_in":0.625},{"face":"back","along_in":3.875,"up_in":1.125,"diameter_in":0.625}],"text":"Two 5/8 inch holes in the back wall: one 5 7/8 inches from the left end and 4 7/8 inches up, the other 3 7/8 inches from the left end and 1 1/8 inches up.","group":"Rect_container:62","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":35,"part_type":"rect_container","base":{"length_a_in":7.0,"length_b_in":6.3,"height_in":6.93,"thickness_in":0.135},"holes":[{"face":"front","along_in":4.875,"up_in":0.375,"diameter_in":0.25}],"text":"front wall, 1/4 inch hole, 4 7/8 inches from the left end, 3/8 inch up","group":"Rect_container:68","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":36,"part_type":"rect_container","base":{"length_a_in":3.6,"length_b_in":6.3,"height_in":2.0,"thickness_in":0.14},"holes":[{"face":"front","along_in":0.75,"up_in":1.375,"diameter_in":0.375}],"text":"Add a 3/8 inch hole to the front wall, 3/4 inch from the left end and 1 3/8 inches above the bottom.","group":"Rect_container:89","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":37,"part_type":"rect_container","base":{"length_a_in":5.0,"length_b_in":4.5,"height_in":2.1,"thickness_in":0.097},"holes":[{"face":"back","along_in":2.875,"up_in":null,"diameter_in":0.625},{"face":"front","along_in":3.0,"up_in":1.375,"diameter_in":0.375},{"face":"right","along_in":1.125,"up_in":null,"diameter_in":0.25}],"text":"Back wall: 5/8 inch hole 2 7/8 inches from the left end, centered vertically. Front wall: 3/8 inch hole 3 inches from the left end and 1 3/8 inches up. Right wall: 1/4 inch hole 1 1/8 inches from the left end, centered vertically.","group":"Rect_container:83","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":38,"part_type":"rect_container","base":{"length_a_in":1.5,"length_b_in":1.0,"height_in":1.04,"thickness_in":0.07},"holes":[{"face":"back","along_in":0.875,"up_in":0.25,"diameter_in":0.125},{"face":"back","along_in":0.5,"up_in":null,"diameter_in":0.25},{"face":"front","along_in":0.875,"up_in":0.5,"diameter_in":0.125}],"text":"Three holes. On the back wall a 1/8 inch hole 7/8 inch from the left end and 1/4 inch up, and a 1/4 inch hole 1/2 inch from the left end, centered vertically. On the front wall a 1/8 inch hole 7/8 inch from the left end and 1/2 inch up.","group":"Rect_container:3","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":39,"part_type":"rect_container","base":{"length_a_in":3.5,"length_b_in":2.82,"height_in":3.54,"thickness_in":0.078},"holes":[{"face":"left","along_in":2.125,"up_in":1.0,"diameter_in":0.25},{"face":"left","along_in":null,"up_in":1.375,"diameter_in":0.3125},{"face":"back","along_in":1.5,"up_in":1.75,"diameter_in":0.25}],"text":"Left wall: a 1/4 inch hole at 2 1/8 inches from the left end and 1 inch up, plus a 5/16 inch hole centered left to right at 1 3/8 inches up. Back wall: a 1/4 inch hole 1.5 inches from the left end and 1 3/4 inches up.","group":"Rect_container:29","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":40,"part_type":"rect_container","base":{"length_a_in":7.5,"length_b_in":5.25,"height_in":3.15,"thickness_in":0.131},"holes":[{"face":"back","along_in":1.75,"up_in":1.375,"diameter_in":0.1875},{"face":"front","along_in":0.875,"up_in":2.375,"diameter_in":0.25},{"face":"front","along_in":null,"up_in":2.5,"diameter_in":0.1875}],"text":"Back wall: 3/16 inch hole, 1 3/4 inches from the left end, 1 3/8 inches up. Front wall: 1/4 inch hole, 7/8 inch from the left end, 2 3/8 inches up; and a 3/16 inch hole centered left to right, 2 1/2 inches up.","group":"Rect_container:71","source":"claude_written"},{"sheet":"Holes_rect","sheet_row":41,"part_type":"rect_container","base":{"length_a_in":5.5,"length_b_in":4.88,"height_in":3.76,"thickness_in":0.105},"holes":[{"face":"back","along_in":1.875,"up_in":null,"diameter_in":0.1875}],"text":"3/16 inch hole in the back wall, 1 7/8 inches from the left end, vertically centered.","group":"Rect_container:54","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":2,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":9.45,"thickness_in":0.1571},"holes":[{"face":"side","angle_deg":225.0,"up_in":7.75,"diameter_in":0.3125}],"text":"Add a 5/16 inch hole at 225 degrees around the cup, 7.75 inches up from the bottom.","group":"Cyl_container:54","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":3,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.37,"thickness_in":0.0523},"holes":[{"face":"side","angle_deg":165.0,"up_in":0.5,"diameter_in":0.625}],"text":"5/8 inch hole at an angle of 165 degrees, half an inch up.","group":"Cyl_container:12","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":4,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":4.73,"thickness_in":0.0931},"holes":[{"face":"side","angle_deg":345.0,"up_in":0.625,"diameter_in":0.25}],"text":"Drill a 1/4 inch hole at 345\u00b0, 5/8 inch above the bottom.","group":"Cyl_container:28","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":5,"part_type":"cyl_container","base":{"id_in":3.5,"height_in":3.38,"thickness_in":0.0921},"holes":[{"face":"side","angle_deg":30.0,"up_in":1.75,"diameter_in":0.625}],"text":"Put a 5/8 inch hole at 30 degrees and 1.75 inches up from the bottom of the cup.","group":"Cyl_container:42","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":6,"part_type":"cyl_container","base":{"id_in":6.5,"height_in":5.99,"thickness_in":0.1197},"holes":[{"face":"side","angle_deg":315.0,"up_in":5.25,"diameter_in":0.1875}],"text":"3/16 inch hole at 315 degrees, 5 1/4 inches up.","group":"Cyl_container:67","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":7,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":255.0,"up_in":7.125,"diameter_in":0.375}],"text":"Please add a 3/8 inch hole at 255 degrees, 7 1/8 inches from the bottom.","group":"Cyl_container:86","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":8,"part_type":"cyl_container","base":{"id_in":2.0,"height_in":3.07,"thickness_in":0.0813},"holes":[{"face":"side","angle_deg":135.0,"up_in":2.75,"diameter_in":0.3125}],"text":"a 5/16 hole at 135 degrees, 2.75 inches above the bottom","group":"Cyl_container:31","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":9,"part_type":"cyl_container","base":{"id_in":6.5,"height_in":8.27,"thickness_in":0.132},"holes":[{"face":"side","angle_deg":270.0,"up_in":0.75,"diameter_in":0.25}],"text":"1/4 inch hole on the front of the cup, 3/4 inch up from the bottom.","group":"Cyl_container:89","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":10,"part_type":"cyl_container","base":{"id_in":2.25,"height_in":0.81,"thickness_in":0.0606},"holes":[{"face":"side","angle_deg":180.0,"up_in":0.375,"diameter_in":0.25},{"face":"side","angle_deg":195.0,"up_in":0.375,"diameter_in":0.125}],"text":"Two holes both 3/8 inch up from the bottom: a 1/4 inch hole on the left side, and a 1/8 inch hole at 195 degrees.","group":"Cyl_container:32","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":11,"part_type":"cyl_container","base":{"id_in":2.5,"height_in":4.97,"thickness_in":0.1014},"holes":[{"face":"side","angle_deg":90.0,"up_in":4.0,"diameter_in":0.25},{"face":"side","angle_deg":210.0,"up_in":1.875,"diameter_in":0.375}],"text":"Add a 1/4 inch hole in the back, 4 inches up, and a 3/8 inch hole at 210 degrees, 1 7/8 inches up.","group":"Cyl_container:35","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":12,"part_type":"cyl_container","base":{"id_in":7.0,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.3125},{"face":"side","angle_deg":120.0,"up_in":8.75,"diameter_in":0.375},{"face":"side","angle_deg":240.0,"up_in":10.875,"diameter_in":0.25}],"text":"Three holes. A 5/16 inch hole at 195 degrees, centered vertically. A 3/8 inch hole at 120 degrees, 8.75 inches up. A 1/4 inch hole at 240 degrees, 10 7/8 inches up.","group":"Cyl_container:70","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":13,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":12.04,"thickness_in":0.1779},"holes":[{"face":"side","angle_deg":165.0,"up_in":null,"diameter_in":0.375},{"face":"side","angle_deg":105.0,"up_in":2.5,"diameter_in":0.5}],"text":"3/8 inch hole at 165 degrees, halfway up the cup, and a half inch hole at 105 degrees, 2.5 inches up.","group":"Cyl_container:59","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":14,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":2.01,"thickness_in":0.0658},"holes":[{"face":"side","angle_deg":210.0,"up_in":null,"diameter_in":0.5}],"text":"Put a half inch hole at 210 degrees, centered vertically.","group":"Cyl_container:26","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":15,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":0.5,"thickness_in":0.0519},"holes":[{"face":"side","angle_deg":180.0,"up_in":0.25,"diameter_in":0.125},{"face":"side","angle_deg":60.0,"up_in":0.25,"diameter_in":0.125},{"face":"side","angle_deg":135.0,"up_in":0.25,"diameter_in":0.125}],"text":"Three 1/8 inch holes all a quarter inch up from the bottom, at 60, 135 and 180 degrees.","group":"Cyl_container:8","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":16,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":10.48,"thickness_in":0.1604},"holes":[{"face":"side","angle_deg":150.0,"up_in":6.5,"diameter_in":0.25},{"face":"side","angle_deg":120.0,"up_in":7.625,"diameter_in":0.25}],"text":"Two 1/4 inch holes: one at 150 degrees, 6.5 inches up, and one at 120 degrees, 7 5/8 inches up.","group":"Cyl_container:55","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":17,"part_type":"cyl_container","base":{"id_in":2.25,"height_in":5.72,"thickness_in":0.1093},"holes":[{"face":"side","angle_deg":105.0,"up_in":null,"diameter_in":0.625}],"text":"5/8 inch hole at 105\u00b0, mid-height","group":"Cyl_container:34","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":18,"part_type":"cyl_container","base":{"id_in":7.5,"height_in":11.63,"thickness_in":0.1682},"holes":[{"face":"side","angle_deg":15.0,"up_in":10.5,"diameter_in":0.3125}],"text":"Drill a 5/16 inch hole at 15 degrees, 10.5 inches up from the bottom.","group":"Cyl_container:71","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":19,"part_type":"cyl_container","base":{"id_in":8.5,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":315.0,"up_in":3.75,"diameter_in":0.125}],"text":"1/8 inch hole at an angle of 315 degrees, 3 3/4 inches from the bottom.","group":"Cyl_container:78","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":20,"part_type":"cyl_container","base":{"id_in":4.0,"height_in":7.73,"thickness_in":0.1272},"holes":[{"face":"side","angle_deg":165.0,"up_in":null,"diameter_in":0.5},{"face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.375}],"text":"Two holes, both centered vertically: a half inch hole at 165 degrees and a 3/8 inch hole at 195 degrees.","group":"Cyl_container:49","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":21,"part_type":"cyl_container","base":{"id_in":1.25,"height_in":2.36,"thickness_in":0.0691},"holes":[{"face":"side","angle_deg":150.0,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole at 150 degrees, halfway up.","group":"Cyl_container:19","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":22,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":0.54,"thickness_in":0.0638},"holes":[{"face":"side","angle_deg":225.0,"up_in":0.375,"diameter_in":0.125},{"face":"side","angle_deg":165.0,"up_in":0.375,"diameter_in":0.125}],"text":"Two 1/8 inch holes, 3/8 inch up from the bottom, at 165 and 225 degrees.","group":"Cyl_container:20","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":23,"part_type":"cyl_container","base":{"id_in":0.75,"height_in":1.79,"thickness_in":0.06},"holes":[{"face":"side","angle_deg":300.0,"up_in":1.5,"diameter_in":0.375},{"face":"side","angle_deg":0.0,"up_in":0.875,"diameter_in":0.5}],"text":"A 3/8 inch hole at 300 degrees, 1.5 inches up, and a half inch hole on the right side, 7/8 inch up.","group":"Cyl_container:7","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":24,"part_type":"cyl_container","base":{"id_in":3.25,"height_in":8.58,"thickness_in":0.1473},"holes":[{"face":"side","angle_deg":285.0,"up_in":4.75,"diameter_in":0.125},{"face":"side","angle_deg":210.0,"up_in":6.5,"diameter_in":0.1875}],"text":"1/8 inch hole at 285 degrees and 4 3/4 inches up; 3/16 inch hole at 210 degrees and 6.5 inches up.","group":"Cyl_container:41","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":25,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":13.06,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":135.0,"up_in":8.0,"diameter_in":0.25}],"text":"Add a 1/4 inch hole at 135 degrees, 8 inches above the bottom.","group":"Cyl_container:75","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":26,"part_type":"cyl_container","base":{"id_in":9.0,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":120.0,"up_in":null,"diameter_in":0.25},{"face":"side","angle_deg":240.0,"up_in":null,"diameter_in":0.3125},{"face":"side","angle_deg":105.0,"up_in":11.0,"diameter_in":0.625}],"text":"Three holes. A 1/4 inch hole at 120 degrees and a 5/16 inch hole at 240 degrees, both centered vertically. A 5/8 inch hole at 105 degrees, 11 inches up.","group":"Cyl_container:80","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":27,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":7.68,"thickness_in":0.1558},"holes":[{"face":"side","angle_deg":120.0,"up_in":7.375,"diameter_in":0.1875},{"face":"side","angle_deg":330.0,"up_in":null,"diameter_in":0.125}],"text":"3/16 inch hole at 120 degrees, 7 3/8 inches up. Also a 1/8 inch hole at 330 degrees, vertically centered.","group":"Cyl_container:82","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":28,"part_type":"cyl_container","base":{"id_in":8.0,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":75.0,"up_in":null,"diameter_in":0.25}],"text":"quarter inch hole at 75 degrees, in the middle of the height","group":"Cyl_container:77","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":29,"part_type":"cyl_container","base":{"id_in":4.5,"height_in":9.18,"thickness_in":0.1537},"holes":[{"face":"side","angle_deg":165.0,"up_in":6.5,"diameter_in":0.75},{"face":"side","angle_deg":300.0,"up_in":null,"diameter_in":0.75}],"text":"Two 3/4 inch holes. One at 165 degrees, 6.5 inches up. One at 300 degrees, centered vertically.","group":"Cyl_container:53","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":30,"part_type":"cyl_container","base":{"id_in":10.0,"height_in":14.0,"thickness_in":0.18},"holes":[{"face":"side","angle_deg":30.0,"up_in":13.5,"diameter_in":0.25}],"text":"1/4 inch hole at 30 degrees, 13.5 inches up from the bottom, near the top.","group":"Cyl_container:83","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":31,"part_type":"cyl_container","base":{"id_in":1.25,"height_in":1.42,"thickness_in":0.0686},"holes":[{"face":"side","angle_deg":210.0,"up_in":1.0,"diameter_in":0.375}],"text":"3/8 inch hole at 210 degrees, 1 inch up.","group":"Cyl_container:15","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":32,"part_type":"cyl_container","base":{"id_in":1.75,"height_in":2.12,"thickness_in":0.0711},"holes":[{"face":"side","angle_deg":225.0,"up_in":null,"diameter_in":0.25}],"text":"Put a 1/4 inch hole at 225 degrees, halfway up the wall.","group":"Cyl_container:27","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":33,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":3.17,"thickness_in":0.0919},"holes":[{"face":"side","angle_deg":330.0,"up_in":2.0,"diameter_in":0.625}],"text":"5/8 inch hole at 330\u00b0, 2 inches above the bottom.","group":"Cyl_container:45","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":34,"part_type":"cyl_container","base":{"id_in":0.5,"height_in":0.5,"thickness_in":0.0526},"holes":[{"face":"side","angle_deg":135.0,"up_in":0.25,"diameter_in":0.125},{"face":"side","angle_deg":75.0,"up_in":0.25,"diameter_in":0.125}],"text":"Two tiny 1/8 inch holes, a quarter inch up, at 75 and 135 degrees.","group":"Cyl_container:3","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":35,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.33,"thickness_in":0.0603},"holes":[{"face":"side","angle_deg":180.0,"up_in":0.75,"diameter_in":0.375}],"text":"3/8 inch hole on the left side of the cup, 3/4 inch up.","group":"Cyl_container:11","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":36,"part_type":"cyl_container","base":{"id_in":1.5,"height_in":2.44,"thickness_in":0.0648},"holes":[{"face":"side","angle_deg":255.0,"up_in":null,"diameter_in":0.625},{"face":"side","angle_deg":75.0,"up_in":1.375,"diameter_in":0.25}],"text":"A 5/8 inch hole at 255 degrees, centered vertically, and a 1/4 inch hole at 75 degrees, 1 3/8 inches up.","group":"Cyl_container:22","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":37,"part_type":"cyl_container","base":{"id_in":1.0,"height_in":1.72,"thickness_in":0.0588},"holes":[{"face":"side","angle_deg":195.0,"up_in":null,"diameter_in":0.375},{"face":"side","angle_deg":120.0,"up_in":0.625,"diameter_in":0.5},{"face":"side","angle_deg":30.0,"up_in":null,"diameter_in":0.125}],"text":"Three holes. At 195 degrees a 3/8 inch hole centered vertically. At 120 degrees a half inch hole, 5/8 inch up. At 30 degrees a 1/8 inch hole, centered vertically.","group":"Cyl_container:13","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":38,"part_type":"cyl_container","base":{"id_in":5.0,"height_in":8.1,"thickness_in":0.1236},"holes":[{"face":"side","angle_deg":75.0,"up_in":1.375,"diameter_in":0.1875},{"face":"side","angle_deg":225.0,"up_in":null,"diameter_in":0.3125}],"text":"3/16 inch hole at 75 degrees, 1 3/8 inches up, and a 5/16 inch hole at 225 degrees halfway up.","group":"Cyl_container:60","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":39,"part_type":"cyl_container","base":{"id_in":2.75,"height_in":4.38,"thickness_in":0.09},"holes":[{"face":"side","angle_deg":90.0,"up_in":3.25,"diameter_in":0.625}],"text":"Put a 5/8 inch hole in the back of the cup, 3 1/4 inches up.","group":"Cyl_container:38","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":40,"part_type":"cyl_container","base":{"id_in":4.0,"height_in":5.63,"thickness_in":0.0975},"holes":[{"face":"side","angle_deg":90.0,"up_in":2.375,"diameter_in":0.375},{"face":"side","angle_deg":75.0,"up_in":1.875,"diameter_in":0.75},{"face":"side","angle_deg":120.0,"up_in":4.75,"diameter_in":0.1875}],"text":"Three holes: 3/8 inch at 90 degrees and 2 3/8 inches up; 3/4 inch at 75 degrees and 1 7/8 inches up; 3/16 inch at 120 degrees and 4 3/4 inches up.","group":"Cyl_container:88","source":"claude_written"},{"sheet":"Holes_cyl","sheet_row":41,"part_type":"cyl_container","base":{"id_in":3.75,"height_in":6.49,"thickness_in":0.125},"holes":[{"face":"side","angle_deg":90.0,"up_in":null,"diameter_in":0.75}],"text":"3/4 inch hole centered on the back, halfway up.","group":"Cyl_container:46","source":"claude_written"}]'''
print(f"built-in hand-written hole sentences: {len(json.loads(FEATURE_WRITTEN_JSON))}")

In [ ]:
# ------------------------------------------------------------------ OPTIONAL: load YOUR written hole-feature sentences
# Uses the 120 sentences written by Claude that are built into this notebook (or your own workbook if FEATURE_DATA_PATH is set) and adds generated sentences (section 5c). Set LOAD_FEATURE_DATA = False to switch
# feature training off; the app's "Add features" section still works through its editable hole table either way.
FEAT_SHEETS = {"bracket": "Holes_bracket", "rect_container": "Holes_rect", "cyl_container": "Holes_cyl"}
feat_df = pd.DataFrame(); FEAT_READY = False

def load_feature_workbook(path):
    xl = pd.ExcelFile(path); rows = []
    for pt, sheet in FEAT_SHEETS.items():
        if sheet not in xl.sheet_names: continue
        d = xl.parse(sheet, dtype={"text": "string"}); k = HOLE_SPEC[pt]["keys"]          # force text to string: an all-blank column would otherwise load as numbers
        for i, r in d.iterrows():
            text = r.get("text"); text = "" if pd.isna(text) else str(text).strip()
            base = {f: float(r[f]) for f in SCHEMA[pt]}
            holes = []
            for n in range(1, int(r["n_holes"]) + 1):
                g = lambda key: (None if pd.isna(r.get(f"h{n}_{key}")) else r.get(f"h{n}_{key}"))
                holes.append({k[0]: coerce_face(pt, g(k[0])), **{key: coerce_value(key, g(key)) for key in k[1:]}})
            src = r.get("source"); src = "claude_written" if pd.isna(src) else str(src)
            rows.append({"sheet": sheet, "sheet_row": i + 2, "part_type": pt, "base": base, "holes": holes, "text": text, "group": str(r.get("base_ref", i)), "source": src})
    return pd.DataFrame(rows)

if LOAD_FEATURE_DATA:
    if FEATURE_DATA_PATH is not None and Path(FEATURE_DATA_PATH).exists():          # optional: your own edited copy of hole_features_filled.xlsx
        feat_written = load_feature_workbook(FEATURE_DATA_PATH); print("hand-written hole sentences: using your workbook", FEATURE_DATA_PATH)
    else:                                                                           # default: the sentences built into this notebook (no upload)
        feat_written = pd.DataFrame(json.loads(FEATURE_WRITTEN_JSON)); print("hand-written hole sentences: using the built-in set")
    print(f"hole-feature rows: {len(feat_written)} | with text written: {(feat_written.text != '').sum()}")
    feat_written = feat_written[feat_written.text != ""]
    if FEATURE_GENERATED_PER_TYPE > 0:                                       # synthetic extras, never placed in validation/test
        feat_gen = generate_feature_rows(FEATURE_GENERATED_PER_TYPE, SEED, exclude_refs={g for g in feat_written.group})
        print(f"generated hole-feature rows: {len(feat_gen)}")
    else: feat_gen = pd.DataFrame(columns=feat_written.columns)
    feat_df = pd.concat([feat_written, feat_gen], ignore_index=True)
    feat_df["text_norm"] = feat_df.text.map(normalize_text)
    feat_df["target"] = [holes_to_json(r.part_type, r.holes) for r in feat_df.itertuples()]
    # audit: numbers must be readable in the sentence; every row's holes must be geometrically valid (blank positions count as centred)
    def _feat_problems(r):
        probs = [f"value not in text: {b}" for b in audit_holes(r.part_type, r.holes, r.text_norm)] + semantic_hole_problems(r.part_type, r.holes, r.text_norm)
        filled = [{**h, **{key: v for key, v in hole_defaults(r.part_type, r.base, h["face"]).items() if h.get(key) is None}} for h in r.holes]
        probs += [i["msg"] for i in check_holes(r.part_type, r.base, filled) if i["level"] == "error"]
        return probs
    feat_df["problems"] = [_feat_problems(r) for r in feat_df.itertuples()]
    _bad = feat_df[feat_df.problems.map(len) > 0]
    print(f"{len(_bad)} feature rows have problems (fix the sheet row shown):")
    show_df(_bad[["sheet", "sheet_row", "text", "problems"]])
    feat_df = feat_df[feat_df.problems.map(len) == 0].reset_index(drop=True)
    # split: validation/test are drawn ONLY from the hand-written groups (so they are worded differently from the generated training text);
    # generated rows are always training. Whole base parts stay together.
    _rng = random.Random(SEED); _wg = sorted(set(feat_df[feat_df.source != "claude_generated"].group)); _rng.shuffle(_wg)
    _n = len(_wg); _gs = {g: ("test" if i < 0.25 * _n else "val" if i < 0.40 * _n else "train") for i, g in enumerate(_wg)}
    feat_df["split"] = [("train" if r.source == "claude_generated" else _gs[r.group]) for r in feat_df.itertuples()]
    feat_train, feat_val, feat_test = (feat_df[feat_df.split == s].reset_index(drop=True) for s in ("train", "val", "test"))
    FEAT_READY = len(feat_train) >= 10 and len(feat_val) >= 1
    show_df(feat_df.groupby(["source", "split"]).size().unstack(fill_value=0))
    print(f"feature rows used: train {len(feat_train)} | val {len(feat_val)} | test {len(feat_test)} (val/test are hand-written only) | feature training enabled: {FEAT_READY}")
    print("All of these rows are SYNTHETIC (written/generated by Claude) - stored separately from your 501 manual samples.")
else:
    feat_train = feat_val = feat_test = pd.DataFrame()
    print("hole-feature training is off (LOAD_FEATURE_DATA = False). The app's hole table still works.")

## 5d. Part-type phrase data for the classifier (synthetic, written by Claude, stored separately)
The part-type classifier is trained from scratch, so it only knows the words in its training set. Casual wording ("make a box thats 40inches by...") can be mis-typed with high confidence. This cell adds **classification-only** examples with varied nouns (box, crate, jar, corner brace, ...), dimension phrasing and units (inch, fractions, mm, cm, feet). They go into the classifier's training set only - validation and test remain your real texts.

In [ ]:
# ------------------------------------------------------------------ 5d: PART-TYPE PHRASE DATA for the classifier (SYNTHETIC - written by Claude, stored separately)
# The classifier is a small from-scratch model: it only knows words it saw in training. Your 501 texts use a limited vocabulary (e.g. few
# "box" / "enclosure" / "jar" / "corner brace" phrasings), so a casually-worded request can be mis-typed with high confidence. This adds
# classification-ONLY examples (no parameter labels) that vary the noun, the dimension wording and the units (inch, fractions, mm, cm, feet).
# They go into the classifier's TRAINING set only: validation and test stay 100% your real texts.
CLF_NOUNS_BASE = {
    "pipe_gasket": ["pipe gasket", "flange gasket", "gasket", "ring gasket", "flat gasket", "gasket ring", "full face gasket", "pipe flange gasket", "gasket for a pipe flange", "flange seal", "sealing ring"],
    "washer": ["washer", "flat washer", "spacer", "round spacer", "screw spacer", "shim washer", "washer for a bolt", "metric washer", "fender washer", "flat spacer ring"],
    "bracket": ["L bracket", "L-bracket", "angle bracket", "corner bracket", "right angle bracket", "L shaped bracket", "90 degree bracket", "connector bracket"],
    "rect_container": ["rectangular container", "rectangular box", "open top box", "storage box", "rectangular bin", "open rectangular container", "rectangular organizer", "rectangular tray", "open box", "rectangular tub"],
    "cyl_container": ["cylindrical container", "cylindrical cup", "round container", "cylinder", "open cylinder", "cylindrical holder", "cylindrical bin", "round cup", "cup", "cylindrical canister"],
}
CLF_NOUNS_EXTRA = {   # more everyday words; also the nouns used by the hand-written stress prompts, which is why they are kept separate when measuring
    "pipe_gasket": ["seal for a pipe flange", "sealing gasket", "pipe seal"],
    "washer": ["standoff", "bushing spacer", "bushing", "shim", "spacer ring"],
    "bracket": ["corner brace", "L-shaped connector", "right angle mount", "shelf bracket", "angle iron piece", "gusset angle", "angle iron", "angle piece"],
    "rect_container": ["box", "crate", "parts tray", "storage bin", "enclosure", "case", "drawer organizer", "shallow tray", "tub", "rectangular enclosure", "tray", "bin", "organizer", "project box"],
    "cyl_container": ["jar", "bucket", "pot", "tin can", "pencil cup", "round canister", "tube cup", "round holder", "canister", "mug", "round tin"],
}
_LENGTHS = [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 3, 3.5, 4, 4.5, 5, 6, 7, 8, 9, 10, 12, 14, 24]
_THICKS = [0.03125, 0.0625, 0.09375, 0.125, 0.1875, 0.25, 0.3125, 0.05, 0.06, 0.075, 0.09, 0.1, 0.12, 0.15, 0.18, 0.2]

def _q(v, rng):
    """a quantity in a random style: inches (decimal / fraction / words), millimetres, centimetres or feet"""
    r = rng.random()
    if r < 0.12: return f"{max(1, int(round(v * 25.4 / 5) * 5))} mm"
    if r < 0.16: return f"{dec(round(v * 2.54, 1))} cm"
    if r < 0.24 and v in (12, 24): return rng.choice(["a foot", "1 ft", "1 foot"]) if v == 12 else rng.choice(["two feet", "2 ft", "2 feet"])
    return _len(v, rng, words=True)

def _clf_clauses(pt, rng):
    L = lambda: rng.choice(_LENGTHS); T = lambda: rng.choice(_THICKS)
    q = lambda v: _q(v, rng)
    thick = lambda: rng.choice([f"{q(T())} thick", f"{q(T())} walls", f"wall thickness {q(T())}", f"{q(T())} wall", f"{q(T())} thick walls"] if pt in ("rect_container", "cyl_container")
                                else [f"{q(T())} thick", f"thickness {q(T())}", f"{q(T())} thick material", f"{q(T())} tall"])
    if pt == "rect_container":
        a, b, c = L(), L(), L()
        dims = rng.choice([f"{q(a)} by {q(b)} by {q(c)}", f"{q(a)} x {q(b)} x {q(c)}", f"{q(a)} long, {q(b)} wide and {q(c)} tall", f"length {q(a)}, width {q(b)}, height {q(c)}",
                           f"{q(a)} by {q(b)} and {q(c)} deep", f"{q(a)} long by {q(b)} wide, {q(c)} high", f"{q(a)} x {q(b)} footprint, {q(c)} tall"])
        return [dims, thick()] + ([rng.choice(["open top", "no lid", "open on top"])] if rng.random() < 0.3 else [])
    if pt == "cyl_container":
        d, h = L(), L()
        dims = rng.choice([f"{q(d)} inner diameter, {q(h)} tall", f"{q(d)} ID by {q(h)} high", f"{q(d)} wide inside and {q(h)} tall", f"inside diameter {q(d)}, height {q(h)}",
                           f"{q(d)} across, {q(h)} deep", f"{q(d)} diameter, {q(h)} tall", f"{q(h)} tall with a {q(d)} inside diameter"])
        return [dims, thick()] + ([rng.choice(["open top", "open on top"])] if rng.random() < 0.25 else [])
    if pt == "bracket":
        x, y, w = L(), L(), rng.choice([0.5, 0.75, 1, 1.25, 1.5, 2, 3])
        dims = rng.choice([f"legs {q(x)} long", f"{q(x)} legs", f"{q(x)} on each side", f"one leg {q(x)} and the other {q(y)}", f"{q(x)} by {q(y)}", f"two legs, {q(x)} each", f"{q(x)} and {q(y)} legs"])
        return [dims, rng.choice([f"{q(w)} wide", f"{q(w)} tall", f"{q(w)} high"]), thick()] + ([rng.choice(["90 degrees", "right angle", "two legs at 90 degrees"])] if rng.random() < 0.3 else [])
    if pt == "pipe_gasket":
        od, idd = rng.choice([3.5, 4.25, 5, 6, 7.5, 9, 11, 12, 16]), rng.choice([0.84, 1.315, 1.9, 2.375, 3.5, 4.5, 6.625, 8.625])
        dims = rng.choice([f"{q(od)} OD, {q(idd)} ID", f"outer diameter {q(od)}, inner diameter {q(idd)}", f"{q(idd)} inner and {q(od)} outer", f"{q(od)} outside diameter, {q(idd)} inside diameter", f"OD {q(od)} ID {q(idd)}"])
        n = rng.choice(["4", "8", "four", "six", "eight", "12", "16"])
        holes = rng.choice([f"{n} bolt holes", f"{n} bolts", f"{n} mounting holes", f"{n} holes", f"{n} bolt holes on a bolt circle"])
        return [dims, rng.choice([f"{q(T())} thick", f"thickness {q(T())}", f"{q(T())} thickness"]), holes] + ([rng.choice([f"for a {rng.choice(['2', '3', '4', '6', '8'])} inch flange", f"for a {rng.choice(['2', '3', '4', '6'])} inch pipe", "150 class", "ANSI 150"])] if rng.random() < 0.35 else [])
    od, idd, h = rng.choice([0.25, 0.375, 0.5, 0.75, 1, 1.5, 2]), rng.choice([0.125, 0.1875, 0.25, 0.375, 0.5, 0.75]), rng.choice([0.05, 0.0625, 0.125, 0.25, 0.5, 0.75, 1])
    n = rng.choice(["3", "4", "5", "6", "8", "10", "12"]); fr = rng.choice(["1/4", "5/16", "3/8", "1/2", "quarter inch", "half inch"])
    size = rng.choice([f"for an M{n} screw", f"M{n}", f"for a #{n} screw", f"for a {fr} bolt", f"{fr} screw size", f"for an M{n} bolt", f"metric {n}", f"for a {fr} screw"])
    dims = rng.choice([f"{q(od)} OD, {q(idd)} ID", f"{q(od)} wide, {q(h)} tall", f"ID {q(idd)} OD {q(od)}", f"{q(od)} diameter with a {q(idd)} hole", f"{q(h)} tall with a {q(idd)} hole", f"{q(od)} wide with a {q(idd)} hole"])
    return [size] + ([dims] if rng.random() < 0.8 else []) + ([rng.choice([f"{q(h)} thick", f"thickness {q(h)}", f"{q(h)} tall"])] if rng.random() < 0.6 else [])

_CLF_OPEN = ["make a", "I need a", "can you make a", "design a", "generate a", "give me a", "build a", "create a", "model a", "I want a", "make me a", "need a", "please make a", ""]
_CLF_JOIN = ["that is", "thats", "with", "which is", ":", "measuring", "of", "", ",", "that has"]
def render_clf(pt, rng, nouns):
    noun, parts = rng.choice(nouns[pt]), _clf_clauses(pt, rng)
    rng.shuffle(parts[1:]) if len(parts) > 2 else None
    opener, join = rng.choice(_CLF_OPEN), rng.choice(_CLF_JOIN)
    if rng.random() < 0.2:                                                   # dimensions first: "4 x 8 x 2 inch box, 1/8 thick"
        text = f"{parts[0]} {noun}" + "".join(", " + p for p in parts[1:])
    else:
        text = f"{opener} {noun}".strip() + (f" {join} " if join not in (":", ",") else join + " ") + rng.choice([", ", ", ", " and "]).join(parts)
    if rng.random() < 0.18: text = text.lower()
    if rng.random() < 0.10:
        words = [m for m in re.finditer(r"[A-Za-z]{6,}", text)]
        if words:
            m = rng.choice(words); i = rng.randrange(1, len(m.group()) - 2); w = list(m.group()); w[i], w[i + 1] = w[i + 1], w[i]
            text = text[:m.start()] + "".join(w) + text[m.end():]
    return re.sub(r"\s+", " ", text).strip(" ,")

def build_clf_phrases(per_type, seed, include_extra=True):
    rng, nouns, rows, seen = random.Random(seed), {pt: CLF_NOUNS_BASE[pt] + (CLF_NOUNS_EXTRA[pt] if include_extra else []) for pt in PART_TYPES}, [], set(data.text_norm)
    for pt in PART_TYPES:
        made, tries = 0, 0
        while made < per_type and tries < per_type * 30:
            tries += 1; t = render_clf(pt, rng, nouns); tn = normalize_text(t)
            if tn in seen: continue
            seen.add(tn); made += 1; rows.append({"text": t, "text_norm": tn, "part_type": pt, "source": "claude_clf_synthetic"})
    return pd.DataFrame(rows)

CLF_PHRASES_PER_TYPE = 100          # synthetic classifier-only examples per part type (5 x 100 = 500). 0 = off
clf_extra_df = build_clf_phrases(CLF_PHRASES_PER_TYPE, SEED + 3) if CLF_PHRASES_PER_TYPE > 0 else pd.DataFrame(columns=["text", "text_norm", "part_type", "source"])
print(f"classifier phrase data: {len(clf_extra_df)} synthetic rows (training only) | real classifier training rows: {len(train_df)}")
for _pt in PART_TYPES:
    if len(clf_extra_df): print(f"  [{_pt}] {clf_extra_df[clf_extra_df.part_type == _pt].text.iloc[0]}")

## 6. Stage 1 — part-type classifier, trained from scratch
A bag-of-words neural net (learned embeddings → small MLP) with **randomly initialised weights**: no pretrained model is involved.

In [ ]:
# ------------------------------------------------------------------ STAGE 1: part-type classifier, TRAINED FROM SCRATCH
# Pipeline box: "Trained from scratch model extracts part type".
# A small neural net with randomly-initialised weights - no pretrained model is used anywhere in this stage.
from collections import Counter

# Words that only describe HOW a request is phrased, not WHAT the part is. In the real data they correlate with who wrote each sheet
# ("it", "be", "thats" are in ~40% of gasket texts but ~0% of container texts), so a model that sees them learns the writer, not the part.
CLF_STOP = set("""i you me my we us it its it's to be is are was am that thats this these those want need would could should can will please make model design create
build generate give get render produce draw show let lets like also then just so which who what too very really a an the and""".split())

def clf_tokens(text_norm: str):
    t = re.sub(r"(?<![\w.])\d+(?:\.\d+)?", " <num> ", text_norm.lower())      # numbers don't help decide the part type
    words = [w for w in re.findall(r"<num>|[a-z]+", t) if w not in CLF_STOP]    # drop request-framing words
    return words + [a + "_" + b for a, b in zip(words, words[1:])]              # unigrams + bigrams

PT2ID = {pt: i for i, pt in enumerate(PART_TYPES)}
# classifier training set = real train rows + their augmented copies + the classifier-only phrase rows from section 5d. Validation/test stay real.
clf_train_df = pd.concat([train_df[["text_norm", "part_type"]], clf_extra_df[["text_norm", "part_type"]]], ignore_index=True)
_doc_freq = Counter(tok for t in clf_train_df.text_norm for tok in set(clf_tokens(t)))
VOCAB = {"<unk>": 0}
for _w, _c in sorted(_doc_freq.items()):
    if _c >= 2: VOCAB[_w] = len(VOCAB)                                          # only tokens seen in >=2 training texts

def clf_encode(text_norm: str):
    return [VOCAB[t] for t in clf_tokens(text_norm) if t in VOCAB] or [0]       # unknown words are ignored (they used to vote as <unk>)

# diagnostic only (never trained on): 60 hand-written, casually-worded prompts with unusual nouns, units and phrasing
STRESS_PROMPTS = {
    "rect_container": [
        'make a box thats 40inches by 12.73 inches and 1/2" thick. i want it to be two feet tall',
        'Box 4 x 8 in. 1/2" thick 1.75" tall',
        'I need an open top box 6 by 4 by 3 inches with quarter inch walls',
        'crate 24 x 18 x 12 inches, 3/4 inch thick walls',
        'parts tray, 200 mm long, 120 mm wide, 30 mm deep, 2 mm walls',
        'make me a storage bin that is 10 inches long 8 wide and 6 tall, wall thickness .125',
        'Rectangular enclosure for a circuit board: 3.5 x 2.25 x 1 inch, 0.08 inch wall',
        'drawer organizer 12 by 3 by 2, walls 1/8',
        'open box, length 5in width 5in height 2in thickness 0.1in',
        'case 100mm x 60mm x 25mm with 1.5mm walls',
        'a shallow tray 14 inches by 9 inches, 1 inch tall, 3/16 thick',
        'build a rectangular tub thats a foot long, 6 inches wide, 4 inches deep and a quarter inch thick',
    ],
    "cyl_container": [
        'make a cup with a 3 inch inside diameter, 4 inches tall and 1/8 inch walls',
        'tin can 2.5 in ID by 5 in tall, 0.06 wall',
        'I want a cylinder 60 mm across, 100 mm tall, 2 mm thick',
        'pencil cup: 3 inch diameter 4.5 inch high wall thickness 0.1',
        'cylindrical container 10 inch inner diameter 14 inch tall 0.18 thick',
        'round canister, inside diameter 1.5 inches, height 3 inches, 1/16 inch wall',
        'Make a jar, 4" wide inside, 6" tall, 1/8" thick',
        'bucket 8 inches inside diameter, a foot tall, 3/16 walls',
        'open top tube cup 1 inch ID 2 inches tall .05 thick',
        'pot with inner diameter 5 in and height 5 in and thickness 0.15 in',
        'a small round holder, 2.25" ID, 3" tall, 0.075" wall',
        'cylinder: 75 mm inner diameter, 120 mm high, wall 3 mm',
    ],
    "pipe_gasket": [
        'gasket for a 6 inch flange, 1/16 thick, 8 bolt holes',
        'make a pipe gasket 4.5 OD 2.375 ID 0.125 thick 4 holes',
        'ring gasket 150 class, 2 inch pipe, 1/8 inch, four bolts',
        'I need a flange gasket with outer diameter 9 inches and inner diameter 6.625 and 8 bolt holes, 1/16 thick',
        'full face gasket for 3 inch pipe, 5/32 thick',
        'seal for a pipe flange, 3.5 inch ID, 7.5 OD, 3/32 inch thick, 4 bolts',
        'gasket 12 inch OD, 8 inch ID, 0.0625 thick, 8 holes on a bolt circle',
        'make a flat ring gasket 1.9 inner 4.25 outer 1/16 thick with 4 mounting holes',
        'pipe flange gasket: OD 11 in ID 8.625 in thick 1/8 in 8 bolts',
        'gasket for 1 1/2 inch pipe, 3 7/8 OD, 1/16 thick, 4 bolt holes',
        'sealing gasket 100 mm ID 160 mm OD 3 mm thick 4 holes',
        'a gasket ring thats 5" OD, 2" ID, 1/16" thick, with six bolt holes',
    ],
    "washer": [
        'M6 washer',
        'washer for an M8 bolt, 16 mm OD, 8.4 mm ID, 1.6 mm thick',
        'spacer for a #8 screw, 1/2 inch wide, 1/4 inch tall',
        'make a standoff with a 0.25 hole, 0.5 OD, 0.75 tall',
        'flat washer 3/8 screw size',
        'shim washer ID 0.5 OD 1 thickness 0.03',
        'round spacer .75" wide .05" tall with a .375 diameter hole for the screw',
        'I need a washer for a quarter inch bolt, 3/4 OD, 1/16 thick',
        'bushing spacer, 5/16 ID, 1/2 OD, 3/8 long',
        'washer M10',
        'spacer 10 mm tall 6 mm hole 12 mm wide',
        'a washer for a 1/4 inch screw',
    ],
    "bracket": [
        'L bracket with 4 inch legs, 1 inch wide, 1/8 inch thick',
        'make a right angle bracket, legs 3 and 5 inches, 1.5 wide, 0.125 thick',
        'corner brace 2 x 2 inch legs 3/4 wide 1/16 thick',
        'angle bracket 100 mm legs 25 mm wide 3 mm thick',
        'I need a 90 degree bracket with 6 inch long legs, 2 inches tall, .25 thick',
        'shelf bracket: 8 inch leg and 6 inch leg, 1 inch wide, 3/16 thick material',
        'L-shaped connector, both legs 1.5 inches, 0.5 tall, 0.07 thick',
        'make an angle iron piece 12 inches on each side, 1.25 wide, 1/8 thick',
        'corner bracket 2.5" by 2.5", 3/4" high, 0.1" thick',
        'right angle mount with 1 inch legs 1/2 inch wide and 1/16 inch thick',
        'L bracket, one leg a foot long and the other 6 inches, 2 inches wide, 3/16 thick',
        'gusset angle 3 inches each side, 1 inch wide, 0.09 thick',
    ],
}

print(f"classifier vocabulary: {len(VOCAB)} tokens | training rows: {len(clf_train_df)} ({len(train_df)} real/augmented + {len(clf_extra_df)} phrase rows)")

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)

class BowClassifier(nn.Module):
    """token ids -> mean of learned embeddings -> small MLP -> part type"""
    def __init__(self, vocab_size, n_classes, dim=48, hidden=64, p=0.3):
        super().__init__()
        self.emb = nn.EmbeddingBag(vocab_size, dim, mode="mean")
        self.net = nn.Sequential(nn.Dropout(p), nn.Linear(dim, hidden), nn.ReLU(), nn.Dropout(p), nn.Linear(hidden, n_classes))
    def forward(self, ids, offsets):
        return self.net(self.emb(ids, offsets))

def make_batch(texts):
    seqs = [clf_encode(t) for t in texts]
    offsets = torch.tensor([0] + list(np.cumsum([len(s) for s in seqs])[:-1]), dtype=torch.long)
    flat = torch.tensor([i for s in seqs for i in s], dtype=torch.long)
    return flat.to(DEVICE), offsets.to(DEVICE)

@torch.no_grad()
def clf_predict_ids(texts):
    clf.eval(); flat, off = make_batch(texts)
    return F.softmax(clf(flat, off), dim=-1).cpu().numpy()

def clf_accuracy(df):
    pred = clf_predict_ids(df.text_norm.tolist()).argmax(1)
    return float((pred == df.part_type.map(PT2ID).values).mean())

clf = BowClassifier(len(VOCAB), len(PART_TYPES)).to(DEVICE)
opt = torch.optim.AdamW(clf.parameters(), lr=5e-3, weight_decay=1e-3)
y_train = torch.tensor(clf_train_df.part_type.map(PT2ID).values, dtype=torch.long, device=DEVICE)
best_acc, best_state = -1.0, None

for epoch in range(1, 81):
    clf.train()
    perm = np.random.permutation(len(clf_train_df))
    for i in range(0, len(perm), 32):
        idx = perm[i:i + 32]
        flat, off = make_batch(clf_train_df.text_norm.iloc[idx].tolist())
        loss = F.cross_entropy(clf(flat, off), y_train[torch.as_tensor(idx, device=DEVICE)])
        opt.zero_grad(); loss.backward(); opt.step()
    va = clf_accuracy(val_df)
    if va >= best_acc: best_acc, best_state = va, copy.deepcopy(clf.state_dict())
    if epoch % 20 == 0: print(f"epoch {epoch:3d}  last-batch loss {loss.item():.3f}  val acc {va:.3f}")
clf.load_state_dict(best_state)

print(f"\nbest val accuracy {best_acc:.3f}  |  TEST accuracy {clf_accuracy(test_df):.3f}")
_pred = clf_predict_ids(test_df.text_norm.tolist()).argmax(1)
print(classification_report(test_df.part_type.map(PT2ID), _pred, target_names=PART_TYPES, zero_division=0))
print(pd.DataFrame(confusion_matrix(test_df.part_type.map(PT2ID), _pred, labels=range(len(PART_TYPES))),
                   index=[f"true {p}" for p in PART_TYPES], columns=[f"pred {p}" for p in PART_TYPES]))

def predict_part_type(text_norm: str):
    probs = clf_predict_ids([text_norm])[0]
    return PART_TYPES[int(probs.argmax())], {pt: float(p) for pt, p in zip(PART_TYPES, probs)}

# ---- diagnostic: casually worded prompts the model has never seen (not used for training or model selection)
_stress = [(normalize_text(p), pt) for pt, ps in STRESS_PROMPTS.items() for p in ps]
_sp = clf_predict_ids([p for p, _ in _stress]).argmax(1)
_hits = np.array([PART_TYPES[i] == pt for i, (_, pt) in zip(_sp, _stress)])
print(f"\nstress prompts (60 casual, unusual wording): {_hits.mean():.1%} correct")
for _p, (_, _pt), _i, _ok in zip([p for ps in STRESS_PROMPTS.values() for p in ps], _stress, _sp, _hits):
    if not _ok: print(f"  wanted {_pt:14s} got {PART_TYPES[_i]:14s} | {_p[:90]}")

## 7. Stage 2 — parameter extractor, fine-tuned with LoRA
Fine-tunes a small open LLM to write the standard-schema JSON. Loss is computed only on the JSON answer. The un-tuned base model doubles as the **off-the-shelf baseline** (adapter switched off), so the report can show what fine-tuning bought you.  
`MODEL_NAME` defaults to Qwen2.5-1.5B-Instruct; use the 0.5B version for a faster run.

In [ ]:
# ------------------------------------------------------------------ STAGE 2 (prompts + parsing): fine-tuned parameter extractor
# Pipeline boxes: "Fine tuned model extracts parameters" -> "Outputs JSON with standard schema for all parts"
SCHEMA_DOC = "\n".join(f"{pt}: " + ", ".join(["part_type"] + fs) for pt, fs in SCHEMA.items())

def ft_prompt(part_type, text_norm):
    """short prompt for the FINE-TUNED model (it has learned the schema, so it doesn't need to be told)"""
    return f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n"

_clean_train = train_df[(train_df.audit_bad.map(len) == 0) & (train_df.source == "real")]                     # few-shot examples must not be mislabelled rows
_FEWSHOT = {pt: _clean_train[_clean_train.part_type == pt].iloc[0] for pt in PART_TYPES}
def baseline_prompt(part_type, text_norm):
    """long prompt for the OFF-THE-SHELF baseline (schema + one worked example per part type, no fine-tuning)"""
    shots = "".join(f"Part type: {pt}\nDescription: {r.text_norm}\nJSON:\n{r.target}\n\n" for pt, r in _FEWSHOT.items())
    return ("Convert a description of a mechanical part into compact JSON. Use exactly these keys, in this order:\n" + SCHEMA_DOC +
            "\nAll dimensions are decimal inches. hole_count is an integer. Output only the JSON.\n\n" + shots +
            f"Part type: {part_type}\nDescription: {text_norm}\nJSON:\n")

def parse_json_obj(s):
    """first {...} in the model's text -> dict, else None (flat JSON, so first closing brace ends it)"""
    m = re.search(r"\{.*?\}", s or "", re.S)
    if not m: return None
    try:
        obj = json.loads(m.group(0))
        return obj if isinstance(obj, dict) else None
    except json.JSONDecodeError:
        return None

# ---- scoring helpers (used by the evaluation cell)
def field_ok(f, pred, true):
    p = coerce_value(f, pred)
    if p is None: return False
    if f in STR_FIELDS: return p.lower() == str(true).lower()
    if f in INT_FIELDS: return p == true
    return abs(p - true) <= tol_for(true)

def score_outputs(df, outs):
    recs = []
    for r, o in zip(df.itertuples(), outs):
        obj = parse_json_obj(o)
        oks = {f: bool(obj) and field_ok(f, obj.get(f), r.params[f]) for f in SCHEMA[r.part_type]}
        recs.append({"part_type": r.part_type, "json_valid": obj is not None, **oks, "all_correct": all(oks.values())})
    return pd.DataFrame(recs)

def summarize(scores, name):
    fields = [c for c in scores.columns if c not in ("part_type", "json_valid", "all_correct")]
    row = {"model": name, "JSON valid": scores.json_valid.mean(), **{f: scores[f].astype(float).mean() for f in fields}, "ALL fields correct": scores.all_correct.mean()}
    return pd.Series(row)

_t = train_df.iloc[:3]
_ok = score_outputs(_t, [r.target + " trailing text" for r in _t.itertuples()])
assert _ok.all_correct.all() and _ok.json_valid.all(), "scoring helpers broken"
print("prompt/parse/score helpers ready | example fine-tune prompt:\n" + ft_prompt("washer", "washer for an M7 screw") + "{...}")

In [ ]:
# ------------------------------------------------------------------ prompt / parse / score helpers for TEXT -> HOLES (second task for the fine-tuned model)
def feat_prompt(pt, base, text_norm):
    """base part dimensions are shown because phrases like 'the long leg' or 'the centre of the wall' depend on them"""
    dims = ", ".join(f"{f.replace('_in', '')} {dec(base[f])}" for f in SCHEMA[pt])
    return f"Add holes to a {pt} ({dims}). Position keys: {', '.join(HOLE_SPEC[pt]['keys'])}.\nRequest: {text_norm}\nHOLES JSON:\n"

def parse_holes_obj(s):
    """first JSON object in the text -> list of hole dicts, else None (handles nested braces, unlike parse_json_obj)"""
    i = (s or "").find("{")
    if i < 0: return None
    try: obj, _ = json.JSONDecoder().raw_decode(s[i:])
    except json.JSONDecodeError: return None
    h = obj.get("holes") if isinstance(obj, dict) else None
    return h if isinstance(h, list) and all(isinstance(x, dict) for x in h) else None

def _resolved(pt, base, h):
    """what the user would actually see: blank (centred) positions replaced by the centred value"""
    k = HOLE_SPEC[pt]["keys"]; face = coerce_face(pt, h.get(k[0]))
    d = hole_defaults(pt, base, face) if face else {}
    out = {k[0]: face}
    for key in k[1:]:
        v = coerce_value(key, h.get(key)); out[key] = d.get(key) if v is None else v
    return out

def score_holes(df, outs):
    """outcome-based: a hole is right if face, both positions (after centring) and diameter match within tol; all holes must match, in any order"""
    recs = []
    for r, o in zip(df.itertuples(), outs):
        pt, k = r.part_type, HOLE_SPEC[r.part_type]["keys"]
        pred = parse_holes_obj(o); rec = {"part_type": pt, "json_valid": pred is not None}
        truth = [_resolved(pt, r.base, h) for h in r.holes]
        rec["count_ok"] = pred is not None and len(pred) == len(truth)
        ok_fields = {key: False for key in k}
        if rec["count_ok"]:
            keyf = lambda h: (str(h[k[0]]), -1 if h[k[1]] is None else h[k[1]], -1 if h[k[2]] is None else h[k[2]])
            P = sorted((_resolved(pt, r.base, h) for h in pred), key=keyf); T = sorted(truth, key=keyf)
            same = lambda a, b, key: (a[key] == b[key]) if key == k[0] else (a[key] is not None and b[key] is not None and abs(a[key] - b[key]) <= max(tol_for(b[key]), 0.01 if key == "angle_deg" else 0))
            ok_fields = {key: all(same(p, t, key) for p, t in zip(P, T)) for key in k}
        rec.update(ok_fields); rec["all_correct"] = rec["count_ok"] and all(ok_fields.values())
        recs.append(rec)
    return pd.DataFrame(recs)

# ---- self-tests with code-made examples (these are test fixtures, not training data)
_b = {"length_a_in": 4.0, "length_b_in": 2.0, "height_in": 1.0, "thickness_in": 0.125}
_truth = [{"face": "leg_a", "from_end_in": 1.0, "across_in": None, "diameter_in": 0.25}]
_df = pd.DataFrame([{"part_type": "bracket", "base": _b, "holes": _truth}] * 4)
_outs = ['{"holes":[{"face":"leg_a","from_end_in":1.0,"across_in":null,"diameter_in":0.25}]} trailing',       # exact
         '{"holes":[{"face":"leg_a","from_end_in":1.0,"across_in":0.5,"diameter_in":0.25}]}',                  # centred value written out -> same outcome
         '{"holes":[{"face":"leg_b","from_end_in":1.0,"across_in":null,"diameter_in":0.25}]}',                 # wrong leg
         'not json']
_sc = score_holes(_df, _outs)
assert _sc.all_correct.tolist() == [True, True, False, False] and _sc.json_valid.tolist() == [True, True, True, False], _sc
assert parse_holes_obj('{"holes":[{"a":1},{"a":2}]} x') == [{"a": 1}, {"a": 2}] and parse_holes_obj('{"holes": 3}') is None
print("hole prompt/parse/score helpers ready | example prompt:\n" + feat_prompt("bracket", _b, "a quarter inch hole 1 inch from the end of leg a") + "{...}")

In [ ]:
# Fine-tunes a small open LLM with LoRA so that:  "Part type + normalized description"  ->  standard-schema JSON.
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_CUDA = DEVICE == "cuda"
USE_BF16 = USE_CUDA and torch.cuda.is_bf16_supported()
LOAD_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32     # T4 has no bf16: use plain fp32 (a bit slower, but no fp16 NaN/overflow risk)
AMP_DTYPE, AMP_ON = torch.bfloat16, USE_BF16                     # mixed precision only on GPUs that support bf16 (L4/A100/H100)
if not USE_CUDA: print("WARNING: no GPU found. Runtime > Change runtime type > T4 GPU (CPU training will be very slow).")
torch.manual_seed(SEED)

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token
try:    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=LOAD_DTYPE)
except TypeError: base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=LOAD_DTYPE)   # older transformers
base.to(DEVICE)
base.config.use_cache = False                               # the key/value cache is only for generation; it wastes memory while training
if GRAD_CHECKPOINTING and USE_CUDA:
    base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
if USE_CUDA: torch.cuda.empty_cache()
model = get_peft_model(base, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

# ---- tokenize: loss is computed ONLY on the JSON answer, not on the prompt
MAX_LEN = 384                                               # hole prompts + JSON are longer than the part prompts
def encode_pair(prompt, target):
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    c = tok(target + tok.eos_token, add_special_tokens=False)["input_ids"]
    return (p + c)[:MAX_LEN], ([-100] * len(p) + c)[:MAX_LEN]
def encode_example(part_type, text_norm, target): return encode_pair(ft_prompt(part_type, text_norm), target)
def encode_df(df): return [encode_example(r.part_type, r.text_norm, r.target) for r in df.itertuples()]
train_enc, val_enc = encode_df(train_df), encode_df(val_df)
if FEAT_READY:                                              # second task: description of holes -> holes JSON, same adapter
    _enc_feat = lambda df: [encode_pair(feat_prompt(r.part_type, r.base, r.text_norm), r.target) for r in df.itertuples()]
    train_enc += _enc_feat(feat_train); val_enc += _enc_feat(feat_val)
    print(f"multi-task training: + {len(feat_train)} hole-feature examples")
print("longest training example:", max(len(i) for i, _ in train_enc), "tokens (MAX_LEN =", MAX_LEN, ")")

def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids = torch.full((len(batch), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(batch), L), -100, dtype=torch.long)
    att = torch.zeros((len(batch), L), dtype=torch.long)
    for k, (i, l) in enumerate(batch):
        ids[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); att[k, :len(i)] = 1
    return ids.to(DEVICE), att.to(DEVICE), lab.to(DEVICE)

def eval_loss(enc):
    model.eval(); total, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(enc), MICRO_BATCH):
            ids, att, lab = collate(enc[i:i + MICRO_BATCH])
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
                out = model(input_ids=ids, attention_mask=att, labels=lab)
            k = int((lab[:, 1:] != -100).sum()); total += out.loss.item() * k; n += k
    return total / n

# ---- training loop (plain PyTorch, so it doesn't depend on Trainer/TRL API changes)
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LORA_LR, weight_decay=0.0)
steps_per_epoch = math.ceil(len(train_enc) / BATCH_SIZE)
total_steps = LORA_EPOCHS * steps_per_epoch; warmup = max(1, int(0.05 * total_steps))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * max(0.0, 1 - s / total_steps))

best_val, best_state, history = float("inf"), None, []
for epoch in range(1, LORA_EPOCHS + 1):
    model.train(); order = np.random.permutation(len(train_enc)); running = 0.0
    for i in range(0, len(order), BATCH_SIZE):
        batch = [train_enc[j] for j in order[i:i + BATCH_SIZE]]
        n_tok = sum(1 for _, l in batch for x in l[1:] if x != -100)             # answer tokens in the whole batch
        opt.zero_grad(set_to_none=True); batch_loss = 0.0
        for m in range(0, len(batch), MICRO_BATCH):                              # gradient accumulation: same result as one big batch, far less memory
            ids, att, lab = collate(batch[m:m + MICRO_BATCH])
            k_tok = int((lab[:, 1:] != -100).sum())
            with torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
                out = model(input_ids=ids, attention_mask=att, labels=lab)
            loss = out.loss * (k_tok / max(n_tok, 1))                            # weight each piece by its share of the batch's answer tokens
            loss.backward(); batch_loss += loss.item()
            del out, loss
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); sched.step(); running += batch_loss
    val_loss = eval_loss(val_enc); history.append((epoch, running / steps_per_epoch, val_loss))
    print(f"epoch {epoch}/{LORA_EPOCHS}  train loss {running / steps_per_epoch:.4f}  val loss {val_loss:.4f}" + (f"  (peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB)" if USE_CUDA else ""))
    if val_loss < best_val:                                   # keep the best epoch (tiny datasets overfit quickly)
        best_val = val_loss
        best_state = {k: v.detach().cpu().clone() for k, v in model.named_parameters() if v.requires_grad}
model.load_state_dict(best_state, strict=False)
if USE_CUDA: torch.cuda.empty_cache()
model.eval()
print(f"restored best epoch (val loss {best_val:.4f})")

@torch.no_grad()
def generate(prompts, use_adapter=True, max_new_tokens=96, batch=8):
    """use_adapter=False runs the ORIGINAL, un-fine-tuned model (that's our off-the-shelf baseline / verifier)"""
    model.eval(); outs = []; tok.padding_side = "left"
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i + batch], return_tensors="pt", padding=True, add_special_tokens=False).to(DEVICE)
        ctx = nullcontext() if use_adapter else model.disable_adapter()
        with ctx, torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=AMP_ON):
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.pad_token_id, use_cache=True)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"
    return outs

def extract_params(part_type, text_norm):
    """Pipeline box 'Fine tuned model extracts parameters': returns a dict, or None if the output wasn't valid JSON."""
    return parse_json_obj(generate([ft_prompt(part_type, text_norm)])[0])

def _extract_holes(part_type, base, text_norm):
    """Add-features box: description of holes -> list of hole dicts (or None if the output wasn't valid JSON)"""
    return parse_holes_obj(generate([feat_prompt(part_type, base, text_norm)], max_new_tokens=240)[0])
extract_holes = _extract_holes if FEAT_READY else None      # None = the app disables 'extract holes from text' (the hole table still works)

_demo = test_df.iloc[0]
print("\nexample ->", _demo.text_norm, "\n  model :", extract_params(_demo.part_type, _demo.text_norm), "\n  label :", _demo.target)

## 8. Evaluation on the held-out test set
Scored on the **real** test rows only (synthetic data is never used for testing). Fine-tuned vs off-the-shelf few-shot, per-field accuracy, and the full classifier → extractor chain. Mistakes on rows the audit flagged are probably label errors, not model errors.

In [ ]:
# ------------------------------------------------------------------ EVALUATION on the held-out TEST split
# "Correct" = within tol_for(value) of the label (labels are catalog-snapped, texts are often rounded), hole_count/screw_size exact.
def run_eval(df, prompt_fn, use_adapter):
    outs = generate([prompt_fn(r.part_type, r.text_norm) for r in df.itertuples()], use_adapter=use_adapter)
    return score_outputs(df, outs), outs

sc_ft,   outs_ft   = run_eval(test_df, ft_prompt, True)
sc_base, outs_base = run_eval(test_df, baseline_prompt, False)
comparison = pd.DataFrame([summarize(sc_ft, "fine-tuned (LoRA)"), summarize(sc_base, "off-the-shelf, few-shot")]).set_index("model")
print("Extractor accuracy on the test set (true part type given, so this isolates the extractor):")
show_df(comparison.round(3))
print("\nfine-tuned, ALL fields correct by part type:")
show_df(sc_ft.groupby("part_type").all_correct.agg(["mean", "size"]).round(3))

# ---- end-to-end: classifier -> extractor (what a real user experiences)
pred_types = [predict_part_type(t)[0] for t in test_df.text_norm]
type_ok = np.array(pred_types) == test_df.part_type.values
e2e = score_outputs(test_df, generate([ft_prompt(pt, t) for pt, t in zip(pred_types, test_df.text_norm)]))
print(f"\nend-to-end on test: part type right {type_ok.mean():.1%} | every field right {(e2e.all_correct.values & type_ok).mean():.1%}")

# ---- look at the mistakes (rows the audit flagged probably have wrong LABELS, not wrong predictions)
bad = np.where(~sc_ft.all_correct.values)[0]
for i in bad[:8]:
    r = test_df.iloc[i]
    print(f"\n[{'label likely wrong' if r.audit_bad else 'model error'}] {r.text}\n   expected: {r.target}\n   got     : {parse_json_obj(outs_ft[i])}")

# ---- feature task (text -> holes), only when you supplied hole-feature sentences
if FEAT_READY and len(feat_test):
    outs_f = generate([feat_prompt(r.part_type, r.base, r.text_norm) for r in feat_test.itertuples()], max_new_tokens=240)
    sc_f = score_holes(feat_test, outs_f)
    print(f"\nHole extraction on {len(feat_test)} held-out feature rows (outcome-based; blank = centred counts as the centred value):")
    show_df(sc_f.drop(columns="part_type").mean().round(3).to_frame("accuracy").T)
    show_df(sc_f.groupby("part_type").all_correct.agg(["mean", "size"]).round(3))
    for i in np.where(~sc_f.all_correct.values)[0][:5]:
        r = feat_test.iloc[i]; print(f"\n[hole miss] {r.text}\n   expected: {r.target}\n   got     : {parse_holes_obj(outs_f[i])}")
else:
    print("\n(hole-feature evaluation skipped: no feature sentences loaded)")

## 9. Pipeline glue
Text → part type → parameters → table, and the confirm step (apply edits → re-validate → CAD → log feedback).

In [ ]:
# ------------------------------------------------------------------ PIPELINE GLUE: text -> type -> parameters -> validated table
FEEDBACK_LOG = WORKDIR / "feedback.jsonl"

def analyze(text: str, forced_type: str = None) -> dict:
    """Boxes: preprocess -> classifier (from scratch) -> fine-tuned extractor -> JSON -> validation -> table."""
    text_norm = normalize_text(text)
    pt, probs = (forced_type, None) if forced_type else predict_part_type(text_norm)
    raw = extract_params(pt, text_norm)                       # dict, or None if the model's output wasn't valid JSON
    try:
        rows = make_table(pt, text_norm, raw)
    except Exception:                                         # defensive: unreadable model output -> empty table the user fills in
        raw = None; rows = make_table(pt, text_norm, None)
    if USE_LLM_VERIFIER:
        suspects = llm_verify(pt, text_norm, rows_to_params(pt, rows))
        for r in rows:
            if r["field"] in suspects and r["status"] in ("extracted", "snapped"):
                r.update(status="llm_flag", note="the off-the-shelf verifier thinks this may be wrong")
    return {"text": text, "text_norm": text_norm, "part_type": pt, "probs": probs, "raw": raw, "rows": rows}

def confirm_and_build(state: dict, ui_values: dict):
    """User pressed Confirm: apply their edits -> re-validate -> (if OK) JSON goes to the CAD script -> STL."""
    pt = state["part_type"]
    rows, edited = apply_edits(pt, state["rows"], ui_values)
    params = rows_to_params(pt, rows)
    issues = check_rules(pt, params)
    result = {"rows": rows, "params": params, "issues": issues, "stl_path": None, "info": None}
    if any(i["level"] == "error" for i in issues):
        return result
    path = WORKDIR / f"{pt}_{time.time_ns() // 1_000_000}.stl"
    result["info"] = build_stl(params, path)
    result["stl_path"] = str(path)
    with open(FEEDBACK_LOG, "a") as fh:                       # feedback loop: (text, model output, what the user corrected)
        fh.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "text": state["text"], "text_norm": state["text_norm"],
                             "predicted_type": pt, "model_output": state["raw"], "final_params": params,
                             "user_edited_fields": sorted(edited)}) + "\n")
    return result

# ---------------------------------------------------------------- ADD FEATURES (holes), after the base part exists
FEEDBACK_FEATURES = WORKDIR / "feedback_features.jsonl"

def analyze_features(state: dict, text: str) -> dict:
    """description of holes -> fine-tuned model -> hole rows (blank positions centred, ungrounded numbers flagged)"""
    pt, p = state["part_type"], state["params"]
    text_norm = normalize_text(text)
    raw = extract_holes(pt, {f: p[f] for f in SCHEMA[pt]}, text_norm)          # list of dicts, or None
    return {"text": text, "text_norm": text_norm, "raw": raw, "rows": make_hole_rows(pt, p, text_norm, raw)}

def build_with_features(state: dict, ui_rows: list) -> dict:
    """User pressed Apply: resolve the table (blank = centred) -> validate -> CAD (base part + holes) -> STL"""
    pt, p = state["part_type"], state["params"]
    rows, holes = resolve_hole_rows(pt, p, ui_rows, state.get("hole_rows"))
    issues = check_holes(pt, p, holes)
    result = {"rows": rows, "holes": holes, "issues": issues, "stl_path": None, "info": None}
    if any(i["level"] == "error" for i in issues): return result
    path = WORKDIR / f"{pt}_{time.time_ns() // 1_000_000}.stl"
    result["info"] = build_stl(p, path, holes or None)
    result["stl_path"] = str(path)
    if holes:
        with open(FEEDBACK_FEATURES, "a") as fh:                                 # (text, model output, final holes) for future retraining
            fh.write(json.dumps({"time": time.strftime("%Y-%m-%d %H:%M:%S"), "part_type": pt, "base": {f: p[f] for f in SCHEMA[pt]},
                                 "text": state.get("feat_text"), "model_output": state.get("feat_raw"), "final_holes": holes}) + "\n")
    return result

## 10. The app (plain HTML)
A single HTML/CSS/JS page (`INDEX_HTML`) served by a small Flask backend. Extract → review the color-coded table → edit any cell in the **value** column → Confirm → 3D preview (three.js), download, or start over.  
Changing the *part type* dropdown re-runs the fine-tuned extractor for that type. The page and the four `/api/...` endpoints are ordinary code you can move to any server, including a Hugging Face Docker Space.

In [ ]:
# ------------------------------------------------------------------ GUI: one plain HTML page + a tiny Flask backend (no Gradio)
# The page (INDEX_HTML) is ordinary HTML/CSS/JS. It talks to JSON endpoints (/api/analyze, /retype, /confirm, /features/extract, /features/apply)
# that call the pipeline functions above.
# ---- the app needs the two trained models from sections 6 and 7; fail early with a clear message if they aren't ready
_not_ready = [n for n in ("predict_part_type", "extract_params") if n not in globals()]
if _not_ready:
    raise RuntimeError(f"Not ready: {_not_ready} not defined. Run the classifier cell (section 6) and the LoRA fine-tuning cell "
                       "(section 7) successfully first - the app uses the models they create.")

import uuid, threading
from flask import Flask, jsonify, request, send_file, abort
from werkzeug.serving import make_server

INDEX_HTML = r"""<!doctype html>
<html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Text to STL</title>
<style>
  body{font-family:system-ui,sans-serif;margin:0;background:#f4f5f7;color:#1c1e21}
  main{max-width:960px;margin:0 auto;padding:20px}
  h1{margin:0 0 4px} .sub{color:#555;margin:0 0 16px}
  .card{background:#fff;border-radius:10px;padding:16px;margin-bottom:14px;box-shadow:0 1px 3px rgba(0,0,0,.12)}
  textarea{width:100%;box-sizing:border-box;font:inherit;padding:8px;border:1px solid #bbb;border-radius:6px}
  button,select{font:inherit;padding:8px 14px;border-radius:6px;border:1px solid #888;background:#fff;cursor:pointer}
  button.primary{background:#1a5fd0;color:#fff;border-color:#1a5fd0} button:disabled{opacity:.45;cursor:default}
  .row{display:flex;gap:10px;align-items:center;flex-wrap:wrap;margin-top:10px}
  table{border-collapse:collapse;width:100%} th,td{text-align:left;padding:6px 8px;border-bottom:1px solid #e3e3e3}
  td input{width:130px;font:inherit;padding:4px 6px;border:1px solid #aaa;border-radius:4px}
  tr.extracted td.st{background:#dff3e0} tr.missing td.st{background:#f9d6d5} tr.not_in_text td.st,tr.llm_flag td.st{background:#fdebc4}
  tr.snapped td.st,tr.lookup td.st{background:#d8e7fb} tr.default td.st{background:#fff6b8} tr.edited td.st{background:#e7dcf7}
  .legend span{display:inline-block;padding:2px 8px;margin:2px 4px 2px 0;border-radius:4px;font-size:13px}
  #msg{white-space:pre-wrap} .err{color:#b3261e} .ok{color:#1b6e2d}
  #view{width:100%;height:340px;background:#eceff3;border-radius:8px;display:flex;align-items:center;justify-content:center;color:#666}
  .c-extracted{background:#dff3e0} .c-missing{background:#f9d6d5} .c-not_in_text,.c-llm_flag{background:#fdebc4} .c-default{background:#fff6b8} .c-edited{background:#e7dcf7}
  #ftable input{width:92px;font:inherit;padding:4px 6px;border:1px solid #aaa;border-radius:4px} #ftable select{padding:4px 6px} .fnote{font-size:13px;color:#555}
  #fmsg{white-space:pre-wrap} .axes{font-size:12px;color:#666;margin-top:6px} .hint{font-size:13px;color:#555;margin:4px 0 8px}
  pre{background:#f0f1f3;padding:8px;border-radius:6px;overflow:auto;font-size:13px;margin:8px 0 0}
</style></head><body><main>
<h1>Text &rarr; STL</h1>
<p class="sub">Describe a <b>pipe gasket, washer/spacer, L-bracket, rectangular container, or cylindrical container</b>. Fractions and decimals both work.</p>

<div class="card">
  <textarea id="text" rows="3" placeholder="e.g. gasket for a 6 inch flange on a 2 3/8 pipe, 1/16 thick with four bolt holes"></textarea>
  <div class="row"><button class="primary" id="bExtract">Extract parameters</button><button id="bReset">Start over</button></div>
</div>

<div class="card" id="tableCard" hidden>
  <div class="row" style="margin-top:0"><label>Part type <select id="ptype"></select></label>
    <span style="color:#666;font-size:13px">changing it re-runs the extractor for that type</span></div>
  <p id="info"></p>
  <table><thead><tr><th>field</th><th>value (editable)</th><th>status</th><th>note</th></tr></thead><tbody id="rows"></tbody></table>
  <div class="legend" style="margin-top:10px">
    <span style="background:#dff3e0">found in your text</span><span style="background:#fdebc4">not in your text</span>
    <span style="background:#f9d6d5">missing</span><span style="background:#d8e7fb">catalog size</span>
    <span style="background:#fff6b8">assumed default</span><span style="background:#e7dcf7">edited by you</span></div>
  <div class="row"><button class="primary" id="bConfirm">Confirm &amp; generate STL</button></div>
  <div id="msg"></div>
</div>

<div class="card" id="outCard" hidden>
  <div id="view">3D preview</div>
  <div class="axes">Preview: the front faces you &middot; red = X (to the right) &middot; green = Y (away from you) &middot; blue = Z (up). Drag to rotate.</div>
  <div class="row"><a id="dl" href="#"><button>Download STL</button></a></div>
  <details><summary>JSON sent to the CAD script</summary><pre id="json"></pre></details>
</div>

<div class="card" id="featCard" hidden>
  <h2 style="margin:0 0 4px">Add features</h2>
  <p class="hint" style="margin-top:0">Add holes to the part you just built. Dimensions are in inches. Every change rebuilds the part above (preview, download and JSON update).</p>
  <p class="hint" id="fhint"></p>
  <textarea id="ftext" rows="2" placeholder="Describe the holes in words (needs the trained feature model), or add them in the table below"></textarea>
  <div class="row"><button id="bFExtract">Extract holes from text</button><span id="fnote" class="fnote"></span></div>
  <table id="ftable"><thead><tr id="fhead"></tr></thead><tbody></tbody></table>
  <div class="row"><button id="bAddHole">Add hole</button><button class="primary" id="bApply">Apply holes &amp; rebuild</button><button id="bClearHoles">Remove all holes</button></div>
  <div class="legend" style="margin-top:8px">
    <span style="background:#dff3e0">found in your text</span><span style="background:#fdebc4">not in your text</span>
    <span style="background:#f9d6d5">missing</span><span style="background:#fff6b8">centred / default</span><span style="background:#e7dcf7">entered by you</span>
    &nbsp; Leave a position blank to centre the hole.</div>
  <div id="fmsg"></div>
</div>
</main>

<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}</script>
<script type="module">
import * as THREE from 'three';
import {STLLoader} from 'three/addons/loaders/STLLoader.js';
import {OrbitControls} from 'three/addons/controls/OrbitControls.js';

const $ = id => document.getElementById(id);
let sid = null, renderer = null, raf = 0;
const TYPES = __PART_TYPES__;
TYPES.forEach(t => { const o = document.createElement('option'); o.value = o.textContent = t; $('ptype').appendChild(o); });

async function api(path, body){
  const r = await fetch(path, {method:'POST', headers:{'Content-Type':'application/json'}, body:JSON.stringify(body)});
  const j = await r.json(); if(!r.ok) throw new Error(j.error || r.statusText); return j;
}
function say(text, cls){ const m = $('msg'); m.textContent = text; m.className = cls || ''; }
function showRows(rows){
  const tb = $('rows'); tb.textContent = '';
  rows.forEach(r => {
    const tr = document.createElement('tr'); tr.className = r.status;
    const f = document.createElement('td'); f.textContent = r.field;
    const v = document.createElement('td'); const i = document.createElement('input'); i.value = r.value; i.dataset.field = r.field; v.appendChild(i);
    const s = document.createElement('td'); s.className = 'st'; s.textContent = r.label;
    const n = document.createElement('td'); n.textContent = r.note;
    tr.append(f, v, s, n); tb.appendChild(tr);
  });
}
let currentType = null, reqId = 0;                       // reqId: ignore a slow response if the user has already moved on
function clearAnalysis(){ sid = null; currentType = null; $('rows').textContent = ''; $('info').textContent = ''; $('outCard').hidden = true; setupFeatures(null); }
function showAnalysis(j){
  sid = j.sid; currentType = j.part_type; $('ptype').value = j.part_type; $('tableCard').hidden = false; $('outCard').hidden = true; $('featCard').hidden = true;
  const conf = j.confidence != null ? ` (${Math.round(j.confidence*100)}% confident)` : '';
  const need = j.rows.filter(r => ['missing','not_in_text','llm_flag'].includes(r.status)).length;
  const alts = (j.alts || []).slice(1).filter(a => a[1] >= 0.1).map(a => `${a[0]} ${Math.round(a[1]*100)}%`);
  const unsure = j.confidence != null && j.confidence < 0.85 && alts.length;
  $('info').textContent = `Detected: ${j.part_type}${conf}. ` + (unsure ? `Not certain - also possible: ${alts.join(', ')}. If the type is wrong, change it in the box above. ` : '') +
    `Read your text as: "${j.text_norm}". ` + (need ? `${need} value(s) need your attention.` : 'Everything was found - review, then confirm.');
  showRows(j.rows); say('');
}
function busy(b){ ['bExtract','bConfirm'].forEach(id => $(id).disabled = b); }

$('bExtract').onclick = async () => {
  const my = ++reqId; busy(true);
  try { const j = await api('/api/analyze', {text: $('text').value}); if(my === reqId) showAnalysis(j); }
  catch(e){ if(my === reqId){ clearAnalysis(); $('tableCard').hidden = false; say(e.message, 'err'); } }
  finally { if(my === reqId) busy(false); }
};
$('ptype').onchange = async () => {
  if(!sid) return; const my = ++reqId; busy(true);
  try { const j = await api('/api/retype', {sid, part_type: $('ptype').value}); if(my === reqId) showAnalysis(j); }
  catch(e){ if(my === reqId){ $('ptype').value = currentType; say(e.message, 'err'); } }          // the old analysis is still on screen, so show its type again
  finally { if(my === reqId) busy(false); }
};
$('bConfirm').onclick = async () => {
  if(!sid) return; busy(true);
  const values = {}; document.querySelectorAll('#rows input').forEach(i => values[i.dataset.field] = i.value);
  try {
    const j = await api('/api/confirm', {sid, values});
    showRows(j.rows);
    const lines = j.issues.map(i => `- ${i.field}: ${i.msg}`).join('\n');
    if(!j.stl_url){ say("Can't build yet:\n" + lines, 'err'); $('outCard').hidden = true; setupFeatures(null); }
    else {
      const s = j.info.size_mm.join(' x ');
      say(`Part generated: ${s} mm, volume ${j.info.volume_mm3} mm3, ${j.info.watertight ? 'watertight' : 'NOT watertight'}` + (lines ? '\nHeads-up:\n' + lines : ''), 'ok');
      $('outCard').hidden = false; $('json').textContent = JSON.stringify(j.params, null, 2);
      $('dl').href = j.stl_url + '?dl=1'; preview(j.stl_url);
      setupFeatures(j.hole_spec);
    }
  } catch(e){ say(e.message, 'err'); }
  finally { busy(false); }
};
$('bReset').onclick = () => {
  ++reqId; clearAnalysis(); $('text').value = ''; $('tableCard').hidden = true; say(''); busy(false);
  if(renderer){ cancelAnimationFrame(raf); renderer.dispose(); renderer = null; }
  $('view').textContent = '3D preview';
};

let spec = null;
const FKEYS = ['face','p1','p2','d'];
function fsay(t, cls){ const m = $('fmsg'); m.textContent = t; m.className = cls || ''; }
function fbusy(b){ ['bApply','bAddHole','bClearHoles'].forEach(id => $(id).disabled = b); $('bFExtract').disabled = b || !(spec && spec.text_ready); }
function setupFeatures(sp){
  spec = sp; const card = $('featCard');
  if(!sp){ card.hidden = true; return; }
  card.hidden = false; $('ftable').querySelector('tbody').textContent = ''; fsay(''); $('ftext').value = '';
  const th = $('fhead'); th.textContent = '';
  sp.labels.concat(['note', '']).forEach(l => { const e = document.createElement('th'); e.textContent = l; th.appendChild(e); });
  $('fhint').textContent = sp.hint; fbusy(false);
  $('fnote').textContent = sp.text_ready ? '' : "Text extraction for features isn't trained yet - add holes with the table below.";
}
function addHoleRow(cells){
  const tb = $('ftable').querySelector('tbody'), tr = document.createElement('tr');
  FKEYS.forEach(k => {
    const td = document.createElement('td'); let el;
    if(k === 'face'){
      el = document.createElement('select');
      const blank = document.createElement('option'); blank.value = ''; blank.textContent = '(choose)'; el.appendChild(blank);
      spec.faces.forEach(f => { const o = document.createElement('option'); o.value = o.textContent = f; el.appendChild(o); });
    } else { el = document.createElement('input'); el.size = 8; }
    const c = cells && cells[k]; el.dataset.k = k;
    el.value = (c && c.value) ? c.value : ((k === 'face' && spec.faces.length === 1) ? spec.faces[0] : '');
    if(c) el.className = 'c-' + c.status;
    td.appendChild(el); tr.appendChild(td);
  });
  const note = document.createElement('td'); note.className = 'fnote';
  note.textContent = cells ? [...new Set(FKEYS.map(k => cells[k]).filter(c => c && c.note).map(c => c.note))].join('; ') : '';
  const rm = document.createElement('td'); const b = document.createElement('button'); b.textContent = 'Remove'; b.onclick = () => tr.remove(); rm.appendChild(b);
  tr.append(note, rm); tb.appendChild(tr);
}
function showHoleRows(rows){ $('ftable').querySelector('tbody').textContent = ''; rows.forEach(r => addHoleRow(r.cells)); }
function readHoleRows(){
  return [...$('ftable').querySelectorAll('tbody tr')].map(tr => { const o = {}; tr.querySelectorAll('[data-k]').forEach(el => o[el.dataset.k] = el.value); return o; });
}
async function applyHoles(rows){
  fbusy(true);
  try {
    const j = await api('/api/features/apply', {sid, rows});
    showHoleRows(j.rows);
    const lines = j.issues.map(i => '- ' + (i.hole != null ? `hole ${i.hole + 1}: ` : '') + i.msg).join('\n');
    if(!j.stl_url){ fsay("Can't build yet:\n" + lines, 'err'); }
    else {
      fsay(`${rows.length} hole(s) applied: ${j.info.size_mm.join(' x ')} mm, volume ${j.info.volume_mm3} mm3, ${j.info.watertight ? 'watertight' : 'NOT watertight'}` + (lines ? '\nHeads-up:\n' + lines : ''), 'ok');
      $('json').textContent = JSON.stringify(j.json, null, 2); $('dl').href = j.stl_url + '?dl=1'; preview(j.stl_url);
    }
  } catch(e){ fsay(e.message, 'err'); }
  fbusy(false);
}
$('bAddHole').onclick = () => addHoleRow(null);
$('bApply').onclick = () => applyHoles(readHoleRows());
$('bClearHoles').onclick = () => { showHoleRows([]); applyHoles([]); };
$('bFExtract').onclick = async () => {
  if(!sid) return; fbusy(true);
  try {
    const j = await api('/api/features/extract', {sid, text: $('ftext').value}); showHoleRows(j.rows);
    fsay(j.rows.length ? `Read your description as: "${j.text_norm}". Check the table, then press Apply.` : 'No holes were found in that description.', j.rows.length ? '' : 'err');
  } catch(e){ fsay(e.message, 'err'); }
  fbusy(false);
};

function preview(url){
  const box = $('view'); box.textContent = 'loading preview...';
  if(renderer){ cancelAnimationFrame(raf); renderer.dispose(); renderer = null; }
  new STLLoader().load(url, geo => {
    box.textContent = '';
    const w = box.clientWidth, h = box.clientHeight;
    renderer = new THREE.WebGLRenderer({antialias:true}); renderer.setSize(w, h); box.appendChild(renderer.domElement);
    const scene = new THREE.Scene(); scene.background = new THREE.Color(0xeceff3);
    geo.computeVertexNormals(); geo.center(); geo.computeBoundingSphere();
    const R = geo.boundingSphere.radius;
    scene.add(new THREE.Mesh(geo, new THREE.MeshStandardMaterial({color:0x4a90d9, metalness:.1, roughness:.6})));
    scene.add(new THREE.AmbientLight(0xffffff, .7)); const d = new THREE.DirectionalLight(0xffffff, .9); d.position.set(1,2,3); scene.add(d);
    const cam = new THREE.PerspectiveCamera(40, w/h, R/100, R*100); cam.up.set(0, 0, 1); cam.position.set(R*1.1, -R*2.6, R*1.4); scene.add(new THREE.AxesHelper(R*1.1));
    const ctl = new OrbitControls(cam, renderer.domElement);
    (function loop(){ raf = requestAnimationFrame(loop); ctl.update(); renderer.render(scene, cam); })();
  }, undefined, () => { box.textContent = '3D preview could not load (needs internet for three.js). The STL download still works.'; });
}
</script></body></html>"""

app = Flask("text_to_stl")
SESSIONS = {}

def _rows_json(rows):
    return [{"field": r["field"], "value": fmt_value(r["value"]), "status": r["status"],
             "label": STATUS_LABEL[r["status"]], "note": r["note"]} for r in rows]

def _analysis_json(sid, s):
    conf = max(s["probs"].values()) if s["probs"] else None
    alts = sorted(s["probs"].items(), key=lambda kv: -kv[1])[:3] if s["probs"] else []
    return {"sid": sid, "part_type": s["part_type"], "confidence": conf, "alts": alts, "text_norm": s["text_norm"], "rows": _rows_json(s["rows"])}

@app.get("/")
def index(): return INDEX_HTML.replace("__PART_TYPES__", json.dumps(PART_TYPES))

def _hole_spec(pt):
    if pt not in HOLE_SPEC: return None
    sp = HOLE_SPEC[pt]
    return {"faces": sp["faces"], "labels": sp["labels"], "hint": sp["hint"], "text_ready": callable(globals().get("extract_holes"))}

def _hole_rows_json(rows):
    return [{"cells": {k: {"value": fmt_value(c["value"]), "status": c["status"], "label": STATUS_LABEL[c["status"]], "note": c["note"]}
                       for k, c in r["cells"].items()}} for r in rows]

@app.post("/api/analyze")
def api_analyze():
    text = (request.get_json(silent=True) or {}).get("text", "").strip()
    if not text: return jsonify(error="Type a description of the part first."), 400
    try:
        s = analyze(text); sid = uuid.uuid4().hex; SESSIONS[sid] = s
        return jsonify(_analysis_json(sid, s))
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/retype")
def api_retype():
    b = request.get_json(silent=True) or {}
    old = SESSIONS.get(b.get("sid"))
    if old is None or b.get("part_type") not in PART_TYPES: return jsonify(error="Extract parameters first."), 400
    try:
        s = analyze(old["text"], forced_type=b["part_type"]); SESSIONS[b["sid"]] = s
        return jsonify(_analysis_json(b["sid"], s))
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/confirm")
def api_confirm():
    b = request.get_json(silent=True) or {}
    s = SESSIONS.get(b.get("sid"))
    if s is None: return jsonify(error="Extract parameters first."), 400
    try:
        res = confirm_and_build(s, {str(k): str(v) for k, v in (b.get("values") or {}).items()})
        s["rows"] = res["rows"]
        if res["stl_path"]: s["params"], s["hole_rows"] = res["params"], None          # the confirmed base part, ready for features
        else: s.pop("params", None)
        return jsonify({"rows": _rows_json(res["rows"]), "params": res["params"], "issues": res["issues"], "info": res["info"],
                        "stl_url": ("/stl/" + Path(res["stl_path"]).name) if res["stl_path"] else None,
                        "hole_spec": _hole_spec(s["part_type"]) if res["stl_path"] else None})
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/features/extract")
def api_features_extract():
    b = request.get_json(silent=True) or {}
    s = SESSIONS.get(b.get("sid"))
    if s is None or "params" not in s: return jsonify(error="Build the part first."), 400
    if s["part_type"] not in HOLE_SPEC: return jsonify(error="Features can only be added to brackets and containers."), 400
    if not callable(globals().get("extract_holes")):
        return jsonify(error="Text extraction for features isn't trained yet - add the holes in the table instead."), 400
    text = str(b.get("text", "")).strip()
    if not text: return jsonify(error="Describe the holes first."), 400
    try:
        res = analyze_features(s, text); s["hole_rows"], s["feat_text"], s["feat_raw"] = res["rows"], text, res["raw"]
        return jsonify({"rows": _hole_rows_json(res["rows"]), "text_norm": res["text_norm"]})
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.post("/api/features/apply")
def api_features_apply():
    b = request.get_json(silent=True) or {}
    s = SESSIONS.get(b.get("sid"))
    if s is None or "params" not in s: return jsonify(error="Build the part first."), 400
    if s["part_type"] not in HOLE_SPEC: return jsonify(error="Features can only be added to brackets and containers."), 400
    rows = [r for r in (b.get("rows") or []) if isinstance(r, dict)]
    try:
        res = build_with_features(s, rows); s["hole_rows"] = res["rows"]
        pt, p = s["part_type"], s["params"]
        return jsonify({"rows": _hole_rows_json(res["rows"]), "issues": res["issues"], "info": res["info"],
                        "stl_url": ("/stl/" + Path(res["stl_path"]).name) if res["stl_path"] else None,
                        "json": {"part": {"part_type": pt, **{f: p[f] for f in SCHEMA[pt]}}, "holes": res["holes"]}})
    except Exception as e: return jsonify(error=f"{type(e).__name__}: {e}"), 500

@app.get("/stl/<name>")
def get_stl(name):
    if not re.fullmatch(r"[a-z_]+_\d+\.stl", name) or not (WORKDIR / name).exists(): abort(404)
    return send_file((WORKDIR / name).resolve(), mimetype="model/stl", as_attachment=request.args.get("dl") == "1", download_name=name)

# ---- start the server (re-running this cell restarts it cleanly)
PORT = 8765
if globals().get("_SERVER"): _SERVER.shutdown()
_SERVER = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_SERVER.serve_forever, daemon=True).start()
try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(PORT, height=950)      # shows the page right here in the notebook
    print("If the page doesn't show, use output.serve_kernel_port_as_window(PORT) for a separate tab.")
except ImportError:
    print(f"Running locally: open http://localhost:{PORT}")

## 11. Save the artifacts
Downloads a zip with the classifier, the LoRA adapter, the catalog tables, the split, and the feedback log — everything needed to move the app to a Hugging Face Space.

In [ ]:
# ------------------------------------------------------------------ SAVE everything you need to deploy on Hugging Face Spaces / for the report
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
torch.save({"state_dict": clf.state_dict(), "vocab": VOCAB, "part_types": PART_TYPES}, ART / "classifier.pt")
model.save_pretrained(ART / "lora_adapter"); tok.save_pretrained(ART / "lora_adapter")
json.dump({"base_model": MODEL_NAME, "schema": SCHEMA, "standards": STANDARDS, "washer_lookup": WASHER_LOOKUP,
           "ranges": RANGES, "num_tol": NUM_TOL}, open(ART / "catalog_and_config.json", "w"), indent=1)
data[["sheet", "sheet_row", "split", "part_type", "text", "text_norm", "target"]].to_csv(ART / "dataset_with_split.csv", index=False)   # REAL data only
if len(aug_df): aug_df[["aug_from", "part_type", "orig_text", "text", "text_norm", "target"]].to_csv(ART / "augmented_samples.csv", index=False)   # SYNTHETIC, kept separate
if FEAT_READY:                                                      # all hole-feature data is SYNTHETIC (Claude-written / Claude-generated); saved apart from the manual data
    _cols = ["source", "sheet", "sheet_row", "split", "part_type", "text", "text_norm", "target"]
    for _src in feat_df.source.unique(): feat_df[feat_df.source == _src][_cols].to_csv(ART / f"hole_feature_{_src}.csv", index=False)
if len(clf_extra_df): clf_extra_df[["part_type", "text", "text_norm", "source"]].to_csv(ART / "classifier_phrases_claude_synthetic.csv", index=False)   # SYNTHETIC, kept separate
if FEEDBACK_FEATURES.exists(): (ART / "feedback_features.jsonl").write_text(FEEDBACK_FEATURES.read_text())
if FEEDBACK_LOG.exists(): (ART / "feedback.jsonl").write_text(FEEDBACK_LOG.read_text())
with zipfile.ZipFile("artifacts.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in ART.rglob("*"):
        if f.is_file(): z.write(f, f.relative_to(ART.parent))
print("saved:", sorted(p.name for p in ART.iterdir()))
try:
    from google.colab import files; files.download("artifacts.zip")
except ImportError:
    pass